In [17]:
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk up from start until we find the project root."""
    p = start.resolve()
    for _ in range(10):  # don't walk forever
        if (p / ".git").is_dir() or (p / "data" / "processed").is_dir():
            return p
        if p.parent == p:  # hit filesystem root
            break
        p = p.parent
    raise RuntimeError(f"Could not find project root from {start}")

ROOT = find_project_root(Path.cwd())

assert (ROOT / "src" / "tutor").is_dir(), f"src/tutor missing under {ROOT}"
assert (ROOT / "data" / "processed" / "memo_step_marks").is_dir(), \
    f"memo_step_marks missing under {ROOT}"

print("CWD was:  ", Path.cwd())
print("ROOT is:  ", ROOT)
print("Ready.")

CWD was:   c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor
ROOT is:   C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence
Ready.


In [18]:
csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final
2023_P1_Q1.1.1,1,Factorise,1,"x^2+x-12=0 -> (x+4)(x-3)=0","(x+4)(x-3)=0|(x-3)(x+4)=0",algebra.quadratic.solve,factorise,A,false
2023_P1_Q1.1.1,2,Set each factor to zero,1,"x+4=0 or x-3=0","x+4=0 or x-3=0",algebra.quadratic.solve,factorise,A,false
2023_P1_Q1.1.1,3,Write both roots,1,"x=3 or x=-4","x=3 or x=-4|x=-4 or x=3|x=3; x=-4",algebra.quadratic.solve,factorise,A,true
2023_P1_Q1.1.2,1,Rearrange to standard form,1,"3x^2-2x=6 -> 3x^2-2x-6=0","3x^2-2x-6=0|3x2-2x-6=0",algebra.quadratic.solve,formula,A,false
2023_P1_Q1.1.2,2,Substitute into quadratic formula,1,"x = (2 ± sqrt(76)) / 6","x=(2±√76)/6|x=(2±sqrt(76))/6|x=(2+√76)/6 or x=(2-√76)/6",algebra.quadratic.solve,formula,A,false
2023_P1_Q1.1.2,3,First root x = 1.79,1,x=1.79,"x=1.79|1.79",algebra.quadratic.solve,formula,A,false
2023_P1_Q1.1.2,4,Second root x = -1.12,1,x=-1.12,"x=-1.12|-1.12",algebra.quadratic.solve,formula,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"
csv_path.write_text(csv_content, encoding="utf-8")

lines = csv_path.read_text(encoding="utf-8").splitlines()
print(f"Wrote {len(lines)} lines to {csv_path.name}")
assert len(lines) == 8, f"Expected 8 lines, got {len(lines)}"
print("OK")

Wrote 8 lines to memo_step_marks_2023_P1.csv
OK


In [19]:
step_grader_source = r'''from __future__ import annotations

import csv
import re
from pathlib import Path
from typing import Any

ROOT = Path(__file__).resolve().parents[2]
DEFAULT_CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"


def load_steps(csv_path: Path | None = None) -> dict[str, list[dict[str, Any]]]:
    path = csv_path or DEFAULT_CSV
    by_q: dict[str, list[dict[str, Any]]] = {}
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            qid = row["question_id"].strip()
            by_q.setdefault(qid, []).append({
                "index": int(row["step_index"]),
                "desc": row["step_description"].strip(),
                "marks": int(row["marks"]),
                "memo_line": row["memo_line"].strip(),
                "forms": [x.strip() for x in row["acceptable_forms"].split("|") if x.strip()],
                "skill_id": row["skill_id"].strip(),
                "method_tag": row["method_tag"].strip(),
                "method_group": row.get("method_group", "A").strip(),
                "is_final": str(row.get("is_final", "false")).strip().lower() == "true",
            })
    for qid in by_q:
        by_q[qid].sort(key=lambda s: s["index"])
    return by_q


_SEP = re.compile(r"\s*(?:,|\bor\b|;|\band\b)\s*")
_METHOD_MARKERS = re.compile(r"sqrt|\u00b1|\+/-|/2a|b2-4ac|b\^2|\\pm|\u221a")


def _norm(s: str) -> str:
    s = s.lower().strip()
    s = s.replace("\u00d7", "*").replace("\u2212", "-").replace("\u2013", "-")
    s = s.replace("\u2234", " ").replace("\u2235", " ")
    s = re.sub(r"^\s*(therefore|so|hence|thus)\b\s*", "", s)
    # Decimal comma -> period (SA/European notation). Do this before tokenising
    # so "x=1,79" doesn't split into tokens "x=1" and "79".
    s = re.sub(r"(\d),(\d)", r"\1.\2", s)
    return s


def _tokens(s: str) -> list[str]:
    parts = _SEP.split(_norm(s))
    return sorted(re.sub(r"\s+", "", p) for p in parts if p.strip())


def _line_matches(line: str, forms: list[str], is_final: bool = False) -> bool:
    lt = _tokens(line)
    if not lt:
        return False
    lt_set = set(lt)
    for form in forms:
        ft = _tokens(form)
        if not ft:
            continue
        if is_final:
            # Final-answer steps: form tokens must all appear in line tokens.
            # Lets "x=1.79 or x=-1.12" satisfy both "x=1.79" and "x=-1.12" steps.
            if set(ft).issubset(lt_set):
                return True
        else:
            if ft == lt:
                return True
    return False


def grade_steps(question_id: str, learner_lines: list[str], csv_path: Path | None = None) -> dict[str, Any]:
    steps_map = load_steps(csv_path)
    steps = steps_map.get(question_id, [])
    if not steps:
        return {
            "question_id": question_id,
            "total_marks": 0,
            "awarded_marks": 0,
            "steps": [],
            "feedback": "No step scheme for " + question_id,
        }
    used: set[int] = set()
    results = []
    awarded = 0
    for step in steps:
        matched = False
        matched_line = None
        is_final = step.get("is_final", False)
        for i, line in enumerate(learner_lines):
            if i in used and not is_final:
                continue
            if _line_matches(line, step["forms"], is_final=is_final):
                matched = True
                matched_line = line
                if not is_final:
                    used.add(i)
                awarded += step["marks"]
                break
        results.append({
            "index": step["index"],
            "desc": step["desc"],
            "marks": step["marks"],
            "matched": matched,
            "learner_line": matched_line,
            "is_final": is_final,
        })

    final_matched = any(r["matched"] and r["is_final"] for r in results)
    unmatched_lines = [ln for i, ln in enumerate(learner_lines) if i not in used]
    has_method_evidence = any(_METHOD_MARKERS.search(ln) for ln in unmatched_lines)

    for r in results:
        if r["matched"]:
            r["match_status"] = "MATCHED"
        elif final_matched and has_method_evidence:
            r["match_status"] = "UNCERTAIN"
        else:
            r["match_status"] = "NOT_MATCHED"

    total = sum(s["marks"] for s in steps)
    missing = [r for r in results if r["match_status"] == "NOT_MATCHED"]
    uncertain = [r for r in results if r["match_status"] == "UNCERTAIN"]

    if not missing and not uncertain:
        feedback = "Full marks: %d/%d" % (awarded, total)
    elif uncertain and not missing:
        bits = ["cannot verify '%s' (alternate method used)" % u["desc"] for u in uncertain]
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)
    else:
        bits = ["missed '%s' (%d mark)" % (m["desc"], m["marks"]) for m in missing]
        if uncertain:
            bits.append("cannot verify " + ", ".join("'%s'" % u["desc"] for u in uncertain))
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)

    return {
        "question_id": question_id,
        "total_marks": total,
        "awarded_marks": awarded,
        "steps": results,
        "feedback": feedback,
    }
'''

sg_path = ROOT / "src" / "tutor" / "step_grader.py"
sg_path.write_text(step_grader_source, encoding="utf-8")

line_count = len(sg_path.read_text(encoding="utf-8").splitlines())
print(f"Wrote {line_count} lines to {sg_path.name}")
assert line_count > 100, f"Too short — paste failed ({line_count} lines)"
print("OK")

Wrote 144 lines to step_grader.py
OK


In [20]:
test_source = r'''"""
Test harness for step_grader.py

Run:  python tests/test_step_grader.py
Exit code 0 if all cases pass, 1 otherwise.
"""
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.tutor.step_grader import grade_steps

CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

# (name, qid, learner_lines, exp_awarded, exp_missing, note, xfail)
CASES = [
    # ---- 2023 P1 Q1.1.1 (factorise) ----
    ("Q1.1.1 / 01 perfect working", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4 or x=3"],
     3, [], "", False),
    ("Q1.1.1 / 02 answer only", "2023_P1_Q1.1.1",
     ["x=-4 or x=3"],
     1, [1, 2], "", False),
    ("Q1.1.1 / 03 factor only", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0"],
     1, [2, 3], "", False),
    ("Q1.1.1 / 04 skip step 2", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x=-4 or x=3"],
     2, [2], "", False),
    ("Q1.1.1 / 05 roots in memo order", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=3 or x=-4"],
     3, [], "", False),
    ("Q1.1.1 / 06 semicolon roots", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=3; x=-4"],
     3, [], "", False),
    ("Q1.1.1 / 07 comma roots", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4, x=3"],
     3, [], "", False),
    ("Q1.1.1 / 08 Therefore lead-in", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "Therefore x=-4 or x=3"],
     3, [], "", False),
    ("Q1.1.1 / 09 wrong factor sign", "2023_P1_Q1.1.1",
     ["(x-4)(x+3)=0", "x=-4 or x=3"],
     1, [1, 2], "", False),
    ("Q1.1.1 / 10 out-of-order roots first", "2023_P1_Q1.1.1",
     ["x=-4 or x=3", "(x+4)(x-3)=0", "x+4=0 or x-3=0"],
     3, [], "", False),
    ("Q1.1.1 / 11 alternate factor order", "2023_P1_Q1.1.1",
     ["(x-3)(x+4)=0", "x-3=0 or x+4=0", "x=-4 or x=3"],
     3, [], "", False),
    ("Q1.1.1 / 12 therefore symbol lead-in", "2023_P1_Q1.1.1",
     ["\u2234 (x+4)(x-3)=0", "x=-4 or x=3"],
     2, [2], "", False),
    ("Q1.1.1 / 13 comma-or mix", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x=-4, or x=3"],
     2, [2], "", False),
    ("Q1.1.1 / 14 one root only", "2023_P1_Q1.1.1",
     ["x=-4"],
     0, [1, 2, 3], "", False),
    ("Q1.1.1 / 15 one root wrong", "2023_P1_Q1.1.1",
     ["x=-4 or x=5"],
     0, [1, 2, 3], "", False),
    ("Q1.1.1 / 16 mixed notation", "2023_P1_Q1.1.1",
     ["x = -4 or 3"],
     0, [1, 2, 3], "", False),
    ("Q1.1.1 / 17 whitespace heavy", "2023_P1_Q1.1.1",
     ["  (x+4)(x-3)  =  0  ", "x = -4  or  x = 3"],
     2, [2], "", False),
    ("Q1.1.1 / 18 factor as (x+4)=0 form", "2023_P1_Q1.1.1",
     ["(x+4)=0 or (x-3)=0", "(x+4)(x-3)=0", "x=-4 or x=3"],
     2, [2], "", False),
    ("Q1.1.1 / 19 So lead-in on step 2", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "So x+4=0 or x-3=0", "x=-4 or x=3"],
     3, [], "", False),
    ("Q1.1.1 / 20 formula method (alternate)", "2023_P1_Q1.1.1",
     ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
     1, [],
     "steps 1-2 UNCERTAIN (alternate method)", False),

    # ---- 2023 P1 Q1.1.2 (formula) ----
    ("Q1.1.2 / 01 perfect separate roots", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2±√76)/6", "x=1.79", "x=-1.12"],
     4, [], "", False),
    ("Q1.1.2 / 02 perfect roots on one line", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2±√76)/6", "x=1.79 or x=-1.12"],
     4, [], "", False),
    ("Q1.1.2 / 03 standard form only", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0"],
     1, [2, 3, 4], "", False),
    ("Q1.1.2 / 04 standard form + formula", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2±√76)/6"],
     2, [3, 4], "", False),
    ("Q1.1.2 / 05 answers only both roots", "2023_P1_Q1.1.2",
     ["x=1.79 or x=-1.12"],
     2, [1, 2], "", False),
    ("Q1.1.2 / 06 answer only one root", "2023_P1_Q1.1.2",
     ["x=1.79"],
     1, [1, 2, 4], "", False),
    ("Q1.1.2 / 07 wrong rounding", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2±√76)/6", "x=1.8 or x=-1.1"],
     2, [3, 4], "", False),
    ("Q1.1.2 / 08 comma decimal notation", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2±√76)/6", "x=1,79 or x=-1,12"],
     4, [], "", False),
]


def run() -> int:
    passed = xfailed = failed = 0
    out = []

    for name, qid, working, exp_awarded, exp_missing, note, xfail in CASES:
        try:
            r = grade_steps(qid, working, CSV)
            awarded = r["awarded_marks"]
            missing = sorted(s["index"] for s in r["steps"] if s["match_status"] == "NOT_MATCHED")
            ok = (awarded == exp_awarded) and (missing == sorted(exp_missing))
        except Exception as e:
            awarded = missing = None
            ok = False
            note = "EXCEPTION: " + str(e) + " | " + note

        if ok and not xfail:
            passed += 1
            status = "PASS "
        elif (not ok) and xfail:
            xfailed += 1
            status = "XFAIL"
        else:
            failed += 1
            status = "FAIL "

        out.append("  [" + status + "] " + name)
        if note:
            out.append("           note: " + note)
        if not ok and not xfail:
            out.append("           expected " + str(exp_awarded) + " missing=" + str(sorted(exp_missing)))
            out.append("           got      " + str(awarded) + " missing=" + str(missing))

    # Assertion: Q1.1.1 case 20 -> UNCERTAIN, UNCERTAIN, MATCHED
    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
                      CSV)
    statuses = [s["match_status"] for s in r20["steps"]]
    if statuses == ["UNCERTAIN", "UNCERTAIN", "MATCHED"]:
        passed += 1
        out.append("  [PASS ] assertion / Q1.1.1 case 20 UNCERTAIN -> " + str(statuses))
    else:
        failed += 1
        out.append("  [FAIL ] assertion / Q1.1.1 case 20 UNCERTAIN")
        out.append("           expected ['UNCERTAIN', 'UNCERTAIN', 'MATCHED']")
        out.append("           got      " + str(statuses))

    print("\n".join(out))
    print()
    print("  " + str(passed) + " passed  |  " + str(xfailed) + " xfail  |  "
          + str(failed) + " failed  |  " + str(len(CASES) + 1) + " total")
    return failed


if __name__ == "__main__":
    sys.exit(1 if run() else 0)
'''

tt_path = ROOT / "tests" / "test_step_grader.py"
tt_path.write_text(test_source, encoding="utf-8")

line_count = len(tt_path.read_text(encoding="utf-8").splitlines())
print(f"Wrote {line_count} lines to {tt_path.name}")
assert line_count > 130, f"Too short — paste failed ({line_count} lines)"
print("OK")

Wrote 165 lines to test_step_grader.py
OK


In [21]:
from pathlib import Path

# Same ROOT from Cell 1 — walk up if this cell runs standalone
def find_project_root(start):
    p = start.resolve()
    for _ in range(10):
        if (p / ".git").is_dir() or (p / "data" / "processed").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("project root not found")

ROOT = find_project_root(Path.cwd())

csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final
2023_P1_Q1.1.1,1,Factorise,1,"x^2+x-12=0 -> (x+4)(x-3)=0","(x+4)(x-3)=0|(x-3)(x+4)=0",algebra.quadratic.solve,factorise,A,false
2023_P1_Q1.1.1,2,Set each factor to zero,1,"x+4=0 or x-3=0","x+4=0 or x-3=0",algebra.quadratic.solve,factorise,A,false
2023_P1_Q1.1.1,3,Write both roots,1,"x=3 or x=-4","x=3 or x=-4|x=-4 or x=3|x=3; x=-4",algebra.quadratic.solve,factorise,A,true
2023_P1_Q1.1.2,1,Rearrange to standard form,1,"3x^2-2x=6 -> 3x^2-2x-6=0","3x^2-2x-6=0|3x2-2x-6=0",algebra.quadratic.solve,formula,A,false
2023_P1_Q1.1.2,2,Substitute into quadratic formula,1,"x = (2 ± sqrt(76)) / 6","x=(2±√76)/6|x=(2±sqrt(76))/6|x=(2+√76)/6 or x=(2-√76)/6",algebra.quadratic.solve,formula,A,false
2023_P1_Q1.1.2,3,First root x = 1.79,1,x=1.79,"x=1.79|1.79",algebra.quadratic.solve,formula,A,true
2023_P1_Q1.1.2,4,Second root x = -1.12,1,x=-1.12,"x=-1.12|-1.12",algebra.quadratic.solve,formula,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"
csv_path.write_text(csv_content, encoding="utf-8")

lines = csv_path.read_text(encoding="utf-8").splitlines()
print(f"Wrote {len(lines)} lines")
assert len(lines) == 8, f"Expected 8 lines, got {len(lines)}"

# Show the two changed rows to confirm
for ln in lines:
    if "Q1.1.2,3" in ln or "Q1.1.2,4" in ln:
        print(" ", ln)
print("OK")

Wrote 8 lines
  2023_P1_Q1.1.2,3,First root x = 1.79,1,x=1.79,"x=1.79|1.79",algebra.quadratic.solve,formula,A,true
  2023_P1_Q1.1.2,4,Second root x = -1.12,1,x=-1.12,"x=-1.12|-1.12",algebra.quadratic.solve,formula,A,true
OK


In [22]:
import importlib
import sys

# Reload so the notebook picks up the file we just wrote
if "src.tutor.step_grader" in sys.modules:
    del sys.modules["src.tutor.step_grader"]

sys.path.insert(0, str(ROOT))
from src.tutor.step_grader import grade_steps

CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

demo = grade_steps(
    "2023_P1_Q1.1.2",
    ["3x^2-2x-6=0", "x=(2±√76)/6", "x=1.79 or x=-1.12"],
    CSV,
)
print("Feedback:", demo["feedback"])
print()
for s in demo["steps"]:
    print(s)

Feedback: Full marks: 4/4

{'index': 1, 'desc': 'Rearrange to standard form', 'marks': 1, 'matched': True, 'learner_line': '3x^2-2x-6=0', 'is_final': False, 'match_status': 'MATCHED'}
{'index': 2, 'desc': 'Substitute into quadratic formula', 'marks': 1, 'matched': True, 'learner_line': 'x=(2±√76)/6', 'is_final': False, 'match_status': 'MATCHED'}
{'index': 3, 'desc': 'First root x = 1.79', 'marks': 1, 'matched': True, 'learner_line': 'x=1.79 or x=-1.12', 'is_final': True, 'match_status': 'MATCHED'}
{'index': 4, 'desc': 'Second root x = -1.12', 'marks': 1, 'matched': True, 'learner_line': 'x=1.79 or x=-1.12', 'is_final': True, 'match_status': 'MATCHED'}


In [23]:
import subprocess

result = subprocess.run(
    [sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
print("Exit code:", result.returncode)

  [PASS ] Q1.1.1 / 01 perfect working
  [PASS ] Q1.1.1 / 02 answer only
  [PASS ] Q1.1.1 / 03 factor only
  [PASS ] Q1.1.1 / 04 skip step 2
  [PASS ] Q1.1.1 / 05 roots in memo order
  [PASS ] Q1.1.1 / 06 semicolon roots
  [PASS ] Q1.1.1 / 07 comma roots
  [PASS ] Q1.1.1 / 08 Therefore lead-in
  [PASS ] Q1.1.1 / 09 wrong factor sign
  [PASS ] Q1.1.1 / 10 out-of-order roots first
  [PASS ] Q1.1.1 / 11 alternate factor order
  [PASS ] Q1.1.1 / 12 therefore symbol lead-in
  [PASS ] Q1.1.1 / 13 comma-or mix
  [PASS ] Q1.1.1 / 14 one root only
  [PASS ] Q1.1.1 / 15 one root wrong
  [PASS ] Q1.1.1 / 16 mixed notation
  [PASS ] Q1.1.1 / 17 whitespace heavy
  [PASS ] Q1.1.1 / 18 factor as (x+4)=0 form
  [PASS ] Q1.1.1 / 19 So lead-in on step 2
  [PASS ] Q1.1.1 / 20 formula method (alternate)
           note: steps 1-2 UNCERTAIN (alternate method)
  [PASS ] Q1.1.2 / 01 perfect separate roots
  [PASS ] Q1.1.2 / 02 perfect roots on one line
  [PASS ] Q1.1.2 / 03 standard form only
  [PASS ] Q1.1.2

In [25]:
from pathlib import Path

def find_project_root(start):
    p = start.resolve()
    for _ in range(10):
        if (p / ".git").is_dir() or (p / "data" / "processed").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("project root not found")

ROOT = find_project_root(Path.cwd())

csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final
2023_P1_Q1.1.1,1,Factorise,1,"x^2+x-12=0 -> (x+4)(x-3)=0","(x+4)(x-3)=0|(x-3)(x+4)=0",algebra.quadratic.solve,factorise,A,false
2023_P1_Q1.1.1,2,Set each factor to zero,1,"x+4=0 or x-3=0","x+4=0 or x-3=0",algebra.quadratic.solve,factorise,A,false
2023_P1_Q1.1.1,3,Write both roots,1,"x=3 or x=-4","x=3 or x=-4|x=-4 or x=3|x=3; x=-4",algebra.quadratic.solve,factorise,A,true
2023_P1_Q1.1.2,1,Rearrange to standard form,1,"3x^2-2x=6 -> 3x^2-2x-6=0","3x^2-2x-6=0|3x2-2x-6=0",algebra.quadratic.solve,formula,A,false
2023_P1_Q1.1.2,2,Substitute into quadratic formula,1,"x = (2 ± sqrt(76)) / 6","x=(2±√76)/6|x=(2±sqrt(76))/6|x=(2+√76)/6 or x=(2-√76)/6",algebra.quadratic.solve,formula,A,false
2023_P1_Q1.1.2,3,First root x = 1.79,1,x=1.79,"x=1.79|1.79",algebra.quadratic.solve,formula,A,true
2023_P1_Q1.1.2,4,Second root x = -1.12,1,x=-1.12,"x=-1.12|-1.12",algebra.quadratic.solve,formula,A,true
2023_P1_Q1.1.3,1,Square both sides,1,"2x+1 = (x-1)^2","2x+1=(x-1)^2|2x+1=(x-1)2",algebra.surd.solve,surd,A,false
2023_P1_Q1.1.3,2,Rearrange to standard form,1,"x^2-4x=0","x^2-4x=0|x(x-4)=0",algebra.surd.solve,surd,A,false
2023_P1_Q1.1.3,3,Both candidate values,1,"x=0 or x=4","x=0 or x=4|x=4 or x=0",algebra.surd.solve,surd,A,true
2023_P1_Q1.1.3,4,Reject invalid root,1,"x=4 (x=0 rejected)","x=4|x≠0 and x=4|x=4 only",algebra.surd.solve,surd,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"
csv_path.write_text(csv_content, encoding="utf-8")

lines = csv_path.read_text(encoding="utf-8").splitlines()
print(f"Wrote {len(lines)} lines")
assert len(lines) == 12, f"Expected 12 lines, got {len(lines)}"
print("OK")

Wrote 12 lines
OK


In [26]:
test_source = r'''"""
Test harness for step_grader.py
"""
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.tutor.step_grader import grade_steps

CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

# (name, qid, learner_lines, exp_awarded, exp_missing, note, xfail)
CASES = [
    # ---- 2023 P1 Q1.1.1 (factorise) ----
    ("Q1.1.1 / 01 perfect working", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4 or x=3"],
     3, [], "", False),
    ("Q1.1.1 / 02 answer only", "2023_P1_Q1.1.1",
     ["x=-4 or x=3"],
     1, [1, 2], "", False),
    ("Q1.1.1 / 03 factor only", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0"],
     1, [2, 3], "", False),
    ("Q1.1.1 / 04 skip step 2", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x=-4 or x=3"],
     2, [2], "", False),
    ("Q1.1.1 / 05 roots in memo order", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=3 or x=-4"],
     3, [], "", False),
    ("Q1.1.1 / 06 semicolon roots", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=3; x=-4"],
     3, [], "", False),
    ("Q1.1.1 / 07 comma roots", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4, x=3"],
     3, [], "", False),
    ("Q1.1.1 / 08 Therefore lead-in", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "Therefore x=-4 or x=3"],
     3, [], "", False),
    ("Q1.1.1 / 09 wrong factor sign", "2023_P1_Q1.1.1",
     ["(x-4)(x+3)=0", "x=-4 or x=3"],
     1, [1, 2], "", False),
    ("Q1.1.1 / 10 out-of-order roots first", "2023_P1_Q1.1.1",
     ["x=-4 or x=3", "(x+4)(x-3)=0", "x+4=0 or x-3=0"],
     3, [], "", False),
    ("Q1.1.1 / 11 alternate factor order", "2023_P1_Q1.1.1",
     ["(x-3)(x+4)=0", "x-3=0 or x+4=0", "x=-4 or x=3"],
     3, [], "", False),
    ("Q1.1.1 / 12 therefore symbol lead-in", "2023_P1_Q1.1.1",
     ["\u2234 (x+4)(x-3)=0", "x=-4 or x=3"],
     2, [2], "", False),
    ("Q1.1.1 / 13 comma-or mix", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x=-4, or x=3"],
     2, [2], "", False),
    ("Q1.1.1 / 14 one root only", "2023_P1_Q1.1.1",
     ["x=-4"],
     0, [1, 2, 3], "", False),
    ("Q1.1.1 / 15 one root wrong", "2023_P1_Q1.1.1",
     ["x=-4 or x=5"],
     0, [1, 2, 3], "", False),
    ("Q1.1.1 / 16 mixed notation", "2023_P1_Q1.1.1",
     ["x = -4 or 3"],
     0, [1, 2, 3], "", False),
    ("Q1.1.1 / 17 whitespace heavy", "2023_P1_Q1.1.1",
     ["  (x+4)(x-3)  =  0  ", "x = -4  or  x = 3"],
     2, [2], "", False),
    ("Q1.1.1 / 18 factor as (x+4)=0 form", "2023_P1_Q1.1.1",
     ["(x+4)=0 or (x-3)=0", "(x+4)(x-3)=0", "x=-4 or x=3"],
     2, [2], "", False),
    ("Q1.1.1 / 19 So lead-in on step 2", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "So x+4=0 or x-3=0", "x=-4 or x=3"],
     3, [], "", False),
    ("Q1.1.1 / 20 formula method (alternate)", "2023_P1_Q1.1.1",
     ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
     1, [],
     "steps 1-2 UNCERTAIN (alternate method)", False),

    # ---- 2023 P1 Q1.1.2 (formula) ----
    ("Q1.1.2 / 01 perfect separate roots", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79", "x=-1.12"],
     4, [], "", False),
    ("Q1.1.2 / 02 perfect roots on one line", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79 or x=-1.12"],
     4, [], "", False),
    ("Q1.1.2 / 03 standard form only", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0"],
     1, [2, 3, 4], "", False),
    ("Q1.1.2 / 04 standard form + formula", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6"],
     2, [3, 4], "", False),
    ("Q1.1.2 / 05 answers only both roots", "2023_P1_Q1.1.2",
     ["x=1.79 or x=-1.12"],
     2, [1, 2], "", False),
    ("Q1.1.2 / 06 answer only one root", "2023_P1_Q1.1.2",
     ["x=1.79"],
     1, [1, 2, 4], "", False),
    ("Q1.1.2 / 07 wrong rounding", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.8 or x=-1.1"],
     2, [3, 4], "", False),
    ("Q1.1.2 / 08 comma decimal notation", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1,79 or x=-1,12"],
     4, [], "", False),

    # ---- 2023 P1 Q1.1.3 (surd) ----
    ("Q1.1.3 / 01 perfect working with rejection", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=4"],
     4, [], "", False),
    ("Q1.1.3 / 02 perfect compact", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x(x-4)=0", "x=0 or x=4", "x=4 only"],
     3, [4], "rejection phrase 'only' not yet recognised", False),
    ("Q1.1.3 / 03 both roots, no rejection", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4"],
     3, [4], "", False),
    ("Q1.1.3 / 04 answer only", "2023_P1_Q1.1.3",
     ["x=4"],
     2, [1, 2], "answer-only: gives final-value steps, misses process", False),
    ("Q1.1.3 / 05 no rejection, wrong root given", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=0"],
     3, [4], "", False),
    ("Q1.1.3 / 06 squaring only", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2"],
     1, [2, 3, 4], "", False),
    ("Q1.1.3 / 07 standard form only", "2023_P1_Q1.1.3",
     ["x^2-4x=0"],
     1, [1, 3, 4], "", False),
    ("Q1.1.3 / 08 reversed candidate order", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=4 or x=0", "x=4"],
     4, [], "", False),
]


def run() -> int:
    passed = xfailed = failed = 0
    out = []

    for name, qid, working, exp_awarded, exp_missing, note, xfail in CASES:
        try:
            r = grade_steps(qid, working, CSV)
            awarded = r["awarded_marks"]
            missing = sorted(s["index"] for s in r["steps"] if s["match_status"] == "NOT_MATCHED")
            ok = (awarded == exp_awarded) and (missing == sorted(exp_missing))
        except Exception as e:
            awarded = missing = None
            ok = False
            note = "EXCEPTION: " + str(e) + " | " + note

        if ok and not xfail:
            passed += 1
            status = "PASS "
        elif (not ok) and xfail:
            xfailed += 1
            status = "XFAIL"
        else:
            failed += 1
            status = "FAIL "

        out.append("  [" + status + "] " + name)
        if note:
            out.append("           note: " + note)
        if not ok and not xfail:
            out.append("           expected " + str(exp_awarded) + " missing=" + str(sorted(exp_missing)))
            out.append("           got      " + str(awarded) + " missing=" + str(missing))

    # Assertion: Q1.1.1 case 20 -> UNCERTAIN, UNCERTAIN, MATCHED
    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
                      CSV)
    statuses = [s["match_status"] for s in r20["steps"]]
    if statuses == ["UNCERTAIN", "UNCERTAIN", "MATCHED"]:
        passed += 1
        out.append("  [PASS ] assertion / Q1.1.1 case 20 UNCERTAIN -> " + str(statuses))
    else:
        failed += 1
        out.append("  [FAIL ] assertion / Q1.1.1 case 20 UNCERTAIN")
        out.append("           expected ['UNCERTAIN', 'UNCERTAIN', 'MATCHED']")
        out.append("           got      " + str(statuses))

    print("\n".join(out))
    print()
    print("  " + str(passed) + " passed  |  " + str(xfailed) + " xfail  |  "
          + str(failed) + " failed  |  " + str(len(CASES) + 1) + " total")
    return failed


if __name__ == "__main__":
    sys.exit(1 if run() else 0)
'''

tt_path = ROOT / "tests" / "test_step_grader.py"
tt_path.write_text(test_source, encoding="utf-8")

line_count = len(tt_path.read_text(encoding="utf-8").splitlines())
print(f"Wrote {line_count} lines to {tt_path.name}")
assert line_count > 180, f"Too short — paste failed ({line_count} lines)"
print("OK")

Wrote 188 lines to test_step_grader.py
OK


In [27]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("Exit code:", result.returncode)

  [PASS ] Q1.1.1 / 01 perfect working
  [PASS ] Q1.1.1 / 02 answer only
  [PASS ] Q1.1.1 / 03 factor only
  [PASS ] Q1.1.1 / 04 skip step 2
  [PASS ] Q1.1.1 / 05 roots in memo order
  [PASS ] Q1.1.1 / 06 semicolon roots
  [PASS ] Q1.1.1 / 07 comma roots
  [PASS ] Q1.1.1 / 08 Therefore lead-in
  [PASS ] Q1.1.1 / 09 wrong factor sign
  [PASS ] Q1.1.1 / 10 out-of-order roots first
  [PASS ] Q1.1.1 / 11 alternate factor order
  [PASS ] Q1.1.1 / 12 therefore symbol lead-in
  [PASS ] Q1.1.1 / 13 comma-or mix
  [PASS ] Q1.1.1 / 14 one root only
  [PASS ] Q1.1.1 / 15 one root wrong
  [PASS ] Q1.1.1 / 16 mixed notation
  [PASS ] Q1.1.1 / 17 whitespace heavy
  [PASS ] Q1.1.1 / 18 factor as (x+4)=0 form
  [PASS ] Q1.1.1 / 19 So lead-in on step 2
  [PASS ] Q1.1.1 / 20 formula method (alternate)
           note: steps 1-2 UNCERTAIN (alternate method)
  [PASS ] Q1.1.2 / 01 perfect separate roots
  [PASS ] Q1.1.2 / 02 perfect roots on one line
  [PASS ] Q1.1.2 / 03 standard form only
  [PASS ] Q1.1.2

In [28]:
from pathlib import Path

def find_project_root(start):
    p = start.resolve()
    for _ in range(10):
        if (p / ".git").is_dir() or (p / "data" / "processed").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("project root not found")

ROOT = find_project_root(Path.cwd())

# ---- Fix 1: CSV — step 4 of Q1.1.3 becomes is_final=false, extra forms ----
csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final
2023_P1_Q1.1.1,1,Factorise,1,"x^2+x-12=0 -> (x+4)(x-3)=0","(x+4)(x-3)=0|(x-3)(x+4)=0",algebra.quadratic.solve,factorise,A,false
2023_P1_Q1.1.1,2,Set each factor to zero,1,"x+4=0 or x-3=0","x+4=0 or x-3=0",algebra.quadratic.solve,factorise,A,false
2023_P1_Q1.1.1,3,Write both roots,1,"x=3 or x=-4","x=3 or x=-4|x=-4 or x=3|x=3; x=-4",algebra.quadratic.solve,factorise,A,true
2023_P1_Q1.1.2,1,Rearrange to standard form,1,"3x^2-2x=6 -> 3x^2-2x-6=0","3x^2-2x-6=0|3x2-2x-6=0",algebra.quadratic.solve,formula,A,false
2023_P1_Q1.1.2,2,Substitute into quadratic formula,1,"x = (2 ± sqrt(76)) / 6","x=(2±√76)/6|x=(2±sqrt(76))/6|x=(2+√76)/6 or x=(2-√76)/6",algebra.quadratic.solve,formula,A,false
2023_P1_Q1.1.2,3,First root x = 1.79,1,x=1.79,"x=1.79|1.79",algebra.quadratic.solve,formula,A,true
2023_P1_Q1.1.2,4,Second root x = -1.12,1,x=-1.12,"x=-1.12|-1.12",algebra.quadratic.solve,formula,A,true
2023_P1_Q1.1.3,1,Square both sides,1,"2x+1 = (x-1)^2","2x+1=(x-1)^2|2x+1=(x-1)2",algebra.surd.solve,surd,A,false
2023_P1_Q1.1.3,2,Rearrange to standard form,1,"x^2-4x=0","x^2-4x=0|x(x-4)=0",algebra.surd.solve,surd,A,false
2023_P1_Q1.1.3,3,Both candidate values,1,"x=0 or x=4","x=0 or x=4|x=4 or x=0",algebra.surd.solve,surd,A,true
2023_P1_Q1.1.3,4,Reject invalid root,1,"x=4 (x=0 rejected)","x=4|x=4 only|x≠0 and x=4|x=4, x=0 rejected",algebra.surd.solve,surd,A,false
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"
csv_path.write_text(csv_content, encoding="utf-8")
print(f"CSV: wrote {len(csv_path.read_text(encoding='utf-8').splitlines())} lines")

# ---- Fix 2: test file — case 02 becomes 4/[], case 04 becomes 1/[1,2,3] ----
test_path = ROOT / "tests" / "test_step_grader.py"
src = test_path.read_text(encoding="utf-8")

src = src.replace(
    '''    ("Q1.1.3 / 02 perfect compact", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x(x-4)=0", "x=0 or x=4", "x=4 only"],
     3, [4], "rejection phrase 'only' not yet recognised", False),''',
    '''    ("Q1.1.3 / 02 perfect compact", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x(x-4)=0", "x=0 or x=4", "x=4 only"],
     4, [], "", False),'''
)

src = src.replace(
    '''    ("Q1.1.3 / 04 answer only", "2023_P1_Q1.1.3",
     ["x=4"],
     2, [1, 2], "answer-only: gives final-value steps, misses process", False),''',
    '''    ("Q1.1.3 / 04 answer only", "2023_P1_Q1.1.3",
     ["x=4"],
     1, [1, 2, 3], "", False),'''
)

test_path.write_text(src, encoding="utf-8")
print(f"Tests: wrote {len(src.splitlines())} lines")

# Sanity check: both replacements actually landed
assert '"x=4 only"],\n     4, [], "", False),' in src, "case 02 patch did not land"
assert '"x=4"],\n     1, [1, 2, 3], "", False),' in src, "case 04 patch did not land"
print("OK")

CSV: wrote 12 lines
Tests: wrote 188 lines
OK


In [29]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("Exit code:", result.returncode)

  [PASS ] Q1.1.1 / 01 perfect working
  [PASS ] Q1.1.1 / 02 answer only
  [PASS ] Q1.1.1 / 03 factor only
  [PASS ] Q1.1.1 / 04 skip step 2
  [PASS ] Q1.1.1 / 05 roots in memo order
  [PASS ] Q1.1.1 / 06 semicolon roots
  [PASS ] Q1.1.1 / 07 comma roots
  [PASS ] Q1.1.1 / 08 Therefore lead-in
  [PASS ] Q1.1.1 / 09 wrong factor sign
  [PASS ] Q1.1.1 / 10 out-of-order roots first
  [PASS ] Q1.1.1 / 11 alternate factor order
  [PASS ] Q1.1.1 / 12 therefore symbol lead-in
  [PASS ] Q1.1.1 / 13 comma-or mix
  [PASS ] Q1.1.1 / 14 one root only
  [PASS ] Q1.1.1 / 15 one root wrong
  [PASS ] Q1.1.1 / 16 mixed notation
  [PASS ] Q1.1.1 / 17 whitespace heavy
  [PASS ] Q1.1.1 / 18 factor as (x+4)=0 form
  [PASS ] Q1.1.1 / 19 So lead-in on step 2
  [PASS ] Q1.1.1 / 20 formula method (alternate)
           note: steps 1-2 UNCERTAIN (alternate method)
  [PASS ] Q1.1.2 / 01 perfect separate roots
  [PASS ] Q1.1.2 / 02 perfect roots on one line
  [PASS ] Q1.1.2 / 03 standard form only
  [PASS ] Q1.1.2

In [31]:
step_grader_source = r'''from __future__ import annotations

import csv
import re
from pathlib import Path
from typing import Any

ROOT = Path(__file__).resolve().parents[2]
DEFAULT_CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"


def load_steps(csv_path: Path | None = None) -> dict[str, list[dict[str, Any]]]:
    path = csv_path or DEFAULT_CSV
    by_q: dict[str, list[dict[str, Any]]] = {}
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            qid = row["question_id"].strip()
            mg = (row.get("method_group") or "A").strip() or "A"
            by_q.setdefault(qid, []).append({
                "index": int(row["step_index"]),
                "desc": row["step_description"].strip(),
                "marks": int(row["marks"]),
                "memo_line": row["memo_line"].strip(),
                "forms": [x.strip() for x in row["acceptable_forms"].split("|") if x.strip()],
                "skill_id": row["skill_id"].strip(),
                "method_tag": row["method_tag"].strip(),
                "method_group": mg,
                "is_final": str(row.get("is_final", "false")).strip().lower() == "true",
            })
    for qid in by_q:
        by_q[qid].sort(key=lambda s: (s["method_group"], s["index"]))
    return by_q


_SEP = re.compile(r"\s*(?:,|\bor\b|;|\band\b)\s*")
_METHOD_MARKERS = re.compile(r"sqrt|\u00b1|\+/-|/2a|b2-4ac|b\^2|\\pm|\u221a")


def _norm(s: str) -> str:
    s = s.lower().strip()
    s = s.replace("\u00d7", "*").replace("\u2212", "-").replace("\u2013", "-")
    s = s.replace("\u2234", " ").replace("\u2235", " ")
    s = re.sub(r"^\s*(therefore|so|hence|thus)\b\s*", "", s)
    # Decimal comma -> period (SA/European notation). Do this before tokenising.
    s = re.sub(r"(\d),(\d)", r"\1.\2", s)
    return s


def _tokens(s: str) -> list[str]:
    parts = _SEP.split(_norm(s))
    return sorted(re.sub(r"\s+", "", p) for p in parts if p.strip())


def _line_matches(line: str, forms: list[str], is_final: bool = False) -> bool:
    lt = _tokens(line)
    if not lt:
        return False
    lt_set = set(lt)
    for form in forms:
        ft = _tokens(form)
        if not ft:
            continue
        if is_final:
            if set(ft).issubset(lt_set):
                return True
        else:
            if ft == lt:
                return True
    return False


def _grade_group(steps: list[dict[str, Any]], learner_lines: list[str]):
    used: set[int] = set()
    results = []
    awarded = 0
    for step in steps:
        matched = False
        matched_line = None
        is_final = step.get("is_final", False)
        for i, line in enumerate(learner_lines):
            if i in used and not is_final:
                continue
            if _line_matches(line, step["forms"], is_final=is_final):
                matched = True
                matched_line = line
                if not is_final:
                    used.add(i)
                awarded += step["marks"]
                break
        results.append({
            "index": step["index"],
            "desc": step["desc"],
            "marks": step["marks"],
            "matched": matched,
            "learner_line": matched_line,
            "is_final": is_final,
        })
    return awarded, results, used


def grade_steps(question_id: str, learner_lines: list[str], csv_path: Path | None = None) -> dict[str, Any]:
    steps_map = load_steps(csv_path)
    steps = steps_map.get(question_id, [])
    if not steps:
        return {
            "question_id": question_id,
            "total_marks": 0,
            "awarded_marks": 0,
            "steps": [],
            "feedback": "No step scheme for " + question_id,
        }

    groups: dict[str, list[dict[str, Any]]] = {}
    for s in steps:
        groups.setdefault(s["method_group"], []).append(s)

    best_awarded = -1
    best_results = None
    best_used = None
    best_group = None
    for g_name, g_steps in groups.items():
        awarded, results, used = _grade_group(g_steps, learner_lines)
        if awarded > best_awarded:
            best_awarded = awarded
            best_results = results
            best_used = used
            best_group = g_steps

    results = best_results
    used = best_used
    awarded = best_awarded
    g_steps = best_group

    final_matched = any(r["matched"] and r["is_final"] for r in results)
    unmatched_lines = [ln for i, ln in enumerate(learner_lines) if i not in used]
    has_method_evidence = any(_METHOD_MARKERS.search(ln) for ln in unmatched_lines)

    for r in results:
        if r["matched"]:
            r["match_status"] = "MATCHED"
        elif final_matched and has_method_evidence:
            r["match_status"] = "UNCERTAIN"
        else:
            r["match_status"] = "NOT_MATCHED"

    total = sum(s["marks"] for s in g_steps)
    missing = [r for r in results if r["match_status"] == "NOT_MATCHED"]
    uncertain = [r for r in results if r["match_status"] == "UNCERTAIN"]

    if not missing and not uncertain:
        feedback = "Full marks: %d/%d" % (awarded, total)
    elif uncertain and not missing:
        bits = ["cannot verify '%s' (alternate method used)" % u["desc"] for u in uncertain]
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)
    else:
        bits = ["missed '%s' (%d mark)" % (m["desc"], m["marks"]) for m in missing]
        if uncertain:
            bits.append("cannot verify " + ", ".join("'%s'" % u["desc"] for u in uncertain))
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)

    return {
        "question_id": question_id,
        "total_marks": total,
        "awarded_marks": awarded,
        "steps": results,
        "feedback": feedback,
    }
'''

sg_path = ROOT / "src" / "tutor" / "step_grader.py"
sg_path.write_text(step_grader_source, encoding="utf-8")

line_count = len(sg_path.read_text(encoding="utf-8").splitlines())
print(f"step_grader.py: wrote {line_count} lines")
assert line_count > 130, f"Too short ({line_count} lines)"
print("OK")

step_grader.py: wrote 167 lines
OK


In [32]:
test_source = r'''"""
Test harness for step_grader.py
"""
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.tutor.step_grader import grade_steps

CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

# (name, qid, learner_lines, exp_awarded, exp_missing, note, xfail)
CASES = [
    # ---- Q1.1.1 (factorise) ----
    ("Q1.1.1 / 01 perfect", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.1 / 02 answer only", "2023_P1_Q1.1.1",
     ["x=-4 or x=3"], 1, [1, 2], "", False),
    ("Q1.1.1 / 03 factor only", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0"], 1, [2, 3], "", False),
    ("Q1.1.1 / 04 skip step 2", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x=-4 or x=3"], 2, [2], "", False),
    ("Q1.1.1 / 05 memo order", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=3 or x=-4"], 3, [], "", False),
    ("Q1.1.1 / 06 semicolon", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=3; x=-4"], 3, [], "", False),
    ("Q1.1.1 / 07 comma", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4, x=3"], 3, [], "", False),
    ("Q1.1.1 / 08 Therefore", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "Therefore x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.1 / 09 wrong factor sign", "2023_P1_Q1.1.1",
     ["(x-4)(x+3)=0", "x=-4 or x=3"], 1, [1, 2], "", False),
    ("Q1.1.1 / 10 out of order", "2023_P1_Q1.1.1",
     ["x=-4 or x=3", "(x+4)(x-3)=0", "x+4=0 or x-3=0"], 3, [], "", False),
    ("Q1.1.1 / 11 alt factor order", "2023_P1_Q1.1.1",
     ["(x-3)(x+4)=0", "x-3=0 or x+4=0", "x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.1 / 12 therefore symbol", "2023_P1_Q1.1.1",
     ["\u2234 (x+4)(x-3)=0", "x=-4 or x=3"], 2, [2], "", False),
    ("Q1.1.1 / 13 comma-or mix", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x=-4, or x=3"], 2, [2], "", False),
    ("Q1.1.1 / 14 one root", "2023_P1_Q1.1.1",
     ["x=-4"], 0, [1, 2, 3], "", False),
    ("Q1.1.1 / 15 wrong root", "2023_P1_Q1.1.1",
     ["x=-4 or x=5"], 0, [1, 2, 3], "", False),
    ("Q1.1.1 / 16 mixed", "2023_P1_Q1.1.1",
     ["x = -4 or 3"], 0, [1, 2, 3], "", False),
    ("Q1.1.1 / 17 whitespace", "2023_P1_Q1.1.1",
     ["  (x+4)(x-3)  =  0  ", "x = -4  or  x = 3"], 2, [2], "", False),
    ("Q1.1.1 / 18 (x+4)=0 form", "2023_P1_Q1.1.1",
     ["(x+4)=0 or (x-3)=0", "(x+4)(x-3)=0", "x=-4 or x=3"], 2, [2], "", False),
    ("Q1.1.1 / 19 So lead-in", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "So x+4=0 or x-3=0", "x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.1 / 20 formula alternate", "2023_P1_Q1.1.1",
     ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
     1, [], "steps 1-2 UNCERTAIN", False),

    # ---- Q1.1.2 (formula) ----
    ("Q1.1.2 / 01 perfect separate", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79", "x=-1.12"], 4, [], "", False),
    ("Q1.1.2 / 02 perfect one line", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79 or x=-1.12"], 4, [], "", False),
    ("Q1.1.2 / 03 std form only", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0"], 1, [2, 3, 4], "", False),
    ("Q1.1.2 / 04 std form + formula", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6"], 2, [3, 4], "", False),
    ("Q1.1.2 / 05 answers only", "2023_P1_Q1.1.2",
     ["x=1.79 or x=-1.12"], 2, [1, 2], "", False),
    ("Q1.1.2 / 06 one root", "2023_P1_Q1.1.2",
     ["x=1.79"], 1, [1, 2, 4], "", False),
    ("Q1.1.2 / 07 wrong rounding", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.8 or x=-1.1"], 2, [3, 4], "", False),
    ("Q1.1.2 / 08 comma decimal", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1,79 or x=-1,12"], 4, [], "", False),

    # ---- Q1.1.3 (surd) ----
    ("Q1.1.3 / 01 perfect with rejection", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=4"], 4, [], "", False),
    ("Q1.1.3 / 02 compact", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x(x-4)=0", "x=0 or x=4", "x=4 only"], 4, [], "", False),
    ("Q1.1.3 / 03 no rejection", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4"], 3, [4], "", False),
    ("Q1.1.3 / 04 answer only", "2023_P1_Q1.1.3",
     ["x=4"], 1, [1, 2, 3], "", False),
    ("Q1.1.3 / 05 wrong root only", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=0"], 3, [4], "", False),
    ("Q1.1.3 / 06 squaring only", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2"], 1, [2, 3, 4], "", False),
    ("Q1.1.3 / 07 std form only", "2023_P1_Q1.1.3",
     ["x^2-4x=0"], 1, [1, 3, 4], "", False),
    ("Q1.1.3 / 08 reversed order", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=4 or x=0", "x=4"], 4, [], "", False),

    # ---- Q1.1.4 (inequality) ----
    ("Q1.1.4 / 01 perfect", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "x<-1 or x>3"], 4, [], "", False),
    ("Q1.1.4 / 02 answer only", "2023_P1_Q1.1.4",
     ["x<-1 or x>3"], 2, [1, 2], "", False),
    ("Q1.1.4 / 03 skip CVs", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "x<-1 or x>3"], 3, [2], "", False),
    ("Q1.1.4 / 04 wrong interval", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "-1<x<3"], 2, [3], "", False),
    ("Q1.1.4 / 05 reversed answer", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "x>3 or x<-1"], 4, [], "", False),

    # ---- Q1.2 (simultaneous, two methods) ----
    ("Q1.2 / 01 method A perfect", "2023_P1_Q1.2",
     ["x=2y-2", "1/(2y-2)+1/y=1", "2y^2-5y+2=0", "y=1/2 or y=2", "x=-1 or x=2"],
     5, [], "", False),
    ("Q1.2 / 02 method B perfect", "2023_P1_Q1.2",
     ["y=x/2+1", "1/x+2/(x+2)=1", "x^2-x-2=0", "x=-1 or x=2", "y=1/2 or y=2"],
     5, [], "", False),
    ("Q1.2 / 03 method A no x-values", "2023_P1_Q1.2",
     ["x=2y-2", "1/(2y-2)+1/y=1", "2y^2-5y+2=0", "y=1/2 or y=2"],
     4, [5], "", False),
    ("Q1.2 / 04 answers only", "2023_P1_Q1.2",
     ["x=-1 or x=2", "y=1/2 or y=2"],
     2, [1, 2, 3], "", False),
    ("Q1.2 / 05 std form only", "2023_P1_Q1.2",
     ["2y^2-5y+2=0"],
     1, [1, 2, 4, 5], "", False),

    # ---- Q1.3 (exponential proof) ----
    ("Q1.3 / 01 perfect", "2023_P1_Q1.3",
     ["2^m(2+1)=3^n(3^2-1)", "2^m(3)=3^n(2^3)", "m=3 and n=1", "m+n=4"],
     4, [], "", False),
    ("Q1.3 / 02 skip factor step", "2023_P1_Q1.3",
     ["2^m(3)=3^n(8)", "m=3 and n=1", "m+n=4"],
     3, [1], "", False),
    ("Q1.3 / 03 answer only", "2023_P1_Q1.3",
     ["m=3 and n=1", "m+n=4"],
     2, [1, 2], "", False),
    ("Q1.3 / 04 skip conclusion", "2023_P1_Q1.3",
     ["2^m(2+1)=3^n(3^2-1)", "2^m(3)=3^n(2^3)", "m+n=4"],
     3, [3], "", False),
    ("Q1.3 / 05 final only", "2023_P1_Q1.3",
     ["m+n=4"],
     1, [1, 2, 3], "", False),
]


def run() -> int:
    passed = xfailed = failed = 0
    out = []

    for name, qid, working, exp_awarded, exp_missing, note, xfail in CASES:
        try:
            r = grade_steps(qid, working, CSV)
            awarded = r["awarded_marks"]
            missing = sorted(s["index"] for s in r["steps"] if s["match_status"] == "NOT_MATCHED")
            ok = (awarded == exp_awarded) and (missing == sorted(exp_missing))
        except Exception as e:
            awarded = missing = None
            ok = False
            note = "EXCEPTION: " + str(e) + " | " + note

        if ok and not xfail:
            passed += 1
            status = "PASS "
        elif (not ok) and xfail:
            xfailed += 1
            status = "XFAIL"
        else:
            failed += 1
            status = "FAIL "

        out.append("  [" + status + "] " + name)
        if note:
            out.append("           note: " + note)
        if not ok and not xfail:
            out.append("           expected " + str(exp_awarded) + " missing=" + str(sorted(exp_missing)))
            out.append("           got      " + str(awarded) + " missing=" + str(missing))

    # Assertion: Q1.1.1 case 20 -> UNCERTAIN, UNCERTAIN, MATCHED
    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
                      CSV)
    statuses = [s["match_status"] for s in r20["steps"]]
    if statuses == ["UNCERTAIN", "UNCERTAIN", "MATCHED"]:
        passed += 1
        out.append("  [PASS ] assertion / Q1.1.1 UNCERTAIN -> " + str(statuses))
    else:
        failed += 1
        out.append("  [FAIL ] assertion / Q1.1.1 UNCERTAIN")
        out.append("           expected ['UNCERTAIN', 'UNCERTAIN', 'MATCHED']")
        out.append("           got      " + str(statuses))

    print("\n".join(out))
    print()
    print("  " + str(passed) + " passed  |  " + str(xfailed) + " xfail  |  "
          + str(failed) + " failed  |  " + str(len(CASES) + 1) + " total")
    return failed


if __name__ == "__main__":
    sys.exit(1 if run() else 0)
'''

tt_path = ROOT / "tests" / "test_step_grader.py"
tt_path.write_text(test_source, encoding="utf-8")

line_count = len(tt_path.read_text(encoding="utf-8").splitlines())
print(f"test_step_grader.py: wrote {line_count} lines")
assert line_count > 180, f"Too short ({line_count} lines)"
print("OK")

test_step_grader.py: wrote 198 lines
OK


In [33]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("Exit code:", result.returncode)

  [PASS ] Q1.1.1 / 01 perfect
  [PASS ] Q1.1.1 / 02 answer only
  [PASS ] Q1.1.1 / 03 factor only
  [PASS ] Q1.1.1 / 04 skip step 2
  [PASS ] Q1.1.1 / 05 memo order
  [PASS ] Q1.1.1 / 06 semicolon
  [PASS ] Q1.1.1 / 07 comma
  [PASS ] Q1.1.1 / 08 Therefore
  [PASS ] Q1.1.1 / 09 wrong factor sign
  [PASS ] Q1.1.1 / 10 out of order
  [PASS ] Q1.1.1 / 11 alt factor order
  [PASS ] Q1.1.1 / 12 therefore symbol
  [PASS ] Q1.1.1 / 13 comma-or mix
  [PASS ] Q1.1.1 / 14 one root
  [PASS ] Q1.1.1 / 15 wrong root
  [PASS ] Q1.1.1 / 16 mixed
  [PASS ] Q1.1.1 / 17 whitespace
  [PASS ] Q1.1.1 / 18 (x+4)=0 form
  [PASS ] Q1.1.1 / 19 So lead-in
  [PASS ] Q1.1.1 / 20 formula alternate
           note: steps 1-2 UNCERTAIN
  [PASS ] Q1.1.2 / 01 perfect separate
  [PASS ] Q1.1.2 / 02 perfect one line
  [PASS ] Q1.1.2 / 03 std form only
  [PASS ] Q1.1.2 / 04 std form + formula
  [PASS ] Q1.1.2 / 05 answers only
  [PASS ] Q1.1.2 / 06 one root
  [PASS ] Q1.1.2 / 07 wrong rounding
  [PASS ] Q1.1.2 / 08 comma

In [34]:
new_rows = '''2023_P1_Q2.1.1,1,Identify common difference,1,d=5,d=5,sequences.arithmetic,arith,A,false
2023_P1_Q2.1.1,2,Substitute into T_n formula,1,"T91=7+(91-1)(5)","T91=7+(91-1)(5)|T_91=7+(91-1)(5)|T91=7+90*5",sequences.arithmetic,arith,A,false
2023_P1_Q2.1.1,3,Compute T91,1,T91=457,"T91=457|T_91=457|T₉₁=457|457",sequences.arithmetic,arith,A,true
2023_P1_Q2.1.2,1,Substitute into S_n formula,1,"S91=91/2[2(7)+(91-1)(5)]","S91=91/2[2(7)+(91-1)(5)]|S_91=91/2[2(7)+(91-1)(5)]|S91=91/2(7+457)",sequences.arithmetic,arith,A,false
2023_P1_Q2.1.2,2,Compute S91,1,S91=21112,"S91=21112|S_91=21112|S₉₁=21112|21112",sequences.arithmetic,arith,A,true
2023_P1_Q2.1.3,1,Substitute into T_n formula,1,"Tn=7+(n-1)(5)","Tn=7+(n-1)(5)|Tn=5n+2",sequences.arithmetic,arith,A,false
2023_P1_Q2.1.3,2,Set Tn equal to 517,1,Tn=517,Tn=517,sequences.arithmetic,arith,A,false
2023_P1_Q2.1.3,3,Solve for n,1,n=103,n=103,sequences.arithmetic,arith,A,true
2023_P1_Q2.2.1,1,First and second differences,1,"9; 21; 33; 45","9;21;33;45|9; 21; 33; 45|9 21 33 45",sequences.quadratic,arith,A,false
2023_P1_Q2.2.1,2,Compute T5,1,T5=111,"T5=111|T_5=111|111",sequences.quadratic,arith,A,true
2023_P1_Q2.2.2,1,Second difference equals 2a,1,2a=12,2a=12,sequences.quadratic,arith,A,false
2023_P1_Q2.2.2,2,Solve for b,1,"3(6)+b=9","3(6)+b=9|18+b=9|b=-9",sequences.quadratic,arith,A,false
2023_P1_Q2.2.2,3,Solve for c and write Tn,1,"Tn=6n^2-9n+6","Tn=6n^2-9n+6|6-9+c=3",sequences.quadratic,arith,A,true
2023_P1_Q2.2.3,1,Find turning point equation,1,"12n-9=0","12n-9=0|T'_n=12n-9=0",sequences.quadratic,arith,A,false
2023_P1_Q2.2.3,2,Solve for n,1,n=3/4,n=3/4,sequences.quadratic,arith,A,false
2023_P1_Q2.2.3,3,Conclude increasing,1,"T is increasing for n in N","T is increasing for n in N|increasing for n in N|increasing for n in ℕ",sequences.quadratic,arith,A,true
2023_P1_Q3.1.1,1,Write T_n formula,1,Tn=3(2)^(n-1),"Tn=3(2)^(n-1)|T_n=3(2)^(n-1)|Tn=3·2^(n-1)",sequences.geometric,geom,A,true
2023_P1_Q3.1.2,1,Expand the sum,1,"3+6+12+...","3+6+12+...|3+6+12+...+",sequences.geometric,geom,A,false
2023_P1_Q3.1.2,2,Identify n equals k,1,n=k,n=k,sequences.geometric,geom,A,false
2023_P1_Q3.1.2,3,Substitute into S_n formula,1,"3(2^k-1)=98301","3(2^k-1)=98301|3((2)^k-1)=98301",sequences.geometric,geom,A,false
2023_P1_Q3.1.2,4,Solve for k,1,k=15,k=15,sequences.geometric,geom,A,true
2023_P1_Q3.2,1,Substitute into S_n formula,1,"S22=22/2[2a+21(3)]","S22=22/2[2a+21(3)]|S_22=22/2[2a+21(3)]",sequences.geometric,geom,A,false
2023_P1_Q3.2,2,Simplify S22,1,S22=22a+693,"S22=22a+693|22a+693",sequences.geometric,geom,A,false
2023_P1_Q3.2,3,Substitute into S_inf formula,1,"S_inf=3a/2","S_inf=3a/2|S_inf=a/(1-1/3)|S∞=3a/2|S∞=a/(1-1/3)",sequences.geometric,geom,A,false
2023_P1_Q3.2,4,Set S22 equal to S_inf plus 734,1,"22a+693=3a/2+734","22a+693=3a/2+734|44a+1386=3a+1468",sequences.geometric,geom,A,false
2023_P1_Q3.2,5,Solve for a,1,a=2,a=2,sequences.geometric,geom,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"
existing = csv_path.read_text(encoding="utf-8").rstrip("\n")
csv_path.write_text(existing + "\n" + new_rows, encoding="utf-8")

lines = csv_path.read_text(encoding="utf-8").splitlines()
print(f"CSV: wrote {len(lines)} lines")
assert len(lines) == 55, f"Expected 55 lines, got {len(lines)}"
print("OK")

CSV: wrote 55 lines
OK


In [35]:
test_source = r'''"""
Test harness for step_grader.py
"""
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.tutor.step_grader import grade_steps

CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

CASES = [
    # ================= Q1 (already proven) =================
    ("Q1.1.1 / 01 perfect", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.1 / 02 answer only", "2023_P1_Q1.1.1", ["x=-4 or x=3"], 1, [1, 2], "", False),
    ("Q1.1.1 / 03 factor only", "2023_P1_Q1.1.1", ["(x+4)(x-3)=0"], 1, [2, 3], "", False),
    ("Q1.1.1 / 04 skip step 2", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x=-4 or x=3"], 2, [2], "", False),
    ("Q1.1.1 / 05 memo order", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=3 or x=-4"], 3, [], "", False),
    ("Q1.1.1 / 06 semicolon", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=3; x=-4"], 3, [], "", False),
    ("Q1.1.1 / 07 comma", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4, x=3"], 3, [], "", False),
    ("Q1.1.1 / 08 Therefore", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "Therefore x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.1 / 09 wrong factor", "2023_P1_Q1.1.1",
     ["(x-4)(x+3)=0", "x=-4 or x=3"], 1, [1, 2], "", False),
    ("Q1.1.1 / 10 out of order", "2023_P1_Q1.1.1",
     ["x=-4 or x=3", "(x+4)(x-3)=0", "x+4=0 or x-3=0"], 3, [], "", False),
    ("Q1.1.1 / 11 alt factor order", "2023_P1_Q1.1.1",
     ["(x-3)(x+4)=0", "x-3=0 or x+4=0", "x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.1 / 12 therefore symbol", "2023_P1_Q1.1.1",
     ["\u2234 (x+4)(x-3)=0", "x=-4 or x=3"], 2, [2], "", False),
    ("Q1.1.1 / 13 comma-or mix", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x=-4, or x=3"], 2, [2], "", False),
    ("Q1.1.1 / 14 one root", "2023_P1_Q1.1.1", ["x=-4"], 0, [1, 2, 3], "", False),
    ("Q1.1.1 / 15 wrong root", "2023_P1_Q1.1.1", ["x=-4 or x=5"], 0, [1, 2, 3], "", False),
    ("Q1.1.1 / 16 mixed", "2023_P1_Q1.1.1", ["x = -4 or 3"], 0, [1, 2, 3], "", False),
    ("Q1.1.1 / 17 whitespace", "2023_P1_Q1.1.1",
     ["  (x+4)(x-3)  =  0  ", "x = -4  or  x = 3"], 2, [2], "", False),
    ("Q1.1.1 / 18 (x+4)=0", "2023_P1_Q1.1.1",
     ["(x+4)=0 or (x-3)=0", "(x+4)(x-3)=0", "x=-4 or x=3"], 2, [2], "", False),
    ("Q1.1.1 / 19 So lead-in", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "So x+4=0 or x-3=0", "x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.1 / 20 formula alt", "2023_P1_Q1.1.1",
     ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
     1, [], "steps 1-2 UNCERTAIN", False),

    ("Q1.1.2 / 01 perfect", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79", "x=-1.12"], 4, [], "", False),
    ("Q1.1.2 / 02 one line", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79 or x=-1.12"], 4, [], "", False),
    ("Q1.1.2 / 03 std form only", "2023_P1_Q1.1.2", ["3x^2-2x-6=0"], 1, [2, 3, 4], "", False),
    ("Q1.1.2 / 04 std + formula", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6"], 2, [3, 4], "", False),
    ("Q1.1.2 / 05 answers only", "2023_P1_Q1.1.2",
     ["x=1.79 or x=-1.12"], 2, [1, 2], "", False),
    ("Q1.1.2 / 06 one root", "2023_P1_Q1.1.2", ["x=1.79"], 1, [1, 2, 4], "", False),
    ("Q1.1.2 / 07 wrong rounding", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.8 or x=-1.1"], 2, [3, 4], "", False),
    ("Q1.1.2 / 08 comma decimal", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1,79 or x=-1,12"], 4, [], "", False),

    ("Q1.1.3 / 01 perfect", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=4"], 4, [], "", False),
    ("Q1.1.3 / 02 compact", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x(x-4)=0", "x=0 or x=4", "x=4 only"], 4, [], "", False),
    ("Q1.1.3 / 03 no rejection", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4"], 3, [4], "", False),
    ("Q1.1.3 / 04 answer only", "2023_P1_Q1.1.3", ["x=4"], 1, [1, 2, 3], "", False),
    ("Q1.1.3 / 05 wrong root", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=0"], 3, [4], "", False),
    ("Q1.1.3 / 06 squaring only", "2023_P1_Q1.1.3", ["2x+1=(x-1)^2"], 1, [2, 3, 4], "", False),
    ("Q1.1.3 / 07 std form only", "2023_P1_Q1.1.3", ["x^2-4x=0"], 1, [1, 3, 4], "", False),
    ("Q1.1.3 / 08 reversed", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=4 or x=0", "x=4"], 4, [], "", False),

    ("Q1.1.4 / 01 perfect", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "x<-1 or x>3"], 4, [], "", False),
    ("Q1.1.4 / 02 answer only", "2023_P1_Q1.1.4", ["x<-1 or x>3"], 2, [1, 2], "", False),
    ("Q1.1.4 / 03 skip CVs", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "x<-1 or x>3"], 3, [2], "", False),
    ("Q1.1.4 / 04 wrong interval", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "-1<x<3"], 2, [3], "", False),
    ("Q1.1.4 / 05 reversed answer", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "x>3 or x<-1"], 4, [], "", False),

    ("Q1.2 / 01 method A", "2023_P1_Q1.2",
     ["x=2y-2", "1/(2y-2)+1/y=1", "2y^2-5y+2=0", "y=1/2 or y=2", "x=-1 or x=2"],
     5, [], "", False),
    ("Q1.2 / 02 method B", "2023_P1_Q1.2",
     ["y=x/2+1", "1/x+2/(x+2)=1", "x^2-x-2=0", "x=-1 or x=2", "y=1/2 or y=2"],
     5, [], "", False),
    ("Q1.2 / 03 no x-values", "2023_P1_Q1.2",
     ["x=2y-2", "1/(2y-2)+1/y=1", "2y^2-5y+2=0", "y=1/2 or y=2"],
     4, [5], "", False),
    ("Q1.2 / 04 answers only", "2023_P1_Q1.2",
     ["x=-1 or x=2", "y=1/2 or y=2"], 2, [1, 2, 3], "", False),
    ("Q1.2 / 05 std form only", "2023_P1_Q1.2", ["2y^2-5y+2=0"], 1, [1, 2, 4, 5], "", False),

    ("Q1.3 / 01 perfect", "2023_P1_Q1.3",
     ["2^m(2+1)=3^n(3^2-1)", "2^m(3)=3^n(2^3)", "m=3 and n=1", "m+n=4"],
     4, [], "", False),
    ("Q1.3 / 02 skip factor", "2023_P1_Q1.3",
     ["2^m(3)=3^n(8)", "m=3 and n=1", "m+n=4"], 3, [1], "", False),
    ("Q1.3 / 03 answer only", "2023_P1_Q1.3",
     ["m=3 and n=1", "m+n=4"], 2, [1, 2], "", False),
    ("Q1.3 / 04 skip conclusion", "2023_P1_Q1.3",
     ["2^m(2+1)=3^n(3^2-1)", "2^m(3)=3^n(2^3)", "m+n=4"], 3, [3], "", False),
    ("Q1.3 / 05 final only", "2023_P1_Q1.3", ["m+n=4"], 1, [1, 2, 3], "", False),

    # ================= Q2 (sequences) =================
    ("Q2.1.1 / 01 perfect", "2023_P1_Q2.1.1",
     ["d=5", "T91=7+(91-1)(5)", "T91=457"], 3, [], "", False),
    ("Q2.1.1 / 02 no d", "2023_P1_Q2.1.1",
     ["T91=7+(91-1)(5)", "T91=457"], 2, [1], "", False),
    ("Q2.1.1 / 03 answer only", "2023_P1_Q2.1.1", ["T91=457"], 1, [1, 2], "", False),
    ("Q2.1.1 / 04 Tn variant", "2023_P1_Q2.1.1",
     ["d=5", "Tn=7+(n-1)(5)", "T91=457"], 3, [], "", False),

    ("Q2.1.2 / 01 perfect", "2023_P1_Q2.1.2",
     ["S91=91/2[2(7)+(91-1)(5)]", "S91=21112"], 2, [], "", False),
    ("Q2.1.2 / 02 answer only", "2023_P1_Q2.1.2", ["S91=21112"], 1, [1], "", False),

    ("Q2.1.3 / 01 perfect", "2023_P1_Q2.1.3",
     ["Tn=7+(n-1)(5)", "Tn=517", "n=103"], 3, [], "", False),
    ("Q2.1.3 / 02 answer only", "2023_P1_Q2.1.3", ["n=103"], 1, [1, 2], "", False),

    ("Q2.2.1 / 01 perfect", "2023_P1_Q2.2.1",
     ["9; 21; 33; 45", "T5=111"], 2, [], "", False),
    ("Q2.2.1 / 02 answer only", "2023_P1_Q2.2.1", ["T5=111"], 1, [1], "", False),

    ("Q2.2.2 / 01 perfect", "2023_P1_Q2.2.2",
     ["2a=12", "3(6)+b=9", "Tn=6n^2-9n+6"], 3, [], "", False),
    ("Q2.2.2 / 02 wrong a", "2023_P1_Q2.2.2", ["2a=14"], 0, [1, 2, 3], "", False),

    ("Q2.2.3 / 01 perfect", "2023_P1_Q2.2.3",
     ["12n-9=0", "n=3/4", "T is increasing for n in N"], 3, [], "", False),
    ("Q2.2.3 / 02 answer only", "2023_P1_Q2.2.3",
     ["T is increasing for n in N"], 1, [1, 2], "", False),

    # ================= Q3 (geometric) =================
    ("Q3.1.1 / 01 perfect", "2023_P1_Q3.1.1", ["Tn=3(2)^(n-1)"], 1, [], "", False),

    ("Q3.1.2 / 01 perfect", "2023_P1_Q3.1.2",
     ["3+6+12+...", "n=k", "3(2^k-1)=98301", "k=15"], 4, [], "", False),
    ("Q3.1.2 / 02 no expansion", "2023_P1_Q3.1.2",
     ["n=k", "3(2^k-1)=98301", "k=15"], 3, [1], "", False),
    ("Q3.1.2 / 03 answer only", "2023_P1_Q3.1.2", ["k=15"], 1, [1, 2, 3], "", False),

    ("Q3.2 / 01 perfect", "2023_P1_Q3.2",
     ["S22=22/2[2a+21(3)]", "S22=22a+693", "S_inf=3a/2", "22a+693=3a/2+734", "a=2"],
     5, [], "", False),
    ("Q3.2 / 02 answer only", "2023_P1_Q3.2", ["a=2"], 1, [1, 2, 3, 4], "", False),
    ("Q3.2 / 03 no final answer", "2023_P1_Q3.2",
     ["S22=22/2[2a+21(3)]", "S22=22a+693", "S_inf=3a/2", "22a+693=3a/2+734"],
     4, [5], "", False),
]


def run() -> int:
    passed = failed = 0
    out = []
    for name, qid, working, exp_awarded, exp_missing, note, _xfail in CASES:
        try:
            r = grade_steps(qid, working, CSV)
            awarded = r["awarded_marks"]
            missing = sorted(s["index"] for s in r["steps"] if s["match_status"] == "NOT_MATCHED")
            ok = (awarded == exp_awarded) and (missing == sorted(exp_missing))
        except Exception as e:
            awarded = missing = None
            ok = False
            note = "EXCEPTION: " + str(e) + " | " + note

        if ok:
            passed += 1
            status = "PASS "
        else:
            failed += 1
            status = "FAIL "

        out.append("  [" + status + "] " + name)
        if note:
            out.append("           note: " + note)
        if not ok:
            out.append("           expected " + str(exp_awarded) + " missing=" + str(sorted(exp_missing)))
            out.append("           got      " + str(awarded) + " missing=" + str(missing))

    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
                      CSV)
    statuses = [s["match_status"] for s in r20["steps"]]
    if statuses == ["UNCERTAIN", "UNCERTAIN", "MATCHED"]:
        passed += 1
        out.append("  [PASS ] assertion / Q1.1.1 UNCERTAIN -> " + str(statuses))
    else:
        failed += 1
        out.append("  [FAIL ] assertion / Q1.1.1 UNCERTAIN")

    print("\n".join(out))
    print()
    print("  " + str(passed) + " passed  |  " + str(failed) + " failed  |  "
          + str(len(CASES) + 1) + " total")
    return failed


if __name__ == "__main__":
    sys.exit(1 if run() else 0)
'''

tt_path = ROOT / "tests" / "test_step_grader.py"
tt_path.write_text(test_source, encoding="utf-8")
print(f"test file: wrote {len(tt_path.read_text(encoding='utf-8').splitlines())} lines")
print("OK")

test file: wrote 213 lines
OK


In [36]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] Q1.1.1 / 01 perfect
  [PASS ] Q1.1.1 / 02 answer only
  [PASS ] Q1.1.1 / 03 factor only
  [PASS ] Q1.1.1 / 04 skip step 2
  [PASS ] Q1.1.1 / 05 memo order
  [PASS ] Q1.1.1 / 06 semicolon
  [PASS ] Q1.1.1 / 07 comma
  [PASS ] Q1.1.1 / 08 Therefore
  [PASS ] Q1.1.1 / 09 wrong factor
  [PASS ] Q1.1.1 / 10 out of order
  [PASS ] Q1.1.1 / 11 alt factor order
  [PASS ] Q1.1.1 / 12 therefore symbol
  [PASS ] Q1.1.1 / 13 comma-or mix
  [PASS ] Q1.1.1 / 14 one root
  [PASS ] Q1.1.1 / 15 wrong root
  [PASS ] Q1.1.1 / 16 mixed
  [PASS ] Q1.1.1 / 17 whitespace
  [PASS ] Q1.1.1 / 18 (x+4)=0
  [PASS ] Q1.1.1 / 19 So lead-in
  [PASS ] Q1.1.1 / 20 formula alt
           note: steps 1-2 UNCERTAIN
  [PASS ] Q1.1.2 / 01 perfect
  [PASS ] Q1.1.2 / 02 one line
  [PASS ] Q1.1.2 / 03 std form only
  [PASS ] Q1.1.2 / 04 std + formula
  [PASS ] Q1.1.2 / 05 answers only
  [PASS ] Q1.1.2 / 06 one root
  [PASS ] Q1.1.2 / 07 wrong rounding
  [PASS ] Q1.1.2 / 08 comma decimal
  [PASS ] Q1.1.3 / 01 perfect

In [37]:
from pathlib import Path

def find_project_root(start):
    p = start.resolve()
    for _ in range(10):
        if (p / ".git").is_dir() or (p / "data" / "processed").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("project root not found")

ROOT = find_project_root(Path.cwd())
csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

old_line = '2023_P1_Q2.1.1,2,Substitute into T_n formula,1,"T91=7+(91-1)(5)","T91=7+(91-1)(5)|T_91=7+(91-1)(5)|T91=7+90*5",sequences.arithmetic,arith,A,false'
new_line = '2023_P1_Q2.1.1,2,Substitute into T_n formula,1,"T91=7+(91-1)(5)","T91=7+(91-1)(5)|T_91=7+(91-1)(5)|T91=7+90*5|Tn=7+(n-1)(5)|T_n=7+(n-1)(5)|Tn=5n+2|T_n=5n+2",sequences.arithmetic,arith,A,false'

src = csv_path.read_text(encoding="utf-8")
assert old_line in src, "old line not found — check exact content"

csv_path.write_text(src.replace(old_line, new_line), encoding="utf-8")
print("CSV patched.")
print(f"Total lines: {len(csv_path.read_text(encoding='utf-8').splitlines())}")

CSV patched.
Total lines: 55


In [38]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] Q1.1.1 / 01 perfect
  [PASS ] Q1.1.1 / 02 answer only
  [PASS ] Q1.1.1 / 03 factor only
  [PASS ] Q1.1.1 / 04 skip step 2
  [PASS ] Q1.1.1 / 05 memo order
  [PASS ] Q1.1.1 / 06 semicolon
  [PASS ] Q1.1.1 / 07 comma
  [PASS ] Q1.1.1 / 08 Therefore
  [PASS ] Q1.1.1 / 09 wrong factor
  [PASS ] Q1.1.1 / 10 out of order
  [PASS ] Q1.1.1 / 11 alt factor order
  [PASS ] Q1.1.1 / 12 therefore symbol
  [PASS ] Q1.1.1 / 13 comma-or mix
  [PASS ] Q1.1.1 / 14 one root
  [PASS ] Q1.1.1 / 15 wrong root
  [PASS ] Q1.1.1 / 16 mixed
  [PASS ] Q1.1.1 / 17 whitespace
  [PASS ] Q1.1.1 / 18 (x+4)=0
  [PASS ] Q1.1.1 / 19 So lead-in
  [PASS ] Q1.1.1 / 20 formula alt
           note: steps 1-2 UNCERTAIN
  [PASS ] Q1.1.2 / 01 perfect
  [PASS ] Q1.1.2 / 02 one line
  [PASS ] Q1.1.2 / 03 std form only
  [PASS ] Q1.1.2 / 04 std + formula
  [PASS ] Q1.1.2 / 05 answers only
  [PASS ] Q1.1.2 / 06 one root
  [PASS ] Q1.1.2 / 07 wrong rounding
  [PASS ] Q1.1.2 / 08 comma decimal
  [PASS ] Q1.1.3 / 01 perfect

In [40]:
test_source = r'''"""
Test harness for step_grader.py
"""
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.tutor.step_grader import grade_steps

CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

CASES = [
    # ================= Q1 =================
    ("Q1.1.1 / 01 perfect", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.1 / 02 answer only", "2023_P1_Q1.1.1", ["x=-4 or x=3"], 1, [1, 2], "", False),
    ("Q1.1.2 / 01 perfect", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79", "x=-1.12"], 4, [], "", False),
    ("Q1.1.3 / 01 perfect", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=4"], 4, [], "", False),
    ("Q1.1.4 / 01 perfect", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "x<-1 or x>3"], 4, [], "", False),
    ("Q1.2 / 01 method A", "2023_P1_Q1.2",
     ["x=2y-2", "1/(2y-2)+1/y=1", "2y^2-5y+2=0", "y=1/2 or y=2", "x=-1 or x=2"],
     5, [], "", False),
    ("Q1.2 / 02 method B", "2023_P1_Q1.2",
     ["y=x/2+1", "1/x+2/(x+2)=1", "x^2-x-2=0", "x=-1 or x=2", "y=1/2 or y=2"],
     5, [], "", False),
    ("Q1.3 / 01 perfect", "2023_P1_Q1.3",
     ["2^m(2+1)=3^n(3^2-1)", "2^m(3)=3^n(2^3)", "m=3 and n=1", "m+n=4"],
     4, [], "", False),

    # ================= Q2 =================
    ("Q2.1.1 / 01 perfect", "2023_P1_Q2.1.1",
     ["d=5", "T91=7+(91-1)(5)", "T91=457"], 3, [], "", False),
    ("Q2.1.1 / 02 Tn variant", "2023_P1_Q2.1.1",
     ["d=5", "Tn=7+(n-1)(5)", "T91=457"], 3, [], "", False),
    ("Q2.1.2 / 01 perfect", "2023_P1_Q2.1.2",
     ["S91=91/2[2(7)+(91-1)(5)]", "S91=21112"], 2, [], "", False),
    ("Q2.1.3 / 01 perfect", "2023_P1_Q2.1.3",
     ["Tn=7+(n-1)(5)", "Tn=517", "n=103"], 3, [], "", False),
    ("Q2.2.1 / 01 perfect", "2023_P1_Q2.2.1",
     ["9; 21; 33; 45", "T5=111"], 2, [], "", False),
    ("Q2.2.2 / 01 perfect", "2023_P1_Q2.2.2",
     ["2a=12", "3(6)+b=9", "Tn=6n^2-9n+6"], 3, [], "", False),
    ("Q2.2.3 / 01 perfect", "2023_P1_Q2.2.3",
     ["12n-9=0", "n=3/4", "T is increasing for n in N"], 3, [], "", False),

    # ================= Q3 =================
    ("Q3.1.1 / 01 perfect", "2023_P1_Q3.1.1", ["Tn=3(2)^(n-1)"], 1, [], "", False),
    ("Q3.1.2 / 01 perfect", "2023_P1_Q3.1.2",
     ["3+6+12+...", "n=k", "3(2^k-1)=98301", "k=15"], 4, [], "", False),
    ("Q3.2 / 01 perfect", "2023_P1_Q3.2",
     ["S22=22/2[2a+21(3)]", "S22=22a+693", "S_inf=3a/2", "22a+693=3a/2+734", "a=2"],
     5, [], "", False),

    # ================= Q4 =================
    ("Q4.1 / 01 perfect", "2023_P1_Q4.1", ["y=-4"], 1, [], "", False),
    ("Q4.2 / 01 perfect", "2023_P1_Q4.2", ["0=2^x-4", "x=2"], 2, [], "", False),
    ("Q4.2 / 02 answer only", "2023_P1_Q4.2", ["x=2"], 1, [1], "", False),
    ("Q4.3 / 01 perfect", "2023_P1_Q4.3",
     ["y=2^0-4=-3", "m=3/2", "k(x)=3/2x-3"], 3, [], "", False),
    ("Q4.3 / 02 answer only", "2023_P1_Q4.3", ["k(x)=3/2x-3"], 1, [1, 2], "", False),
    ("Q4.4 / 01 perfect", "2023_P1_Q4.4",
     ["k(1)=-3/2", "f(1)=-2", "1/2"], 3, [], "", False),
    ("Q4.5 / 01 perfect", "2023_P1_Q4.5", ["g(x)=2^x"], 1, [], "", False),
    ("Q4.6 / 01 perfect", "2023_P1_Q4.6", ["x in [1/4;16]"], 1, [], "", False),
    ("Q4.7 / 01 perfect", "2023_P1_Q4.7",
     ["x=2^y", "g^-1(x)=log_2 x"], 2, [], "", False),

    # ================= Q5 =================
    ("Q5.1 / 01 perfect", "2023_P1_Q5.1", ["x=1", "y=8"], 2, [], "", False),
    ("Q5.2 / 01 perfect", "2023_P1_Q5.2",
     ["y=-1/2(0-1)^2+8", "C(0;15/2)"], 2, [], "", False),
    ("Q5.3 / 01 perfect", "2023_P1_Q5.3", ["8=d/1"], 1, [], "", False),
    ("Q5.4 / 01 perfect", "2023_P1_Q5.4", ["y != 0"], 1, [], "", False),
    ("Q5.5 / 01 perfect", "2023_P1_Q5.5",
     ["-3<=x<0", "x>=5", "-3<=x<0 or x>=5"], 3, [], "", False),
    ("Q5.6 / 01 method A", "2023_P1_Q5.6",
     ["-2x+k=8/x", "-2x^2+kx-8=0", "k^2-64<0", "-8<k<8"], 4, [], "", False),
    ("Q5.6 / 02 method B", "2023_P1_Q5.6",
     ["-8/x^2=-2", "x=2 or x=-2", "k=8", "-8<k<8"], 4, [], "", False),
    ("Q5.7 / 01 perfect", "2023_P1_Q5.7",
     ["-2x+8=8/x", "(x-2)^2=0", "f(2)=15/2; h(2)=4", "t=-7/2"], 4, [], "", False),
]


def run() -> int:
    passed = failed = 0
    out = []
    for name, qid, working, exp_awarded, exp_missing, note, _xfail in CASES:
        try:
            r = grade_steps(qid, working, CSV)
            awarded = r["awarded_marks"]
            missing = sorted(s["index"] for s in r["steps"] if s["match_status"] == "NOT_MATCHED")
            ok = (awarded == exp_awarded) and (missing == sorted(exp_missing))
        except Exception as e:
            awarded = missing = None
            ok = False
            note = "EXCEPTION: " + str(e) + " | " + note

        if ok:
            passed += 1
            status = "PASS "
        else:
            failed += 1
            status = "FAIL "

        out.append("  [" + status + "] " + name)
        if note:
            out.append("           note: " + note)
        if not ok:
            out.append("           expected " + str(exp_awarded) + " missing=" + str(sorted(exp_missing)))
            out.append("           got      " + str(awarded) + " missing=" + str(missing))

    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
                      CSV)
    statuses = [s["match_status"] for s in r20["steps"]]
    if statuses == ["UNCERTAIN", "UNCERTAIN", "MATCHED"]:
        passed += 1
        out.append("  [PASS ] assertion / Q1.1.1 UNCERTAIN")
    else:
        failed += 1
        out.append("  [FAIL ] assertion / Q1.1.1 UNCERTAIN")

    print("\n".join(out))
    print()
    print("  " + str(passed) + " passed  |  " + str(failed) + " failed  |  "
          + str(len(CASES) + 1) + " total")
    return failed


if __name__ == "__main__":
    sys.exit(1 if run() else 0)
'''

tt_path = ROOT / "tests" / "test_step_grader.py"
tt_path.write_text(test_source, encoding="utf-8")
print(f"test file: wrote {len(tt_path.read_text(encoding='utf-8').splitlines())} lines")
print("OK")

test file: wrote 138 lines
OK


In [41]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] Q1.1.1 / 01 perfect
  [PASS ] Q1.1.1 / 02 answer only
  [PASS ] Q1.1.2 / 01 perfect
  [PASS ] Q1.1.3 / 01 perfect
  [PASS ] Q1.1.4 / 01 perfect
  [PASS ] Q1.2 / 01 method A
  [PASS ] Q1.2 / 02 method B
  [PASS ] Q1.3 / 01 perfect
  [PASS ] Q2.1.1 / 01 perfect
  [PASS ] Q2.1.1 / 02 Tn variant
  [PASS ] Q2.1.2 / 01 perfect
  [PASS ] Q2.1.3 / 01 perfect
  [PASS ] Q2.2.1 / 01 perfect
  [PASS ] Q2.2.2 / 01 perfect
  [PASS ] Q2.2.3 / 01 perfect
  [PASS ] Q3.1.1 / 01 perfect
  [PASS ] Q3.1.2 / 01 perfect
  [PASS ] Q3.2 / 01 perfect
  [PASS ] Q4.1 / 01 perfect
  [PASS ] Q4.2 / 01 perfect
  [PASS ] Q4.2 / 02 answer only
  [FAIL ] Q4.3 / 01 perfect
           expected 3 missing=[]
           got      2 missing=[1]
  [PASS ] Q4.3 / 02 answer only
  [PASS ] Q4.4 / 01 perfect
  [PASS ] Q4.5 / 01 perfect
  [PASS ] Q4.6 / 01 perfect
  [PASS ] Q4.7 / 01 perfect
  [PASS ] Q5.1 / 01 perfect
  [PASS ] Q5.2 / 01 perfect
  [PASS ] Q5.3 / 01 perfect
  [PASS ] Q5.4 / 01 perfect
  [PASS ] Q5.5 / 01 

In [42]:
from pathlib import Path

def find_project_root(start):
    p = start.resolve()
    for _ in range(10):
        if (p / ".git").is_dir() or (p / "data" / "processed").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("project root not found")

ROOT = find_project_root(Path.cwd())
csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

old = '2023_P1_Q4.3,1,Compute y-intercept,1,y=2^0-4=-3,"y=-3|A(0;-3)|A(0,-3)",functions.exponential,exponential,A,false'
new = '2023_P1_Q4.3,1,Compute y-intercept,1,y=2^0-4=-3,"y=-3|y=2^0-4=-3|y=2^0-4|A(0;-3)|A(0,-3)",functions.exponential,exponential,A,false'

src = csv_path.read_text(encoding="utf-8")
assert old in src, "old line not found"

csv_path.write_text(src.replace(old, new), encoding="utf-8")
print("CSV patched.")
print(f"Total lines: {len(csv_path.read_text(encoding='utf-8').splitlines())}")

CSV patched.
Total lines: 89


In [43]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] Q1.1.1 / 01 perfect
  [PASS ] Q1.1.1 / 02 answer only
  [PASS ] Q1.1.2 / 01 perfect
  [PASS ] Q1.1.3 / 01 perfect
  [PASS ] Q1.1.4 / 01 perfect
  [PASS ] Q1.2 / 01 method A
  [PASS ] Q1.2 / 02 method B
  [PASS ] Q1.3 / 01 perfect
  [PASS ] Q2.1.1 / 01 perfect
  [PASS ] Q2.1.1 / 02 Tn variant
  [PASS ] Q2.1.2 / 01 perfect
  [PASS ] Q2.1.3 / 01 perfect
  [PASS ] Q2.2.1 / 01 perfect
  [PASS ] Q2.2.2 / 01 perfect
  [PASS ] Q2.2.3 / 01 perfect
  [PASS ] Q3.1.1 / 01 perfect
  [PASS ] Q3.1.2 / 01 perfect
  [PASS ] Q3.2 / 01 perfect
  [PASS ] Q4.1 / 01 perfect
  [PASS ] Q4.2 / 01 perfect
  [PASS ] Q4.2 / 02 answer only
  [PASS ] Q4.3 / 01 perfect
  [PASS ] Q4.3 / 02 answer only
  [PASS ] Q4.4 / 01 perfect
  [PASS ] Q4.5 / 01 perfect
  [PASS ] Q4.6 / 01 perfect
  [PASS ] Q4.7 / 01 perfect
  [PASS ] Q5.1 / 01 perfect
  [PASS ] Q5.2 / 01 perfect
  [PASS ] Q5.3 / 01 perfect
  [PASS ] Q5.4 / 01 perfect
  [PASS ] Q5.5 / 01 perfect
  [PASS ] Q5.6 / 01 method A
  [PASS ] Q5.6 / 02 method B
 

In [44]:
new_rows = '''2023_P1_Q6.1.1,1,State n,1,n=6,n=6,finance.compound,finance,A,false
2023_P1_Q6.1.1,2,Substitute into A=P(1+i)^n,1,"19319.48=18500(1+r/1200)^6","19319.48=18500(1+r/1200)^6|19319.48=18500(1+r/1200)6",finance.compound,finance,A,false
2023_P1_Q6.1.1,3,Solve for r,1,r=8.7%,"r=8.7%|r=8.7|r=0.087",finance.compound,finance,A,true
2023_P1_Q6.1.2,1,Substitute into nominal rate formula,1,"1+i/100=(1+8.7/1200)^12","1+i/100=(1+8.7/1200)^12|1+i/100=(1+8.7/1200)12",finance.compound,finance,A,false
2023_P1_Q6.1.2,2,Solve for effective rate,1,r=9.06%,"r=9.06%|r=9.06|r=0.0906",finance.compound,finance,A,true
2023_P1_Q6.2.1,1,Substitute into A=P(1-in),1,"0=10000(1-0.2n)","0=10000(1-0.2n)",finance.simple,finance,A,false
2023_P1_Q6.2.1,2,Solve for n,1,n=5,n=5,finance.simple,finance,A,true
2023_P1_Q6.2.2,1,State i,1,i=8.7/1200,i=8.7/1200,finance.future_value,finance,A,false
2023_P1_Q6.2.2,2,State n,1,n=60,n=60,finance.future_value,finance,A,false
2023_P1_Q6.2.2,3,Substitute into F formula,1,"20000=x[(1+8.7/1200)^60-1]/(8.7/1200)","20000=x[(1+8.7/1200)^60-1]/(8.7/1200)",finance.future_value,finance,A,false
2023_P1_Q6.2.2,4,Solve for x,1,x=267.26,"x=267.26|x=R267.26|x=267",finance.future_value,finance,A,true
2023_P1_Q6.3,1,State i,1,i=0.112/12,i=0.112/12,finance.present_value,finance,A,false
2023_P1_Q6.3,2,Substitute into P formula,1,"1600000=20000[1-(1+0.112/12)^-n]/(0.112/12)","1600000=20000[1-(1+0.112/12)^-n]/(0.112/12)",finance.present_value,finance,A,false
2023_P1_Q6.3,3,Use logs,1,"-n=log_(1+0.112/12)(19/75)","-n=log_(1+0.112/12)(19/75)|log_(1+0.112/12)(19/75)|-n=log(1+0.112/12)^(19/75)",finance.present_value,finance,A,false
2023_P1_Q6.3,4,Compute -n,1,-n=-147.80,-n=-147.80,finance.present_value,finance,A,false
2023_P1_Q6.3,5,Answer n,1,n=147,"n=147|147 withdrawals|147",finance.present_value,finance,A,true
2023_P1_Q7.1,1,Substitute into first-principles formula,1,"f'(x)=lim(f(x+h)-f(x))/h","f'(x)=lim(f(x+h)-f(x))/h|lim(f(x+h)-f(x))/h|f'(x)=lim (f(x+h)-f(x))/h",calculus.first_principles,first_principles,A,false
2023_P1_Q7.1,2,Expand f(x+h),1,"f(x+h)=-4x^2-8xh-4h^2","f(x+h)=-4x^2-8xh-4h^2|f(x+h)=-4x2-8xh-4h2",calculus.first_principles,first_principles,A,false
2023_P1_Q7.1,3,Simplify numerator,1,"-8xh-4h^2","-8xh-4h^2|-8xh-4h2",calculus.first_principles,first_principles,A,false
2023_P1_Q7.1,4,Factor out h,1,"h(-8x-4h)","h(-8x-4h)",calculus.first_principles,first_principles,A,false
2023_P1_Q7.1,5,Final answer,1,-8x,-8x,calculus.first_principles,first_principles,A,true
2023_P1_Q7.2.1,1,Derivative of f,2,"f'(x)=6x^2-3","f'(x)=6x^2-3|6x^2-3|6x^2 - 3",calculus.polynomial,derivative,A,true
2023_P1_Q7.2.2,1,Rewrite as rational exponent,1,x^(2/3),x^(2/3),calculus.polynomial,derivative,A,false
2023_P1_Q7.2.2,2,Derivative of first term,1,(14/3)x^(-1/3),(14/3)x^(-1/3),calculus.polynomial,derivative,A,false
2023_P1_Q7.2.2,3,Derivative of second term,1,-10x^(-6),"-10x^(-6)|-10x^-6",calculus.polynomial,derivative,A,true
2023_P1_Q7.3,1,Critical values,1,"x=-2/sqrt(3) or x=2/sqrt(3)","x=-2/sqrt(3) or x=2/sqrt(3)|x=2/sqrt(3) or x=-2/sqrt(3)|x=±2/sqrt(3)",calculus.inequality,inequality,A,false
2023_P1_Q7.3,2,Interval answer,2,"-2/sqrt(3)<x<2/sqrt(3)","-2/sqrt(3)<x<2/sqrt(3)|-2/sqrt(3) < x < 2/sqrt(3)",calculus.inequality,inequality,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"
existing = csv_path.read_text(encoding="utf-8").rstrip("\n")
before = len(existing.splitlines())
csv_path.write_text(existing + "\n" + new_rows, encoding="utf-8")

lines = csv_path.read_text(encoding="utf-8").splitlines()
added = len(lines) - before
print(f"CSV: {before} -> {len(lines)} lines (added {added})")
assert added == 27, f"Expected 27 new rows, added {added}"
print("OK")

CSV: 89 -> 116 lines (added 27)
OK


In [45]:
test_source = r'''"""
Test harness for step_grader.py
"""
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.tutor.step_grader import grade_steps

CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

CASES = [
    # ================= Q1 =================
    ("Q1.1.1 / 01 perfect", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.2 / 01 perfect", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79", "x=-1.12"], 4, [], "", False),
    ("Q1.1.3 / 01 perfect", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=4"], 4, [], "", False),
    ("Q1.1.4 / 01 perfect", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "x<-1 or x>3"], 4, [], "", False),
    ("Q1.2 / 01 method A", "2023_P1_Q1.2",
     ["x=2y-2", "1/(2y-2)+1/y=1", "2y^2-5y+2=0", "y=1/2 or y=2", "x=-1 or x=2"],
     5, [], "", False),
    ("Q1.2 / 02 method B", "2023_P1_Q1.2",
     ["y=x/2+1", "1/x+2/(x+2)=1", "x^2-x-2=0", "x=-1 or x=2", "y=1/2 or y=2"],
     5, [], "", False),
    ("Q1.3 / 01 perfect", "2023_P1_Q1.3",
     ["2^m(2+1)=3^n(3^2-1)", "2^m(3)=3^n(2^3)", "m=3 and n=1", "m+n=4"],
     4, [], "", False),

    # ================= Q2 =================
    ("Q2.1.1 / 01 perfect", "2023_P1_Q2.1.1",
     ["d=5", "T91=7+(91-1)(5)", "T91=457"], 3, [], "", False),
    ("Q2.1.1 / 02 Tn variant", "2023_P1_Q2.1.1",
     ["d=5", "Tn=7+(n-1)(5)", "T91=457"], 3, [], "", False),
    ("Q2.1.2 / 01 perfect", "2023_P1_Q2.1.2",
     ["S91=91/2[2(7)+(91-1)(5)]", "S91=21112"], 2, [], "", False),
    ("Q2.1.3 / 01 perfect", "2023_P1_Q2.1.3",
     ["Tn=7+(n-1)(5)", "Tn=517", "n=103"], 3, [], "", False),
    ("Q2.2.2 / 01 perfect", "2023_P1_Q2.2.2",
     ["2a=12", "3(6)+b=9", "Tn=6n^2-9n+6"], 3, [], "", False),
    ("Q2.2.3 / 01 perfect", "2023_P1_Q2.2.3",
     ["12n-9=0", "n=3/4", "T is increasing for n in N"], 3, [], "", False),

    # ================= Q3 =================
    ("Q3.1.2 / 01 perfect", "2023_P1_Q3.1.2",
     ["3+6+12+...", "n=k", "3(2^k-1)=98301", "k=15"], 4, [], "", False),
    ("Q3.2 / 01 perfect", "2023_P1_Q3.2",
     ["S22=22/2[2a+21(3)]", "S22=22a+693", "S_inf=3a/2", "22a+693=3a/2+734", "a=2"],
     5, [], "", False),

    # ================= Q4 =================
    ("Q4.1 / 01 perfect", "2023_P1_Q4.1", ["y=-4"], 1, [], "", False),
    ("Q4.2 / 01 perfect", "2023_P1_Q4.2", ["0=2^x-4", "x=2"], 2, [], "", False),
    ("Q4.3 / 01 perfect", "2023_P1_Q4.3",
     ["y=2^0-4=-3", "m=3/2", "k(x)=3/2x-3"], 3, [], "", False),
    ("Q4.4 / 01 perfect", "2023_P1_Q4.4",
     ["k(1)=-3/2", "f(1)=-2", "1/2"], 3, [], "", False),
    ("Q4.7 / 01 perfect", "2023_P1_Q4.7",
     ["x=2^y", "g^-1(x)=log_2 x"], 2, [], "", False),

    # ================= Q5 =================
    ("Q5.1 / 01 perfect", "2023_P1_Q5.1", ["x=1", "y=8"], 2, [], "", False),
    ("Q5.5 / 01 perfect", "2023_P1_Q5.5",
     ["-3<=x<0", "x>=5", "-3<=x<0 or x>=5"], 3, [], "", False),
    ("Q5.6 / 01 method A", "2023_P1_Q5.6",
     ["-2x+k=8/x", "-2x^2+kx-8=0", "k^2-64<0", "-8<k<8"], 4, [], "", False),
    ("Q5.6 / 02 method B", "2023_P1_Q5.6",
     ["-8/x^2=-2", "x=2 or x=-2", "k=8", "-8<k<8"], 4, [], "", False),
    ("Q5.7 / 01 perfect", "2023_P1_Q5.7",
     ["-2x+8=8/x", "(x-2)^2=0", "f(2)=15/2; h(2)=4", "t=-7/2"], 4, [], "", False),

    # ================= Q6 (finance) =================
    ("Q6.1.1 / 01 perfect", "2023_P1_Q6.1.1",
     ["n=6", "19319.48=18500(1+r/1200)^6", "r=8.7%"], 3, [], "", False),
    ("Q6.1.1 / 02 answer only", "2023_P1_Q6.1.1", ["r=8.7%"], 1, [1, 2], "", False),
    ("Q6.1.2 / 01 perfect", "2023_P1_Q6.1.2",
     ["1+i/100=(1+8.7/1200)^12", "r=9.06%"], 2, [], "", False),
    ("Q6.2.1 / 01 perfect", "2023_P1_Q6.2.1",
     ["0=10000(1-0.2n)", "n=5"], 2, [], "", False),
    ("Q6.2.2 / 01 perfect", "2023_P1_Q6.2.2",
     ["i=8.7/1200", "n=60", "20000=x[(1+8.7/1200)^60-1]/(8.7/1200)", "x=267.26"],
     4, [], "", False),
    ("Q6.3 / 01 perfect", "2023_P1_Q6.3",
     ["i=0.112/12", "1600000=20000[1-(1+0.112/12)^-n]/(0.112/12)",
      "-n=log_(1+0.112/12)(19/75)", "-n=-147.80", "n=147"],
     5, [], "", False),
    ("Q6.3 / 02 answer only", "2023_P1_Q6.3", ["n=147"], 1, [1, 2, 3, 4], "", False),

    # ================= Q7 (calculus) =================
    ("Q7.1 / 01 perfect", "2023_P1_Q7.1",
     ["f'(x)=lim(f(x+h)-f(x))/h", "f(x+h)=-4x^2-8xh-4h^2",
      "-8xh-4h^2", "h(-8x-4h)", "-8x"], 5, [], "", False),
    ("Q7.1 / 02 answer only", "2023_P1_Q7.1", ["-8x"], 1, [1, 2, 3, 4], "", False),
    ("Q7.1 / 03 no final", "2023_P1_Q7.1",
     ["f'(x)=lim(f(x+h)-f(x))/h", "f(x+h)=-4x^2-8xh-4h^2",
      "-8xh-4h^2", "h(-8x-4h)"], 4, [5], "", False),
    ("Q7.2.1 / 01 perfect", "2023_P1_Q7.2.1", ["f'(x)=6x^2-3"], 2, [], "", False),
    ("Q7.2.2 / 01 perfect", "2023_P1_Q7.2.2",
     ["x^(2/3)", "(14/3)x^(-1/3)", "-10x^(-6)"], 3, [], "", False),
    ("Q7.3 / 01 perfect", "2023_P1_Q7.3",
     ["x=-2/sqrt(3) or x=2/sqrt(3)", "-2/sqrt(3)<x<2/sqrt(3)"], 3, [], "", False),
    ("Q7.3 / 02 answer only", "2023_P1_Q7.3",
     ["-2/sqrt(3)<x<2/sqrt(3)"], 2, [1], "", False),
]


def run() -> int:
    passed = failed = 0
    out = []
    for name, qid, working, exp_awarded, exp_missing, note, _xfail in CASES:
        try:
            r = grade_steps(qid, working, CSV)
            awarded = r["awarded_marks"]
            missing = sorted(s["index"] for s in r["steps"] if s["match_status"] == "NOT_MATCHED")
            ok = (awarded == exp_awarded) and (missing == sorted(exp_missing))
        except Exception as e:
            awarded = missing = None
            ok = False
            note = "EXCEPTION: " + str(e) + " | " + note

        if ok:
            passed += 1
            status = "PASS "
        else:
            failed += 1
            status = "FAIL "

        out.append("  [" + status + "] " + name)
        if note:
            out.append("           note: " + note)
        if not ok:
            out.append("           expected " + str(exp_awarded) + " missing=" + str(sorted(exp_missing)))
            out.append("           got      " + str(awarded) + " missing=" + str(missing))

    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
                      CSV)
    statuses = [s["match_status"] for s in r20["steps"]]
    if statuses == ["UNCERTAIN", "UNCERTAIN", "MATCHED"]:
        passed += 1
        out.append("  [PASS ] assertion / Q1.1.1 UNCERTAIN")
    else:
        failed += 1
        out.append("  [FAIL ] assertion / Q1.1.1 UNCERTAIN")

    print("\n".join(out))
    print()
    print("  " + str(passed) + " passed  |  " + str(failed) + " failed  |  "
          + str(len(CASES) + 1) + " total")
    return failed


if __name__ == "__main__":
    sys.exit(1 if run() else 0)
'''

tt_path = ROOT / "tests" / "test_step_grader.py"
tt_path.write_text(test_source, encoding="utf-8")
print(f"test file: wrote {len(tt_path.read_text(encoding='utf-8').splitlines())} lines")
print("OK")

test file: wrote 159 lines
OK


In [46]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] Q1.1.1 / 01 perfect
  [PASS ] Q1.1.2 / 01 perfect
  [PASS ] Q1.1.3 / 01 perfect
  [PASS ] Q1.1.4 / 01 perfect
  [PASS ] Q1.2 / 01 method A
  [PASS ] Q1.2 / 02 method B
  [PASS ] Q1.3 / 01 perfect
  [PASS ] Q2.1.1 / 01 perfect
  [PASS ] Q2.1.1 / 02 Tn variant
  [PASS ] Q2.1.2 / 01 perfect
  [PASS ] Q2.1.3 / 01 perfect
  [PASS ] Q2.2.2 / 01 perfect
  [PASS ] Q2.2.3 / 01 perfect
  [PASS ] Q3.1.2 / 01 perfect
  [PASS ] Q3.2 / 01 perfect
  [PASS ] Q4.1 / 01 perfect
  [PASS ] Q4.2 / 01 perfect
  [PASS ] Q4.3 / 01 perfect
  [PASS ] Q4.4 / 01 perfect
  [PASS ] Q4.7 / 01 perfect
  [PASS ] Q5.1 / 01 perfect
  [PASS ] Q5.5 / 01 perfect
  [PASS ] Q5.6 / 01 method A
  [PASS ] Q5.6 / 02 method B
  [PASS ] Q5.7 / 01 perfect
  [PASS ] Q6.1.1 / 01 perfect
  [PASS ] Q6.1.1 / 02 answer only
  [PASS ] Q6.1.2 / 01 perfect
  [PASS ] Q6.2.1 / 01 perfect
  [PASS ] Q6.2.2 / 01 perfect
  [PASS ] Q6.3 / 01 perfect
  [PASS ] Q6.3 / 02 answer only
  [PASS ] Q7.1 / 01 perfect
  [PASS ] Q7.1 / 02 answer on

In [47]:
step_grader_source = r'''from __future__ import annotations

import csv
import re
from pathlib import Path
from typing import Any

ROOT = Path(__file__).resolve().parents[2]
DEFAULT_CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"


def load_steps(csv_path: Path | None = None) -> dict[str, list[dict[str, Any]]]:
    path = csv_path or DEFAULT_CSV
    by_q: dict[str, list[dict[str, Any]]] = {}
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            qid = row["question_id"].strip()
            mg = (row.get("method_group") or "A").strip() or "A"
            by_q.setdefault(qid, []).append({
                "index": int(row["step_index"]),
                "desc": row["step_description"].strip(),
                "marks": int(row["marks"]),
                "memo_line": row["memo_line"].strip(),
                "forms": [x.strip() for x in row["acceptable_forms"].split("|") if x.strip()],
                "skill_id": row["skill_id"].strip(),
                "method_tag": row["method_tag"].strip(),
                "method_group": mg,
                "is_final": str(row.get("is_final", "false")).strip().lower() == "true",
            })
    for qid in by_q:
        by_q[qid].sort(key=lambda s: (s["method_group"], s["index"]))
    return by_q


_SEP = re.compile(r"\s*(?:,|\bor\b|;|\band\b)\s*")
_METHOD_MARKERS = re.compile(r"sqrt|\u00b1|\+/-|/2a|b2-4ac|b\^2|\\pm|\u221a")


def _norm(s: str) -> str:
    s = s.lower().strip()
    s = s.replace("\u00d7", "*").replace("\u2212", "-").replace("\u2013", "-")
    s = s.replace("\u2234", " ").replace("\u2235", " ")
    s = re.sub(r"^\s*(therefore|so|hence|thus)\b\s*", "", s)
    s = re.sub(r"(\d),(\d)", r"\1.\2", s)
    return s


def _tokens(s: str) -> list[str]:
    parts = _SEP.split(_norm(s))
    return sorted(re.sub(r"\s+", "", p) for p in parts if p.strip())


def _line_matches(line: str, forms: list[str], is_final: bool = False) -> bool:
    lt = _tokens(line)
    if not lt:
        return False
    lt_set = set(lt)
    for form in forms:
        ft = _tokens(form)
        if not ft:
            continue
        if is_final:
            if set(ft).issubset(lt_set):
                return True
        else:
            if ft == lt:
                return True
    return False


def _grade_group(steps, learner_lines):
    used: set[int] = set()
    results = []
    awarded = 0
    for step in steps:
        matched = False
        matched_line = None
        is_final = step.get("is_final", False)
        for i, line in enumerate(learner_lines):
            if i in used and not is_final:
                continue
            if _line_matches(line, step["forms"], is_final=is_final):
                matched = True
                matched_line = line
                if not is_final:
                    used.add(i)
                awarded += step["marks"]
                break
        results.append({
            "index": step["index"],
            "desc": step["desc"],
            "marks": step["marks"],
            "matched": matched,
            "learner_line": matched_line,
            "is_final": is_final,
        })
    return awarded, results, used


def grade_steps(question_id: str, learner_lines: list[str], csv_path: Path | None = None) -> dict[str, Any]:
    steps_map = load_steps(csv_path)
    steps = steps_map.get(question_id, [])
    if not steps:
        return {
            "question_id": question_id,
            "total_marks": 0,
            "awarded_marks": 0,
            "steps": [],
            "feedback": "No step scheme for " + question_id,
        }

    groups: dict[str, list] = {}
    for s in steps:
        groups.setdefault(s["method_group"], []).append(s)

    best_awarded = -1
    best_results = None
    best_group = None
    for g_name, g_steps in groups.items():
        awarded, results, used = _grade_group(g_steps, learner_lines)
        if awarded > best_awarded:
            best_awarded = awarded
            best_results = results
            best_group = g_steps

    results = best_results
    awarded = best_awarded
    g_steps = best_group

    # UNCERTAIN heuristic: only look at learner lines that no step matched.
    # Uses learner_line values from results, so lines consumed by final steps
    # (which don't add to `used`) are still excluded from the "leftover" check.
    matched_values = {r["learner_line"] for r in results if r["matched"]}
    unmatched_lines = [ln for ln in learner_lines if ln not in matched_values]

    final_matched = any(r["matched"] and r["is_final"] for r in results)
    has_method_evidence = any(_METHOD_MARKERS.search(ln) for ln in unmatched_lines)

    for r in results:
        if r["matched"]:
            r["match_status"] = "MATCHED"
        elif final_matched and has_method_evidence:
            r["match_status"] = "UNCERTAIN"
        else:
            r["match_status"] = "NOT_MATCHED"

    total = sum(s["marks"] for s in g_steps)
    missing = [r for r in results if r["match_status"] == "NOT_MATCHED"]
    uncertain = [r for r in results if r["match_status"] == "UNCERTAIN"]

    if not missing and not uncertain:
        feedback = "Full marks: %d/%d" % (awarded, total)
    elif uncertain and not missing:
        bits = ["cannot verify '%s' (alternate method used)" % u["desc"] for u in uncertain]
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)
    else:
        bits = ["missed '%s' (%d mark)" % (m["desc"], m["marks"]) for m in missing]
        if uncertain:
            bits.append("cannot verify " + ", ".join("'%s'" % u["desc"] for u in uncertain))
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)

    return {
        "question_id": question_id,
        "total_marks": total,
        "awarded_marks": awarded,
        "steps": results,
        "feedback": feedback,
    }
'''

sg_path = ROOT / "src" / "tutor" / "step_grader.py"
sg_path.write_text(step_grader_source, encoding="utf-8")

print(f"step_grader.py: wrote {len(sg_path.read_text(encoding='utf-8').splitlines())} lines")
print("OK")

step_grader.py: wrote 168 lines
OK


In [48]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] Q1.1.1 / 01 perfect
  [PASS ] Q1.1.2 / 01 perfect
  [PASS ] Q1.1.3 / 01 perfect
  [PASS ] Q1.1.4 / 01 perfect
  [PASS ] Q1.2 / 01 method A
  [PASS ] Q1.2 / 02 method B
  [PASS ] Q1.3 / 01 perfect
  [PASS ] Q2.1.1 / 01 perfect
  [PASS ] Q2.1.1 / 02 Tn variant
  [PASS ] Q2.1.2 / 01 perfect
  [PASS ] Q2.1.3 / 01 perfect
  [PASS ] Q2.2.2 / 01 perfect
  [PASS ] Q2.2.3 / 01 perfect
  [PASS ] Q3.1.2 / 01 perfect
  [PASS ] Q3.2 / 01 perfect
  [PASS ] Q4.1 / 01 perfect
  [PASS ] Q4.2 / 01 perfect
  [PASS ] Q4.3 / 01 perfect
  [PASS ] Q4.4 / 01 perfect
  [PASS ] Q4.7 / 01 perfect
  [PASS ] Q5.1 / 01 perfect
  [PASS ] Q5.5 / 01 perfect
  [PASS ] Q5.6 / 01 method A
  [PASS ] Q5.6 / 02 method B
  [PASS ] Q5.7 / 01 perfect
  [PASS ] Q6.1.1 / 01 perfect
  [PASS ] Q6.1.1 / 02 answer only
  [PASS ] Q6.1.2 / 01 perfect
  [PASS ] Q6.2.1 / 01 perfect
  [PASS ] Q6.2.2 / 01 perfect
  [PASS ] Q6.3 / 01 perfect
  [PASS ] Q6.3 / 02 answer only
  [PASS ] Q7.1 / 01 perfect
  [PASS ] Q7.1 / 02 answer on

In [50]:
test_source = r'''"""
Test harness for step_grader.py
"""
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.tutor.step_grader import grade_steps

CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

CASES = [
    # ================= Q1 =================
    ("Q1.1.1 / 01 perfect", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.2 / 01 perfect", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79", "x=-1.12"], 4, [], "", False),
    ("Q1.1.3 / 01 perfect", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=4"], 4, [], "", False),
    ("Q1.1.4 / 01 perfect", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "x<-1 or x>3"], 4, [], "", False),
    ("Q1.2 / 01 method A", "2023_P1_Q1.2",
     ["x=2y-2", "1/(2y-2)+1/y=1", "2y^2-5y+2=0", "y=1/2 or y=2", "x=-1 or x=2"],
     5, [], "", False),
    ("Q1.2 / 02 method B", "2023_P1_Q1.2",
     ["y=x/2+1", "1/x+2/(x+2)=1", "x^2-x-2=0", "x=-1 or x=2", "y=1/2 or y=2"],
     5, [], "", False),
    ("Q1.3 / 01 perfect", "2023_P1_Q1.3",
     ["2^m(2+1)=3^n(3^2-1)", "2^m(3)=3^n(2^3)", "m=3 and n=1", "m+n=4"],
     4, [], "", False),

    # ================= Q2 =================
    ("Q2.1.1 / 01 perfect", "2023_P1_Q2.1.1",
     ["d=5", "T91=7+(91-1)(5)", "T91=457"], 3, [], "", False),
    ("Q2.1.1 / 02 Tn variant", "2023_P1_Q2.1.1",
     ["d=5", "Tn=7+(n-1)(5)", "T91=457"], 3, [], "", False),
    ("Q2.1.2 / 01 perfect", "2023_P1_Q2.1.2",
     ["S91=91/2[2(7)+(91-1)(5)]", "S91=21112"], 2, [], "", False),
    ("Q2.1.3 / 01 perfect", "2023_P1_Q2.1.3",
     ["Tn=7+(n-1)(5)", "Tn=517", "n=103"], 3, [], "", False),
    ("Q2.2.2 / 01 perfect", "2023_P1_Q2.2.2",
     ["2a=12", "3(6)+b=9", "Tn=6n^2-9n+6"], 3, [], "", False),
    ("Q2.2.3 / 01 perfect", "2023_P1_Q2.2.3",
     ["12n-9=0", "n=3/4", "T is increasing for n in N"], 3, [], "", False),

    # ================= Q3 =================
    ("Q3.1.2 / 01 perfect", "2023_P1_Q3.1.2",
     ["3+6+12+...", "n=k", "3(2^k-1)=98301", "k=15"], 4, [], "", False),
    ("Q3.2 / 01 perfect", "2023_P1_Q3.2",
     ["S22=22/2[2a+21(3)]", "S22=22a+693", "S_inf=3a/2", "22a+693=3a/2+734", "a=2"],
     5, [], "", False),

    # ================= Q4 =================
    ("Q4.1 / 01 perfect", "2023_P1_Q4.1", ["y=-4"], 1, [], "", False),
    ("Q4.2 / 01 perfect", "2023_P1_Q4.2", ["0=2^x-4", "x=2"], 2, [], "", False),
    ("Q4.3 / 01 perfect", "2023_P1_Q4.3",
     ["y=2^0-4=-3", "m=3/2", "k(x)=3/2x-3"], 3, [], "", False),
    ("Q4.4 / 01 perfect", "2023_P1_Q4.4",
     ["k(1)=-3/2", "f(1)=-2", "1/2"], 3, [], "", False),
    ("Q4.7 / 01 perfect", "2023_P1_Q4.7",
     ["x=2^y", "g^-1(x)=log_2 x"], 2, [], "", False),

    # ================= Q5 =================
    ("Q5.1 / 01 perfect", "2023_P1_Q5.1", ["x=1", "y=8"], 2, [], "", False),
    ("Q5.5 / 01 perfect", "2023_P1_Q5.5",
     ["-3<=x<0", "x>=5", "-3<=x<0 or x>=5"], 3, [], "", False),
    ("Q5.6 / 01 method A", "2023_P1_Q5.6",
     ["-2x+k=8/x", "-2x^2+kx-8=0", "k^2-64<0", "-8<k<8"], 4, [], "", False),
    ("Q5.6 / 02 method B", "2023_P1_Q5.6",
     ["-8/x^2=-2", "x=2 or x=-2", "k=8", "-8<k<8"], 4, [], "", False),
    ("Q5.7 / 01 perfect", "2023_P1_Q5.7",
     ["-2x+8=8/x", "(x-2)^2=0", "f(2)=15/2; h(2)=4", "t=-7/2"], 4, [], "", False),

    # ================= Q6 =================
    ("Q6.1.1 / 01 perfect", "2023_P1_Q6.1.1",
     ["n=6", "19319.48=18500(1+r/1200)^6", "r=8.7%"], 3, [], "", False),
    ("Q6.1.2 / 01 perfect", "2023_P1_Q6.1.2",
     ["1+i/100=(1+8.7/1200)^12", "r=9.06%"], 2, [], "", False),
    ("Q6.2.1 / 01 perfect", "2023_P1_Q6.2.1",
     ["0=10000(1-0.2n)", "n=5"], 2, [], "", False),
    ("Q6.2.2 / 01 perfect", "2023_P1_Q6.2.2",
     ["i=8.7/1200", "n=60", "20000=x[(1+8.7/1200)^60-1]/(8.7/1200)", "x=267.26"],
     4, [], "", False),
    ("Q6.3 / 01 perfect", "2023_P1_Q6.3",
     ["i=0.112/12", "1600000=20000[1-(1+0.112/12)^-n]/(0.112/12)",
      "-n=log_(1+0.112/12)(19/75)", "-n=-147.80", "n=147"],
     5, [], "", False),

    # ================= Q7 =================
    ("Q7.1 / 01 perfect", "2023_P1_Q7.1",
     ["f'(x)=lim(f(x+h)-f(x))/h", "f(x+h)=-4x^2-8xh-4h^2",
      "-8xh-4h^2", "h(-8x-4h)", "-8x"], 5, [], "", False),
    ("Q7.2.1 / 01 perfect", "2023_P1_Q7.2.1", ["f'(x)=6x^2-3"], 2, [], "", False),
    ("Q7.2.2 / 01 perfect", "2023_P1_Q7.2.2",
     ["x^(2/3)", "(14/3)x^(-1/3)", "-10x^(-6)"], 3, [], "", False),
    ("Q7.3 / 01 perfect", "2023_P1_Q7.3",
     ["x=-2/sqrt(3) or x=2/sqrt(3)", "-2/sqrt(3)<x<2/sqrt(3)"], 3, [], "", False),

    # ================= Q8 (cubic) =================
    ("Q8.1 / 01 perfect", "2023_P1_Q8.1",
     ["f'(x)=-3x^2+12x-9", "f'(x)=0", "x=3 or x=1", "(3;4) and (1;0)"],
     4, [], "", False),
    ("Q8.1 / 02 no y-values", "2023_P1_Q8.1",
     ["f'(x)=-3x^2+12x-9", "f'(x)=0", "x=3 or x=1"],
     3, [4], "", False),
    ("Q8.1 / 03 answer only", "2023_P1_Q8.1",
     ["(3;4) and (1;0)"], 1, [1, 2, 3], "", False),
    ("Q8.1 / 04 reversed points", "2023_P1_Q8.1",
     ["f'(x)=-3x^2+12x-9", "f'(x)=0", "x=3 or x=1", "(1;0) and (3;4)"],
     4, [], "", False),

    ("Q8.3 / 01 perfect", "2023_P1_Q8.3", ["0<k<4"], 2, [], "", False),
    ("Q8.3 / 02 wrong interval", "2023_P1_Q8.3", ["0<k<3"], 0, [1], "", False),

    ("Q8.4 / 01 perfect", "2023_P1_Q8.4",
     ["f''(x)=-6x+12", "f''(x)=0", "x=2", "y=2", "f'(2)=3", "g(x)=3x-4"],
     6, [], "", False),
    ("Q8.4 / 02 skip tangent eq", "2023_P1_Q8.4",
     ["f''(x)=-6x+12", "f''(x)=0", "x=2", "y=2", "f'(2)=3"],
     5, [6], "", False),
    ("Q8.4 / 03 answer only", "2023_P1_Q8.4", ["g(x)=3x-4"], 1, [1, 2, 3, 4, 5], "", False),

    ("Q8.5 / 01 perfect", "2023_P1_Q8.5", ["tan theta=3", "theta=71.57"], 2, [], "", False),
    ("Q8.5 / 02 answer only", "2023_P1_Q8.5", ["theta=71.57"], 1, [1], "", False),

    # ================= Q9 (optimisation) =================
    ("Q9.1 / 01 perfect", "2023_P1_Q9.1",
     ["b=432/x", "A(x)=(x+8)(432/x+6)"], 3, [], "", False),
    ("Q9.1 / 02 b only", "2023_P1_Q9.1",
     ["b=432/x"], 1, [2], "", False),
    ("Q9.1 / 03 answer only", "2023_P1_Q9.1",
     ["A(x)=(x+8)(432/x+6)"], 2, [1], "", False),

    ("Q9.2 / 01 perfect", "2023_P1_Q9.2",
     ["A(x)=3456x^-1+6x+480", "A'(x)=-3456/x^2+6", "x=24"], 3, [], "", False),
    ("Q9.2 / 02 answer only", "2023_P1_Q9.2", ["x=24"], 1, [1, 2], "", False),
]


def run() -> int:
    passed = failed = 0
    out = []
    for name, qid, working, exp_awarded, exp_missing, note, _xfail in CASES:
        try:
            r = grade_steps(qid, working, CSV)
            awarded = r["awarded_marks"]
            missing = sorted(s["index"] for s in r["steps"] if s["match_status"] == "NOT_MATCHED")
            ok = (awarded == exp_awarded) and (missing == sorted(exp_missing))
        except Exception as e:
            awarded = missing = None
            ok = False
            note = "EXCEPTION: " + str(e) + " | " + note

        if ok:
            passed += 1
            status = "PASS "
        else:
            failed += 1
            status = "FAIL "

        out.append("  [" + status + "] " + name)
        if note:
            out.append("           note: " + note)
        if not ok:
            out.append("           expected " + str(exp_awarded) + " missing=" + str(sorted(exp_missing)))
            out.append("           got      " + str(awarded) + " missing=" + str(missing))

    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
                      CSV)
    statuses = [s["match_status"] for s in r20["steps"]]
    if statuses == ["UNCERTAIN", "UNCERTAIN", "MATCHED"]:
        passed += 1
        out.append("  [PASS ] assertion / Q1.1.1 UNCERTAIN")
    else:
        failed += 1
        out.append("  [FAIL ] assertion / Q1.1.1 UNCERTAIN")

    print("\n".join(out))
    print()
    print("  " + str(passed) + " passed  |  " + str(failed) + " failed  |  "
          + str(len(CASES) + 1) + " total")
    return failed


if __name__ == "__main__":
    sys.exit(1 if run() else 0)
'''

tt_path = ROOT / "tests" / "test_step_grader.py"
tt_path.write_text(test_source, encoding="utf-8")
print(f"test file: wrote {len(tt_path.read_text(encoding='utf-8').splitlines())} lines")
print("OK")

test file: wrote 190 lines
OK


In [51]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] Q1.1.1 / 01 perfect
  [PASS ] Q1.1.2 / 01 perfect
  [PASS ] Q1.1.3 / 01 perfect
  [PASS ] Q1.1.4 / 01 perfect
  [PASS ] Q1.2 / 01 method A
  [PASS ] Q1.2 / 02 method B
  [PASS ] Q1.3 / 01 perfect
  [PASS ] Q2.1.1 / 01 perfect
  [PASS ] Q2.1.1 / 02 Tn variant
  [PASS ] Q2.1.2 / 01 perfect
  [PASS ] Q2.1.3 / 01 perfect
  [PASS ] Q2.2.2 / 01 perfect
  [PASS ] Q2.2.3 / 01 perfect
  [PASS ] Q3.1.2 / 01 perfect
  [PASS ] Q3.2 / 01 perfect
  [PASS ] Q4.1 / 01 perfect
  [PASS ] Q4.2 / 01 perfect
  [PASS ] Q4.3 / 01 perfect
  [PASS ] Q4.4 / 01 perfect
  [PASS ] Q4.7 / 01 perfect
  [PASS ] Q5.1 / 01 perfect
  [PASS ] Q5.5 / 01 perfect
  [PASS ] Q5.6 / 01 method A
  [PASS ] Q5.6 / 02 method B
  [PASS ] Q5.7 / 01 perfect
  [PASS ] Q6.1.1 / 01 perfect
  [PASS ] Q6.1.2 / 01 perfect
  [PASS ] Q6.2.1 / 01 perfect
  [PASS ] Q6.2.2 / 01 perfect
  [PASS ] Q6.3 / 01 perfect
  [PASS ] Q7.1 / 01 perfect
  [PASS ] Q7.2.1 / 01 perfect
  [PASS ] Q7.2.2 / 01 perfect
  [PASS ] Q7.3 / 01 perfect
  [PASS

In [52]:
new_rows = '''2023_P1_Q10.1.1,1,Formula for independent events,1,"P(A and B)=P(A)*P(B)","P(A and B)=P(A)*P(B)|P(A and B)=1/3*3/4",probability.independent,probability,A,false
2023_P1_Q10.1.1,2,Compute answer,1,"P(A and B)=1/4","P(A and B)=1/4|1/4",probability.independent,probability,A,true
2023_P1_Q10.1.2,1,Addition rule substitution,1,"P(A or B)=1/3+3/4-1/4","P(A or B)=1/3+3/4-1/4|P(A or B)=P(A)+P(B)-P(A and B)",probability.independent,probability,A,false
2023_P1_Q10.1.2,2,Compute answer,1,"P(A or B)=5/6","P(A or B)=5/6|5/6",probability.independent,probability,A,true
2023_P1_Q10.2.1,1,Tree diagram probabilities,1,"0.72; 0.28; 0.05; 0.95","0.72; 0.28; 0.05; 0.95",probability.tree,tree,A,true
2023_P1_Q10.2.2,1,Compute first branch,1,"P(S; NOT below 0)=0.05*0.28","P(S; NOT below 0)=0.05*0.28",probability.tree,tree,A,false
2023_P1_Q10.2.2,2,Compute second branch,1,"P(NS; NOT below 0)=0.95*0.65","P(NS; NOT below 0)=0.95*0.65",probability.tree,tree,A,false
2023_P1_Q10.2.2,3,Total probability,1,"P(NOT below 0)=0.6315","P(NOT below 0)=0.6315|0.6315",probability.tree,tree,A,true
2023_P1_Q10.3.1,1,Total permutations,1,"n(S)=10!","n(S)=10!|10!",probability.counting,counting,A,true
2023_P1_Q10.3.2,1,Count valid arrangements,1,"4*2*8!","4*2*8!|4(2x8!x1)",probability.counting,counting,A,false
2023_P1_Q10.3.2,2,Compute probability,1,"P=322560/10!","P=322560/10!|15/56",probability.counting,counting,A,false
2023_P1_Q10.3.2,3,Simplify answer,1,"P=15/56","P=15/56|15/56",probability.counting,counting,A,true
2023_P1_Q11.1.1,1,Algebra with powers,1,"n=8.81","n=8.81",algebra.sequences,arithmetic,A,true
2023_P1_Q11.1.2,1,Second difference,1,"d=-3 or d=4","d=-3 or d=4|d=4 or d=-3",algebra.sequences,arithmetic,A,false
2023_P1_Q11.1.2,2,Final answer,1,n=8,n=8,algebra.sequences,arithmetic,A,true
2023_P1_Q11.2.1,1,Find k,1,k=9,k=9,algebra.sequences,arithmetic,A,false
2023_P1_Q11.2.1,2,Compute Tk,1,"Tk=199","Tk=199",algebra.sequences,arithmetic,A,true
2023_P1_Q11.3.1,1,k=21,1,k=21,k=21,algebra.sequences,arithmetic,A,true
2023_P1_Q11.3.2,1,Count outcomes,1,"4(2*8!*1)","4(2*8!*1)|322560",probability.counting,counting,A,false
2023_P1_Q11.3.2,2,Probability,1,"15/56","15/56|P=15/56",probability.counting,counting,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"
existing = csv_path.read_text(encoding="utf-8").rstrip("\n")
before = len(existing.splitlines())
csv_path.write_text(existing + "\n" + new_rows, encoding="utf-8")

lines = csv_path.read_text(encoding="utf-8").splitlines()
added = len(lines) - before
print(f"CSV: {before} -> {len(lines)} lines (added {added})")
print("OK")

CSV: 134 -> 154 lines (added 20)
OK


In [53]:
test_source = r'''"""
Test harness for step_grader.py
"""
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.tutor.step_grader import grade_steps

CSV = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P1.csv"

CASES = [
    # ============ Q1–Q9 (already proven) ============
    ("Q1.1.1 / 01 perfect", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4 or x=3"], 3, [], "", False),
    ("Q1.1.2 / 01 perfect", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79", "x=-1.12"], 4, [], "", False),
    ("Q1.1.3 / 01 perfect", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=4"], 4, [], "", False),
    ("Q1.1.4 / 01 perfect", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "x<-1 or x>3"], 4, [], "", False),
    ("Q1.2 / 01 method A", "2023_P1_Q1.2",
     ["x=2y-2", "1/(2y-2)+1/y=1", "2y^2-5y+2=0", "y=1/2 or y=2", "x=-1 or x=2"],
     5, [], "", False),
    ("Q1.2 / 02 method B", "2023_P1_Q1.2",
     ["y=x/2+1", "1/x+2/(x+2)=1", "x^2-x-2=0", "x=-1 or x=2", "y=1/2 or y=2"],
     5, [], "", False),
    ("Q1.3 / 01 perfect", "2023_P1_Q1.3",
     ["2^m(2+1)=3^n(3^2-1)", "2^m(3)=3^n(2^3)", "m=3 and n=1", "m+n=4"],
     4, [], "", False),
    ("Q2.1.1 / 01 perfect", "2023_P1_Q2.1.1",
     ["d=5", "T91=7+(91-1)(5)", "T91=457"], 3, [], "", False),
    ("Q2.1.2 / 01 perfect", "2023_P1_Q2.1.2",
     ["S91=91/2[2(7)+(91-1)(5)]", "S91=21112"], 2, [], "", False),
    ("Q2.1.3 / 01 perfect", "2023_P1_Q2.1.3",
     ["Tn=7+(n-1)(5)", "Tn=517", "n=103"], 3, [], "", False),
    ("Q2.2.2 / 01 perfect", "2023_P1_Q2.2.2",
     ["2a=12", "3(6)+b=9", "Tn=6n^2-9n+6"], 3, [], "", False),
    ("Q2.2.3 / 01 perfect", "2023_P1_Q2.2.3",
     ["12n-9=0", "n=3/4", "T is increasing for n in N"], 3, [], "", False),
    ("Q3.1.2 / 01 perfect", "2023_P1_Q3.1.2",
     ["3+6+12+...", "n=k", "3(2^k-1)=98301", "k=15"], 4, [], "", False),
    ("Q3.2 / 01 perfect", "2023_P1_Q3.2",
     ["S22=22/2[2a+21(3)]", "S22=22a+693", "S_inf=3a/2", "22a+693=3a/2+734", "a=2"],
     5, [], "", False),
    ("Q4.1 / 01 perfect", "2023_P1_Q4.1", ["y=-4"], 1, [], "", False),
    ("Q4.2 / 01 perfect", "2023_P1_Q4.2", ["0=2^x-4", "x=2"], 2, [], "", False),
    ("Q4.3 / 01 perfect", "2023_P1_Q4.3",
     ["y=2^0-4=-3", "m=3/2", "k(x)=3/2x-3"], 3, [], "", False),
    ("Q4.4 / 01 perfect", "2023_P1_Q4.4",
     ["k(1)=-3/2", "f(1)=-2", "1/2"], 3, [], "", False),
    ("Q4.7 / 01 perfect", "2023_P1_Q4.7",
     ["x=2^y", "g^-1(x)=log_2 x"], 2, [], "", False),
    ("Q5.1 / 01 perfect", "2023_P1_Q5.1", ["x=1", "y=8"], 2, [], "", False),
    ("Q5.5 / 01 perfect", "2023_P1_Q5.5",
     ["-3<=x<0", "x>=5", "-3<=x<0 or x>=5"], 3, [], "", False),
    ("Q5.6 / 01 method A", "2023_P1_Q5.6",
     ["-2x+k=8/x", "-2x^2+kx-8=0", "k^2-64<0", "-8<k<8"], 4, [], "", False),
    ("Q5.6 / 02 method B", "2023_P1_Q5.6",
     ["-8/x^2=-2", "x=2 or x=-2", "k=8", "-8<k<8"], 4, [], "", False),
    ("Q5.7 / 01 perfect", "2023_P1_Q5.7",
     ["-2x+8=8/x", "(x-2)^2=0", "f(2)=15/2; h(2)=4", "t=-7/2"], 4, [], "", False),
    ("Q6.1.1 / 01 perfect", "2023_P1_Q6.1.1",
     ["n=6", "19319.48=18500(1+r/1200)^6", "r=8.7%"], 3, [], "", False),
    ("Q6.2.2 / 01 perfect", "2023_P1_Q6.2.2",
     ["i=8.7/1200", "n=60", "20000=x[(1+8.7/1200)^60-1]/(8.7/1200)", "x=267.26"],
     4, [], "", False),
    ("Q6.3 / 01 perfect", "2023_P1_Q6.3",
     ["i=0.112/12", "1600000=20000[1-(1+0.112/12)^-n]/(0.112/12)",
      "-n=log_(1+0.112/12)(19/75)", "-n=-147.80", "n=147"],
     5, [], "", False),
    ("Q7.1 / 01 perfect", "2023_P1_Q7.1",
     ["f'(x)=lim(f(x+h)-f(x))/h", "f(x+h)=-4x^2-8xh-4h^2",
      "-8xh-4h^2", "h(-8x-4h)", "-8x"], 5, [], "", False),
    ("Q7.2.1 / 01 perfect", "2023_P1_Q7.2.1", ["f'(x)=6x^2-3"], 2, [], "", False),
    ("Q7.2.2 / 01 perfect", "2023_P1_Q7.2.2",
     ["x^(2/3)", "(14/3)x^(-1/3)", "-10x^(-6)"], 3, [], "", False),
    ("Q7.3 / 01 perfect", "2023_P1_Q7.3",
     ["x=-2/sqrt(3) or x=2/sqrt(3)", "-2/sqrt(3)<x<2/sqrt(3)"], 3, [], "", False),
    ("Q8.1 / 01 perfect", "2023_P1_Q8.1",
     ["f'(x)=-3x^2+12x-9", "f'(x)=0", "x=3 or x=1", "(3;4) and (1;0)"],
     4, [], "", False),
    ("Q8.3 / 01 perfect", "2023_P1_Q8.3", ["0<k<4"], 2, [], "", False),
    ("Q8.4 / 01 perfect", "2023_P1_Q8.4",
     ["f''(x)=-6x+12", "f''(x)=0", "x=2", "y=2", "f'(2)=3", "g(x)=3x-4"],
     6, [], "", False),
    ("Q8.5 / 01 perfect", "2023_P1_Q8.5", ["tan theta=3", "theta=71.57"], 2, [], "", False),
    ("Q9.1 / 01 perfect", "2023_P1_Q9.1",
     ["b=432/x", "A(x)=(x+8)(432/x+6)"], 3, [], "", False),
    ("Q9.2 / 01 perfect", "2023_P1_Q9.2",
     ["A(x)=3456x^-1+6x+480", "A'(x)=-3456/x^2+6", "x=24"], 3, [], "", False),

    # ============ Q10 (probability) ============
    ("Q10.1.1 / 01 perfect", "2023_P1_Q10.1.1",
     ["P(A and B)=P(A)*P(B)", "P(A and B)=1/4"], 2, [], "", False),
    ("Q10.1.1 / 02 answer only", "2023_P1_Q10.1.1", ["P(A and B)=1/4"], 1, [1], "", False),
    ("Q10.1.2 / 01 perfect", "2023_P1_Q10.1.2",
     ["P(A or B)=1/3+3/4-1/4", "P(A or B)=5/6"], 2, [], "", False),
    ("Q10.1.2 / 02 answer only", "2023_P1_Q10.1.2", ["P(A or B)=5/6"], 1, [1], "", False),
    ("Q10.2.1 / 01 perfect", "2023_P1_Q10.2.1",
     ["0.72; 0.28; 0.05; 0.95"], 1, [], "", False),
    ("Q10.2.2 / 01 perfect", "2023_P1_Q10.2.2",
     ["P(S; NOT below 0)=0.05*0.28", "P(NS; NOT below 0)=0.95*0.65", "P(NOT below 0)=0.6315"],
     3, [], "", False),
    ("Q10.2.2 / 02 answer only", "2023_P1_Q10.2.2", ["P(NOT below 0)=0.6315"], 1, [1, 2], "", False),
    ("Q10.3.1 / 01 perfect", "2023_P1_Q10.3.1", ["n(S)=10!"], 1, [], "", False),
    ("Q10.3.2 / 01 perfect", "2023_P1_Q10.3.2",
     ["4*2*8!", "P=322560/10!", "P=15/56"], 3, [], "", False),
    ("Q10.3.2 / 02 answer only", "2023_P1_Q10.3.2", ["P=15/56"], 1, [1, 2], "", False),

    # ============ Q11 (counting) ============
    ("Q11.1.1 / 01 perfect", "2023_P1_Q11.1.1", ["n=8.81"], 1, [], "", False),
    ("Q11.1.2 / 01 perfect", "2023_P1_Q11.1.2",
     ["d=-3 or d=4", "n=8"], 2, [], "", False),
    ("Q11.2.1 / 01 perfect", "2023_P1_Q11.2.1", ["k=9", "Tk=199"], 2, [], "", False),
    ("Q11.3.1 / 01 perfect", "2023_P1_Q11.3.1", ["k=21"], 1, [], "", False),
    ("Q11.3.2 / 01 perfect", "2023_P1_Q11.3.2",
     ["4(2*8!*1)", "15/56"], 2, [], "", False),
]


def run() -> int:
    passed = failed = 0
    out = []
    for name, qid, working, exp_awarded, exp_missing, note, _xfail in CASES:
        try:
            r = grade_steps(qid, working, CSV)
            awarded = r["awarded_marks"]
            missing = sorted(s["index"] for s in r["steps"] if s["match_status"] == "NOT_MATCHED")
            ok = (awarded == exp_awarded) and (missing == sorted(exp_missing))
        except Exception as e:
            awarded = missing = None
            ok = False
            note = "EXCEPTION: " + str(e) + " | " + note

        if ok:
            passed += 1
            status = "PASS "
        else:
            failed += 1
            status = "FAIL "

        out.append("  [" + status + "] " + name)
        if note:
            out.append("           note: " + note)
        if not ok:
            out.append("           expected " + str(exp_awarded) + " missing=" + str(sorted(exp_missing)))
            out.append("           got      " + str(awarded) + " missing=" + str(missing))

    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"],
                      CSV)
    statuses = [s["match_status"] for s in r20["steps"]]
    if statuses == ["UNCERTAIN", "UNCERTAIN", "MATCHED"]:
        passed += 1
        out.append("  [PASS ] assertion / Q1.1.1 UNCERTAIN")
    else:
        failed += 1
        out.append("  [FAIL ] assertion / Q1.1.1 UNCERTAIN")

    print("\n".join(out))
    print()
    print("  " + str(passed) + " passed  |  " + str(failed) + " failed  |  "
          + str(len(CASES) + 1) + " total")
    return failed


if __name__ == "__main__":
    sys.exit(1 if run() else 0)
'''

tt_path = ROOT / "tests" / "test_step_grader.py"
tt_path.write_text(test_source, encoding="utf-8")
print(f"test file: wrote {len(tt_path.read_text(encoding='utf-8').splitlines())} lines")
print("OK")

test file: wrote 172 lines
OK


In [54]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] Q1.1.1 / 01 perfect
  [PASS ] Q1.1.2 / 01 perfect
  [PASS ] Q1.1.3 / 01 perfect
  [PASS ] Q1.1.4 / 01 perfect
  [PASS ] Q1.2 / 01 method A
  [PASS ] Q1.2 / 02 method B
  [PASS ] Q1.3 / 01 perfect
  [PASS ] Q2.1.1 / 01 perfect
  [PASS ] Q2.1.2 / 01 perfect
  [PASS ] Q2.1.3 / 01 perfect
  [PASS ] Q2.2.2 / 01 perfect
  [PASS ] Q2.2.3 / 01 perfect
  [PASS ] Q3.1.2 / 01 perfect
  [PASS ] Q3.2 / 01 perfect
  [PASS ] Q4.1 / 01 perfect
  [PASS ] Q4.2 / 01 perfect
  [PASS ] Q4.3 / 01 perfect
  [PASS ] Q4.4 / 01 perfect
  [PASS ] Q4.7 / 01 perfect
  [PASS ] Q5.1 / 01 perfect
  [PASS ] Q5.5 / 01 perfect
  [PASS ] Q5.6 / 01 method A
  [PASS ] Q5.6 / 02 method B
  [PASS ] Q5.7 / 01 perfect
  [PASS ] Q6.1.1 / 01 perfect
  [PASS ] Q6.2.2 / 01 perfect
  [PASS ] Q6.3 / 01 perfect
  [PASS ] Q7.1 / 01 perfect
  [PASS ] Q7.2.1 / 01 perfect
  [PASS ] Q7.2.2 / 01 perfect
  [PASS ] Q7.3 / 01 perfect
  [PASS ] Q8.1 / 01 perfect
  [PASS ] Q8.3 / 01 perfect
  [PASS ] Q8.4 / 01 perfect
  [PASS ] Q8.5 /

In [55]:
csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final
2023_P2_Q1.1,1,Compute a,1,a=-23.85,"a=-23.85|a=-23.846",statistics.regression,regression,A,false
2023_P2_Q1.1,2,Compute b,1,b=0.23,"b=0.23|b=0.227",statistics.regression,regression,A,false
2023_P2_Q1.1,3,Write regression equation,1,"y_hat=-23.85+0.23x","y_hat=-23.85+0.23x|yhat=-23.85+0.23x|y=-23.85+0.23x",statistics.regression,regression,A,true
2023_P2_Q1.2,1,Substitute 550,1,"y_hat=-23.85+0.23(550)","y_hat=-23.85+0.23(550)|y=-23.85+0.23(550)",statistics.regression,regression,A,false
2023_P2_Q1.2,2,Compute prediction,1,"y_hat=102.65","y_hat=102.65|y=102.65|102.65|101.02",statistics.regression,regression,A,true
2023_P2_Q1.3,1,Correlation coefficient,1,r=0.98,r=0.98,statistics.regression,regression,A,true
2023_P2_Q1.4,1,Describe correlation,1,strong positive,"strong positive|very strong positive",statistics.regression,regression,A,true
2023_P2_Q1.5.1,1,Compute mean,1,"x_bar=1200/8","x_bar=1200/8|xbar=1200/8",statistics.mean,mean,A,false
2023_P2_Q1.5.1,2,State mean,1,x_bar=150,"x_bar=150|xbar=150|150",statistics.mean,mean,A,true
2023_P2_Q1.5.2,1,Standard deviation,1,sigma=50.50,"sigma=50.50|σ=50.50|50.50",statistics.stdev,stdev,A,true
2023_P2_Q1.5.3,1,Compute lower bound,1,"x_bar-sigma=99.50","x_bar-sigma=99.50|xbar-sigma=99.50",statistics.stdev,stdev,A,false
2023_P2_Q1.5.3,2,State answer,1,1 stop,"1 stop|1 bus stop",statistics.stdev,stdev,A,true
2023_P2_Q2.2,1,State total staff,1,40 staff,"40 staff|40",statistics.frequency,frequency,A,true
2023_P2_Q2.3,1,State answer,1,33 staff,"33 staff|33",statistics.frequency,frequency,A,true
2023_P2_Q2.4,1,Substitute into mean formula,1,"x_bar=(1(5+k/2)+3(15)+5(13+k/2)+7(5)+9(2))/(40+k)","x_bar=(1(5+k/2)+3(15)+5(13+k/2)+7(5)+9(2))/(40+k)",statistics.frequency,frequency,A,false
2023_P2_Q2.4,2,Simplify to equation,1,"3k+168=160+4k","3k+168=160+4k",statistics.frequency,frequency,A,false
2023_P2_Q2.4,3,Solve for k,1,k=8,k=8,statistics.frequency,frequency,A,true
2023_P2_Q3.1,1,Substitute into distance formula,1,"SL=sqrt((4-(-4))^2+(5-1)^2)","SL=sqrt((4-(-4))^2+(5-1)^2)|SL=√((4+4)^2+(5-1)^2)",analytical.distance,distance,A,false
2023_P2_Q3.1,2,Answer,1,"SL=4sqrt(5)=8.94","SL=4sqrt(5)=8.94|4√5|8.94",analytical.distance,distance,A,true
2023_P2_Q3.2,1,Substitute into gradient formula,1,"m_SN=(5-(-3))/(4-(-2))","m_SN=(5-(-3))/(4-(-2))",analytical.gradient,gradient,A,false
2023_P2_Q3.2,2,Answer gradient,1,m_SN=4/3,"m_SN=4/3|4/3",analytical.gradient,gradient,A,true
2023_P2_Q3.3,1,Set tan theta equal to gradient,1,"tan theta=m_SN","tan theta=m_SN|tan(theta)=m_SN|tanθ=m_SN",analytical.gradient,gradient,A,false
2023_P2_Q3.3,2,Answer angle,1,theta=53.13,"theta=53.13|θ=53.13|53.13",analytical.gradient,gradient,A,true
2023_P2_Q3.4,1,Find m_LN,1,m_LN=-2,"m_LN=-2|m=−2",analytical.gradient,gradient,A,false
2023_P2_Q3.4,2,Angle LKO,1,LKO=116.57,"LKO=116.57|116.57",analytical.gradient,gradient,A,false
2023_P2_Q3.4,3,Answer angle LNS,1,LNS=63.44,"LNS=63.44|63.44",analytical.gradient,gradient,A,true
2023_P2_Q3.5,1,Substitute point into line,1,"y=4/3x+19/3","y=4/3x+19/3|y=(4/3)x+19/3",analytical.line,line,A,true
2023_P2_Q3.6,1,Length LN,1,"LN=2sqrt(5)","LN=2sqrt(5)|2√5",analytical.area,area,A,false
2023_P2_Q3.6,2,Area formula,1,"Area=1/2(4sqrt(5))(2sqrt(5))","Area=1/2(4sqrt(5))(2sqrt(5))",analytical.area,area,A,false
2023_P2_Q3.6,3,Area answer,1,"Area=20","Area=20|20 units^2|20",analytical.area,area,A,true
2023_P2_Q3.7,1,Right angle at L,1,L=90,"L=90|L_hat=90|angle L=90",analytical.circle,circle,A,false
2023_P2_Q3.7,2,Centre coordinates,1,"P(1;1)","P(1;1)|P(1,1)",analytical.circle,circle,A,true
2023_P2_Q3.8,1,Find angle LPN,1,"LPN=53.13","LPN=53.13|LPN=θ=53.13",analytical.circle,circle,A,false
2023_P2_Q3.8,2,Find angle LPS,1,"LPS=126.87","LPS=126.87|L_PS=126.87",analytical.circle,circle,A,true
2023_P2_Q4.1,1,Substitute into circle eq,1,"p^2+(-2)^2=20","p^2+(-2)^2=20",analytical.circle,circle,A,false
2023_P2_Q4.1,2,Solve for p,1,p=4,"p=4",analytical.circle,circle,A,true
2023_P2_Q4.2,1,x-coordinate of F,1,x_F=8,"x_F=8",analytical.circle,circle,A,false
2023_P2_Q4.2,2,F coordinates,1,"F(8;6)","F(8;6)|F(8,6)",analytical.circle,circle,A,true
2023_P2_Q4.3,1,Gradient of DE,1,m_DE=2,"m_DE=2",analytical.line,line,A,false
2023_P2_Q4.3,2,Substitute point,1,"-2=2(4)+c","-2=2(4)+c",analytical.line,line,A,false
2023_P2_Q4.3,3,Equation of DE,1,y=2x-10,"y=2x-10|y=2x−10",analytical.line,line,A,true
2023_P2_Q4.4,1,Gradient of GF,1,m_GF=-1/2,"m_GF=-1/2|m_GF=−0.5",analytical.line,line,A,false
2023_P2_Q4.4,2,Substitute F,1,"(0-6)/(t-8)=-1/2","(0-6)/(t-8)=-1/2",analytical.line,line,A,false
2023_P2_Q4.4,3,Solve for t,1,t=20,t=20,analytical.line,line,A,true
2023_P2_Q4.5,1,Substitute F and G,1,"(8-20)^2+(6-0)^2=r^2","(8-20)^2+(6-0)^2=r^2",analytical.circle,circle,A,false
2023_P2_Q4.5,2,Value of r squared,1,r^2=180,"r^2=180|r2=180",analytical.circle,circle,A,false
2023_P2_Q4.5,3,Circle equation,1,"(x-20)^2+y^2=180","(x-20)^2+y^2=180",analytical.circle,circle,A,false
2023_P2_Q4.5,4,General form,1,"x^2+y^2-40x+220=0","x^2+y^2-40x+220=0",analytical.circle,circle,A,true
2023_P2_Q4.6,1,Method for k,1,"k=20-(6sqrt(5)-2sqrt(5)) or k=20+(6sqrt(5)-2sqrt(5))","k=20-4sqrt(5) or k=20+4sqrt(5)",analytical.circle,circle,A,false
2023_P2_Q4.6,2,Smaller value,1,k=11.06,"k=11.06|11.06",analytical.circle,circle,A,false
2023_P2_Q4.6,3,Larger value,1,k=28.94,"k=28.94|28.94",analytical.circle,circle,A,true
2023_P2_Q5.1.1,1,Pythagoras for third side,1,"x^2+y^2=r^2","x^2+y^2=r^2",trig.identity,trig,A,false
2023_P2_Q5.1.1,2,Compute x,1,"x=-2sqrt(2)","x=-2sqrt(2)|-2√2",trig.identity,trig,A,false
2023_P2_Q5.1.1,3,Answer cos beta,1,"cos beta=-2sqrt(2)/3","cos beta=-2sqrt(2)/3|cosβ=-2√2/3",trig.identity,trig,A,true
2023_P2_Q5.1.2,1,Double angle formula,1,"sin 2beta=2 sin beta cos beta","sin2beta=2sinb cosb|sin2β=2sinβcosβ",trig.identity,trig,A,false
2023_P2_Q5.1.2,2,Substitute,1,"=2(1/3)(-sqrt(8)/3)","2(1/3)(-sqrt(8)/3)",trig.identity,trig,A,false
2023_P2_Q5.1.2,3,Answer,1,"sin 2beta=-4sqrt(2)/9","sin2beta=-4sqrt(2)/9|−4√2/9",trig.identity,trig,A,true
2023_P2_Q5.2,1,Rewrite cos(450-beta),1,"cos(450-beta)","cos(450-beta)|cos(450−β)",trig.reduction,trig,A,false
2023_P2_Q5.2,2,Expand,1,"cos450 cosb + sin450 sinb","cos450cosb+sin450sinb",trig.reduction,trig,A,false
2023_P2_Q5.2,3,Answer,1,"=sin beta=1/3","sin beta=1/3|sinβ=1/3",trig.reduction,trig,A,true
2023_P2_Q5.2.2,1,Factor equals zero,1,"sin x+1=0","sin x+1=0|sinx=−1",trig.equation,trig,A,false
2023_P2_Q5.2.2,2,Answer x,1,x=270,"x=270|x=270°",trig.equation,trig,A,true
2023_P2_Q5.2.3,1,Minimum value,1,min=0,"min=0|minimum=0",trig.equation,trig,A,true
2023_P2_Q5.3.2,1,Apply compound formula,1,"sin(48-x)=cos 2x","sin(48-x)=cos2x",trig.equation,trig,A,false
2023_P2_Q5.3.2,2,Rewrite cos as sin,1,"sin(48-x)=sin(90-2x)","sin(48-x)=sin(90-2x)",trig.equation,trig,A,false
2023_P2_Q5.3.2,3,First solution,1,"x=42+k.360","x=42+k.360|x=42°+k.360°",trig.equation,trig,A,false
2023_P2_Q5.3.2,4,Second solution,1,"x=-14-k.120","x=-14-k.120|x=-14°-k.120°",trig.equation,trig,A,true
2023_P2_Q6.1,1,State period,1,period=180,"period=180|180|180°",trig.graph,graph,A,true
2023_P2_Q6.2,1,State range,1,"y in [-sqrt(2)/2;1]","y in [-sqrt(2)/2;1]|y∈[−√2/2;1]",trig.graph,graph,A,true
2023_P2_Q6.3.1,1,State interval,1,"x in (45;90)","x in (45;90)|45<x<90",trig.graph,graph,A,true
2023_P2_Q6.3.2,1,State interval,1,"x in [105;165]","x in [105;165]|105<=x<=165",trig.graph,graph,A,true
2023_P2_Q6.4,1,Set up equation,1,"-2sin2x=-1 or 2sin2x=1","-2sin2x=-1|2sin2x=1",trig.graph,graph,A,false
2023_P2_Q6.4,2,First value k,1,k=15,k=15,trig.graph,graph,A,false
2023_P2_Q6.4,3,Second value k,1,k=75,k=75,trig.graph,graph,A,true
2023_P2_Q6.5,1,Substitute shift,1,"h(x)=-cos(x+90)","h(x)=-cos(x+90)|h(x)=−cos(x+90)",trig.graph,graph,A,false
2023_P2_Q6.5,2,Simplify answer,1,"h(x)=sin x","h(x)=sin x|h(x)=sinx",trig.graph,graph,A,true
2023_P2_Q7.1,1,Area formula for triangle,1,"q=1/2 p(SK) sin alpha","q=1/2 p(SK) sin alpha|q=1/2 p(SK) sinα",trig.3d,trig3d,A,false
2023_P2_Q7.1,2,Solve for SK,1,"SK=2q/(p sin alpha)","SK=2q/(p sin alpha)|SK=2q/(p sinα)",trig.3d,trig3d,A,true
2023_P2_Q7.2,1,Right angle at S,1,RKS=beta,"RKS=beta|RKS=β",trig.3d,trig3d,A,false
2023_P2_Q7.2,2,Correct trig ratio,1,"RS/SK=tan beta","RS/SK=tan beta|RS/SK=tanβ",trig.3d,trig3d,A,false
2023_P2_Q7.2,3,Answer RS,1,"RS=2q tan beta/(p sin alpha)","RS=2q tan beta/(p sin alpha)|RS=2q tanβ/(p sinα)",trig.3d,trig3d,A,true
2023_P2_Q7.3,1,Substitute values,1,"70=2(2500)tan42/(80 sin alpha)","70=2(2500)tan42/(80 sin alpha)",trig.3d,trig3d,A,false
2023_P2_Q7.3,2,Value of sin alpha,1,"sin alpha=25/28 tan42","sin alpha=25/28 tan42|sinα=0.80",trig.3d,trig3d,A,false
2023_P2_Q7.3,3,Answer alpha,1,alpha=53.51,"alpha=53.51|α=53.51|53.51",trig.3d,trig3d,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P2.csv"
csv_path.write_text(csv_content, encoding="utf-8")
lines = csv_path.read_text(encoding="utf-8").splitlines()
print(f"2023 P2 CSV: wrote {len(lines)} lines")

2023 P2 CSV: wrote 85 lines


In [56]:
step_grader_source = r'''from __future__ import annotations

import csv
import re
from pathlib import Path
from typing import Any

ROOT = Path(__file__).resolve().parents[2]
CSV_DIR = ROOT / "data" / "processed" / "memo_step_marks"


def _load_one(path: Path, by_q: dict) -> None:
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            qid = row["question_id"].strip()
            if not qid:
                continue
            mg = (row.get("method_group") or "A").strip() or "A"
            by_q.setdefault(qid, []).append({
                "index": int(row["step_index"]),
                "desc": row["step_description"].strip(),
                "marks": int(row["marks"]),
                "memo_line": row["memo_line"].strip(),
                "forms": [x.strip() for x in row["acceptable_forms"].split("|") if x.strip()],
                "skill_id": row["skill_id"].strip(),
                "method_tag": row["method_tag"].strip(),
                "method_group": mg,
                "is_final": str(row.get("is_final", "false")).strip().lower() == "true",
            })


def load_steps(csv_path: Path | None = None) -> dict[str, list[dict[str, Any]]]:
    by_q: dict[str, list[dict[str, Any]]] = {}
    if csv_path is not None:
        _load_one(csv_path, by_q)
    else:
        for p in sorted(CSV_DIR.glob("memo_step_marks_*.csv")):
            _load_one(p, by_q)
    for qid in by_q:
        by_q[qid].sort(key=lambda s: (s["method_group"], s["index"]))
    return by_q


_SEP = re.compile(r"\s*(?:,|\bor\b|;|\band\b)\s*")
_METHOD_MARKERS = re.compile(r"sqrt|\u00b1|\+/-|/2a|b2-4ac|b\^2|\\pm|\u221a|\u221a|\u00b1")


def _norm(s: str) -> str:
    s = s.lower().strip()
    s = s.replace("\u00d7", "*").replace("\u2212", "-").replace("\u2013", "-")
    s = s.replace("\u2234", " ").replace("\u2235", " ")
    s = re.sub(r"^\s*(therefore|so|hence|thus)\b\s*", "", s)
    s = re.sub(r"(\d),(\d)", r"\1.\2", s)
    return s


def _tokens(s: str) -> list[str]:
    parts = _SEP.split(_norm(s))
    return sorted(re.sub(r"\s+", "", p) for p in parts if p.strip())


def _line_matches(line: str, forms: list[str], is_final: bool = False) -> bool:
    lt = _tokens(line)
    if not lt:
        return False
    lt_set = set(lt)
    for form in forms:
        ft = _tokens(form)
        if not ft:
            continue
        if is_final:
            if set(ft).issubset(lt_set):
                return True
        else:
            if ft == lt:
                return True
    return False


def _grade_group(steps, learner_lines):
    used: set[int] = set()
    results = []
    awarded = 0
    for step in steps:
        matched = False
        matched_line = None
        is_final = step.get("is_final", False)
        for i, line in enumerate(learner_lines):
            if i in used and not is_final:
                continue
            if _line_matches(line, step["forms"], is_final=is_final):
                matched = True
                matched_line = line
                if not is_final:
                    used.add(i)
                awarded += step["marks"]
                break
        results.append({
            "index": step["index"],
            "desc": step["desc"],
            "marks": step["marks"],
            "matched": matched,
            "learner_line": matched_line,
            "is_final": is_final,
        })
    return awarded, results, used


def grade_steps(question_id: str, learner_lines: list[str], csv_path: Path | None = None) -> dict[str, Any]:
    steps_map = load_steps(csv_path)
    steps = steps_map.get(question_id, [])
    if not steps:
        return {
            "question_id": question_id,
            "total_marks": 0,
            "awarded_marks": 0,
            "steps": [],
            "feedback": "No step scheme for " + question_id,
        }

    groups: dict[str, list] = {}
    for s in steps:
        groups.setdefault(s["method_group"], []).append(s)

    best_awarded = -1
    best_results = None
    best_group = None
    for g_name, g_steps in groups.items():
        awarded, results, used = _grade_group(g_steps, learner_lines)
        if awarded > best_awarded:
            best_awarded = awarded
            best_results = results
            best_group = g_steps

    results = best_results
    awarded = best_awarded
    g_steps = best_group

    matched_values = {r["learner_line"] for r in results if r["matched"]}
    unmatched_lines = [ln for ln in learner_lines if ln not in matched_values]

    final_matched = any(r["matched"] and r["is_final"] for r in results)
    has_method_evidence = any(_METHOD_MARKERS.search(ln) for ln in unmatched_lines)

    for r in results:
        if r["matched"]:
            r["match_status"] = "MATCHED"
        elif final_matched and has_method_evidence:
            r["match_status"] = "UNCERTAIN"
        else:
            r["match_status"] = "NOT_MATCHED"

    total = sum(s["marks"] for s in g_steps)
    missing = [r for r in results if r["match_status"] == "NOT_MATCHED"]
    uncertain = [r for r in results if r["match_status"] == "UNCERTAIN"]

    if not missing and not uncertain:
        feedback = "Full marks: %d/%d" % (awarded, total)
    elif uncertain and not missing:
        bits = ["cannot verify '%s' (alternate method used)" % u["desc"] for u in uncertain]
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)
    else:
        bits = ["missed '%s' (%d mark)" % (m["desc"], m["marks"]) for m in missing]
        if uncertain:
            bits.append("cannot verify " + ", ".join("'%s'" % u["desc"] for u in uncertain))
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)

    return {
        "question_id": question_id,
        "total_marks": total,
        "awarded_marks": awarded,
        "steps": results,
        "feedback": feedback,
    }
'''

sg_path = ROOT / "src" / "tutor" / "step_grader.py"
sg_path.write_text(step_grader_source, encoding="utf-8")
print(f"step_grader.py: {len(sg_path.read_text(encoding='utf-8').splitlines())} lines")
print("OK — now loads ALL memo_step_marks_*.csv files in the folder")

step_grader.py: 174 lines
OK — now loads ALL memo_step_marks_*.csv files in the folder


In [57]:
test_source = r'''"""
Test harness for step_grader.py
"""
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.tutor.step_grader import grade_steps

CASES = [
    # ============ 2023 P1 (regression) ============
    ("P1 Q1.1.1 perfect", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4 or x=3"], 3, [], "", False),
    ("P1 Q1.1.2 perfect", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79", "x=-1.12"], 4, [], "", False),
    ("P1 Q1.1.3 perfect", "2023_P1_Q1.1.3",
     ["2x+1=(x-1)^2", "x^2-4x=0", "x=0 or x=4", "x=4"], 4, [], "", False),
    ("P1 Q1.1.4 perfect", "2023_P1_Q1.1.4",
     ["x^2-2x-3>0", "(x-3)(x+1)>0", "x<-1 or x>3"], 4, [], "", False),
    ("P1 Q1.2 method A", "2023_P1_Q1.2",
     ["x=2y-2", "1/(2y-2)+1/y=1", "2y^2-5y+2=0", "y=1/2 or y=2", "x=-1 or x=2"],
     5, [], "", False),
    ("P1 Q1.2 method B", "2023_P1_Q1.2",
     ["y=x/2+1", "1/x+2/(x+2)=1", "x^2-x-2=0", "x=-1 or x=2", "y=1/2 or y=2"],
     5, [], "", False),
    ("P1 Q1.3 perfect", "2023_P1_Q1.3",
     ["2^m(2+1)=3^n(3^2-1)", "2^m(3)=3^n(2^3)", "m=3 and n=1", "m+n=4"],
     4, [], "", False),
    ("P1 Q3.2 perfect", "2023_P1_Q3.2",
     ["S22=22/2[2a+21(3)]", "S22=22a+693", "S_inf=3a/2", "22a+693=3a/2+734", "a=2"],
     5, [], "", False),
    ("P1 Q5.6 method A", "2023_P1_Q5.6",
     ["-2x+k=8/x", "-2x^2+kx-8=0", "k^2-64<0", "-8<k<8"], 4, [], "", False),
    ("P1 Q6.2.2 perfect", "2023_P1_Q6.2.2",
     ["i=8.7/1200", "n=60", "20000=x[(1+8.7/1200)^60-1]/(8.7/1200)", "x=267.26"],
     4, [], "", False),
    ("P1 Q7.1 perfect", "2023_P1_Q7.1",
     ["f'(x)=lim(f(x+h)-f(x))/h", "f(x+h)=-4x^2-8xh-4h^2",
      "-8xh-4h^2", "h(-8x-4h)", "-8x"], 5, [], "", False),
    ("P1 Q8.4 perfect", "2023_P1_Q8.4",
     ["f''(x)=-6x+12", "f''(x)=0", "x=2", "y=2", "f'(2)=3", "g(x)=3x-4"],
     6, [], "", False),
    ("P1 Q9.2 perfect", "2023_P1_Q9.2",
     ["A(x)=3456x^-1+6x+480", "A'(x)=-3456/x^2+6", "x=24"], 3, [], "", False),
    ("P1 Q10.1.1 perfect", "2023_P1_Q10.1.1",
     ["P(A and B)=P(A)*P(B)", "P(A and B)=1/4"], 2, [], "", False),

    # ============ 2023 P2 Q1 (statistics regression) ============
    ("P2 Q1.1 / 01 perfect", "2023_P2_Q1.1",
     ["a=-23.85", "b=0.23", "y_hat=-23.85+0.23x"], 3, [], "", False),
    ("P2 Q1.1 / 02 answer only", "2023_P2_Q1.1",
     ["y_hat=-23.85+0.23x"], 1, [1, 2], "", False),
    ("P2 Q1.2 / 01 perfect", "2023_P2_Q1.2",
     ["y_hat=-23.85+0.23(550)", "y_hat=102.65"], 2, [], "", False),
    ("P2 Q1.2 / 02 answer only", "2023_P2_Q1.2", ["y_hat=102.65"], 1, [1], "", False),
    ("P2 Q1.3 / 01 perfect", "2023_P2_Q1.3", ["r=0.98"], 1, [], "", False),
    ("P2 Q1.4 / 01 perfect", "2023_P2_Q1.4", ["strong positive"], 1, [], "", False),
    ("P2 Q1.5.1 / 01 perfect", "2023_P2_Q1.5.1",
     ["x_bar=1200/8", "x_bar=150"], 2, [], "", False),
    ("P2 Q1.5.2 / 01 perfect", "2023_P2_Q1.5.2", ["sigma=50.50"], 1, [], "", False),
    ("P2 Q1.5.3 / 01 perfect", "2023_P2_Q1.5.3",
     ["x_bar-sigma=99.50", "1 stop"], 2, [], "", False),

    # ============ Q2 (frequency) ============
    ("P2 Q2.2 / 01 perfect", "2023_P2_Q2.2", ["40 staff"], 1, [], "", False),
    ("P2 Q2.3 / 01 perfect", "2023_P2_Q2.3", ["33 staff"], 1, [], "", False),
    ("P2 Q2.4 / 01 perfect", "2023_P2_Q2.4",
     ["x_bar=(1(5+k/2)+3(15)+5(13+k/2)+7(5)+9(2))/(40+k)",
      "3k+168=160+4k", "k=8"], 3, [], "", False),
    ("P2 Q2.4 / 02 answer only", "2023_P2_Q2.4", ["k=8"], 1, [1, 2], "", False),

    # ============ Q3 (analytical geometry) ============
    ("P2 Q3.1 / 01 perfect", "2023_P2_Q3.1",
     ["SL=sqrt((4-(-4))^2+(5-1)^2)", "SL=4sqrt(5)=8.94"], 2, [], "", False),
    ("P2 Q3.2 / 01 perfect", "2023_P2_Q3.2",
     ["m_SN=(5-(-3))/(4-(-2))", "m_SN=4/3"], 2, [], "", False),
    ("P2 Q3.3 / 01 perfect", "2023_P2_Q3.3",
     ["tan theta=m_SN", "theta=53.13"], 2, [], "", False),
    ("P2 Q3.4 / 01 perfect", "2023_P2_Q3.4",
     ["m_LN=-2", "LKO=116.57", "LNS=63.44"], 3, [], "", False),
    ("P2 Q3.5 / 01 perfect", "2023_P2_Q3.5", ["y=4/3x+19/3"], 1, [], "", False),
    ("P2 Q3.6 / 01 perfect", "2023_P2_Q3.6",
     ["LN=2sqrt(5)", "Area=1/2(4sqrt(5))(2sqrt(5))", "Area=20"], 3, [], "", False),
    ("P2 Q3.7 / 01 perfect", "2023_P2_Q3.7", ["L=90", "P(1;1)"], 2, [], "", False),
    ("P2 Q3.8 / 01 perfect", "2023_P2_Q3.8", ["LPN=53.13", "LPS=126.87"], 2, [], "", False),

    # ============ Q4 (analytical geometry 2) ============
    ("P2 Q4.1 / 01 perfect", "2023_P2_Q4.1",
     ["p^2+(-2)^2=20", "p=4"], 2, [], "", False),
    ("P2 Q4.2 / 01 perfect", "2023_P2_Q4.2", ["x_F=8", "F(8;6)"], 2, [], "", False),
    ("P2 Q4.3 / 01 perfect", "2023_P2_Q4.3",
     ["m_DE=2", "-2=2(4)+c", "y=2x-10"], 3, [], "", False),
    ("P2 Q4.4 / 01 perfect", "2023_P2_Q4.4",
     ["m_GF=-1/2", "(0-6)/(t-8)=-1/2", "t=20"], 3, [], "", False),
    ("P2 Q4.5 / 01 perfect", "2023_P2_Q4.5",
     ["(8-20)^2+(6-0)^2=r^2", "r^2=180", "(x-20)^2+y^2=180",
      "x^2+y^2-40x+220=0"], 4, [], "", False),
    ("P2 Q4.6 / 01 perfect", "2023_P2_Q4.6",
     ["k=20-4sqrt(5) or k=20+4sqrt(5)", "k=11.06", "k=28.94"], 3, [], "", False),

    # ============ Q5 (trig) ============
    ("P2 Q5.1.1 / 01 perfect", "2023_P2_Q5.1.1",
     ["x^2+y^2=r^2", "x=-2sqrt(2)", "cos beta=-2sqrt(2)/3"], 3, [], "", False),
    ("P2 Q5.1.2 / 01 perfect", "2023_P2_Q5.1.2",
     ["sin2beta=2sinb cosb", "2(1/3)(-sqrt(8)/3)", "sin2beta=-4sqrt(2)/9"],
     3, [], "", False),
    ("P2 Q5.2 / 01 perfect", "2023_P2_Q5.2",
     ["cos(450-beta)", "cos450cosb+sin450sinb", "sin beta=1/3"], 3, [], "", False),
    ("P2 Q5.2.2 / 01 perfect", "2023_P2_Q5.2.2",
     ["sin x+1=0", "x=270"], 2, [], "", False),
    ("P2 Q5.2.3 / 01 perfect", "2023_P2_Q5.2.3", ["min=0"], 1, [], "", False),
    ("P2 Q5.3.2 / 01 perfect", "2023_P2_Q5.3.2",
     ["sin(48-x)=cos2x", "sin(48-x)=sin(90-2x)", "x=42+k.360", "x=-14-k.120"],
     4, [], "", False),

    # ============ Q6 (trig graphs) ============
    ("P2 Q6.1 / 01 perfect", "2023_P2_Q6.1", ["period=180"], 1, [], "", False),
    ("P2 Q6.2 / 01 perfect", "2023_P2_Q6.2", ["y in [-sqrt(2)/2;1]"], 1, [], "", False),
    ("P2 Q6.3.1 / 01 perfect", "2023_P2_Q6.3.1", ["x in (45;90)"], 1, [], "", False),
    ("P2 Q6.3.2 / 01 perfect", "2023_P2_Q6.3.2", ["x in [105;165]"], 1, [], "", False),
    ("P2 Q6.4 / 01 perfect", "2023_P2_Q6.4",
     ["-2sin2x=-1", "k=15", "k=75"], 3, [], "", False),
    ("P2 Q6.5 / 01 perfect", "2023_P2_Q6.5",
     ["h(x)=-cos(x+90)", "h(x)=sin x"], 2, [], "", False),

    # ============ Q7 (3D trig) ============
    ("P2 Q7.1 / 01 perfect", "2023_P2_Q7.1",
     ["q=1/2 p(SK) sin alpha", "SK=2q/(p sin alpha)"], 2, [], "", False),
    ("P2 Q7.2 / 01 perfect", "2023_P2_Q7.2",
     ["RKS=beta", "RS/SK=tan beta", "RS=2q tan beta/(p sin alpha)"], 3, [], "", False),
    ("P2 Q7.3 / 01 perfect", "2023_P2_Q7.3",
     ["70=2(2500)tan42/(80 sin alpha)", "sin alpha=25/28 tan42", "alpha=53.51"],
     3, [], "", False),
]


def run() -> int:
    passed = failed = 0
    out = []
    for name, qid, working, exp_awarded, exp_missing, note, _xfail in CASES:
        try:
            r = grade_steps(qid, working)
            awarded = r["awarded_marks"]
            missing = sorted(s["index"] for s in r["steps"] if s["match_status"] == "NOT_MATCHED")
            ok = (awarded == exp_awarded) and (missing == sorted(exp_missing))
        except Exception as e:
            awarded = missing = None
            ok = False
            note = "EXCEPTION: " + str(e) + " | " + note

        if ok:
            passed += 1
            status = "PASS "
        else:
            failed += 1
            status = "FAIL "

        out.append("  [" + status + "] " + name)
        if note:
            out.append("           note: " + note)
        if not ok:
            out.append("           expected " + str(exp_awarded) + " missing=" + str(sorted(exp_missing)))
            out.append("           got      " + str(awarded) + " missing=" + str(missing))

    # UNCERTAIN assertion
    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"])
    statuses = [s["match_status"] for s in r20["steps"]]
    if statuses == ["UNCERTAIN", "UNCERTAIN", "MATCHED"]:
        passed += 1
        out.append("  [PASS ] assertion / UNCERTAIN")
    else:
        failed += 1
        out.append("  [FAIL ] assertion / UNCERTAIN")

    print("\n".join(out))
    print()
    print("  " + str(passed) + " passed  |  " + str(failed) + " failed  |  "
          + str(len(CASES) + 1) + " total")
    return failed


if __name__ == "__main__":
    sys.exit(1 if run() else 0)
'''

tt_path = ROOT / "tests" / "test_step_grader.py"
tt_path.write_text(test_source, encoding="utf-8")
print(f"test file: {len(tt_path.read_text(encoding='utf-8').splitlines())} lines")
print("OK")

test file: 187 lines
OK


In [58]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] P1 Q1.1.1 perfect
  [PASS ] P1 Q1.1.2 perfect
  [PASS ] P1 Q1.1.3 perfect
  [PASS ] P1 Q1.1.4 perfect
  [PASS ] P1 Q1.2 method A
  [PASS ] P1 Q1.2 method B
  [PASS ] P1 Q1.3 perfect
  [PASS ] P1 Q3.2 perfect
  [PASS ] P1 Q5.6 method A
  [PASS ] P1 Q6.2.2 perfect
  [PASS ] P1 Q7.1 perfect
  [PASS ] P1 Q8.4 perfect
  [PASS ] P1 Q9.2 perfect
  [PASS ] P1 Q10.1.1 perfect
  [PASS ] P2 Q1.1 / 01 perfect
  [PASS ] P2 Q1.1 / 02 answer only
  [PASS ] P2 Q1.2 / 01 perfect
  [PASS ] P2 Q1.2 / 02 answer only
  [PASS ] P2 Q1.3 / 01 perfect
  [PASS ] P2 Q1.4 / 01 perfect
  [PASS ] P2 Q1.5.1 / 01 perfect
  [PASS ] P2 Q1.5.2 / 01 perfect
  [PASS ] P2 Q1.5.3 / 01 perfect
  [PASS ] P2 Q2.2 / 01 perfect
  [PASS ] P2 Q2.3 / 01 perfect
  [PASS ] P2 Q2.4 / 01 perfect
  [PASS ] P2 Q2.4 / 02 answer only
  [PASS ] P2 Q3.1 / 01 perfect
  [PASS ] P2 Q3.2 / 01 perfect
  [PASS ] P2 Q3.3 / 01 perfect
  [PASS ] P2 Q3.4 / 01 perfect
  [PASS ] P2 Q3.5 / 01 perfect
  [PASS ] P2 Q3.6 / 01 perfect
  [PASS ] P2 

In [59]:
csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final
2024_P1_Q1.1.1,1,First root x=0,1,x=0,x=0,algebra.quadratic.solve,factorise,A,true
2024_P1_Q1.1.1,2,Second root x=3,1,x=3,x=3,algebra.quadratic.solve,factorise,A,true
2024_P1_Q1.1.2,1,Rearrange to standard form,1,"2x^2-4x+1=0","2x^2-4x+1=0|2x2-4x+1=0",algebra.quadratic.solve,formula,A,false
2024_P1_Q1.1.2,2,Substitute into formula,1,"x=(4±sqrt(8))/(4)","x=(4±sqrt(8))/(4)|x=(4±√8)/4",algebra.quadratic.solve,formula,A,false
2024_P1_Q1.1.2,3,First root,1,x=1.71,"x=1.71|1.71",algebra.quadratic.solve,formula,A,true
2024_P1_Q1.1.2,4,Second root,1,x=0.29,"x=0.29|0.29",algebra.quadratic.solve,formula,A,true
2024_P1_Q1.1.3,1,Factorise,1,"(x-3)(x+1)>0","(x-3)(x+1)>0|(x+1)(x-3)>0",algebra.quadratic.inequality,inequality,A,false
2024_P1_Q1.1.3,2,Critical values,1,"x=3 or x=-1","x=3 or x=-1|x=-1 or x=3",algebra.quadratic.inequality,inequality,A,false
2024_P1_Q1.1.3,3,Solution intervals,2,"x<-1 or x>3","x<-1 or x>3|x>3 or x<-1",algebra.quadratic.inequality,inequality,A,true
2024_P1_Q1.1.4,1,Apply exponential law,1,"2^2x-2^2*2^x-32=0","2^2x-2^2*2^x-32=0|2^2x-4.2^x-32=0",algebra.exponential.solve,exponential,A,false
2024_P1_Q1.1.4,2,Factorise,1,"(2^x-8)(2^x+4)=0","(2^x-8)(2^x+4)=0",algebra.exponential.solve,exponential,A,false
2024_P1_Q1.1.4,3,State 2^x=8,1,2^x=8,2^x=8,algebra.exponential.solve,exponential,A,false
2024_P1_Q1.1.4,4,Reject 2^x=-4,1,"2^x != -4","2^x != -4|2^x=-4 no solution|2^x≠-4",algebra.exponential.solve,exponential,A,false
2024_P1_Q1.1.4,5,Answer x=3,1,x=3,x=3,algebra.exponential.solve,exponential,A,true
2024_P1_Q1.1.5,1,Isolate root,1,"sqrt(-2x+4)=2+x","sqrt(-2x+4)=2+x",algebra.surd.solve,surd,A,false
2024_P1_Q1.1.5,2,Square both sides,1,"-2x+4=(2+x)^2","-2x+4=(2+x)^2",algebra.surd.solve,surd,A,false
2024_P1_Q1.1.5,3,Rearrange to standard form,1,"x^2+6x=0","x^2+6x=0",algebra.surd.solve,surd,A,false
2024_P1_Q1.1.5,4,Answers with selection,1,"x=0 (x=-6 rejected)","x=0|x=0 only|x≠-6, x=0",algebra.surd.solve,surd,A,true
2024_P1_Q1.2,1,Write eq 1,1,"y=3-2x","y=3-2x",algebra.simultaneous.solve,substitution,A,false
2024_P1_Q1.2,2,Substitute,1,"(3-2x)^2+(3-2x)x=2","(3-2x)^2+(3-2x)x=2",algebra.simultaneous.solve,substitution,A,false
2024_P1_Q1.2,3,Standard form,1,"2x^2-9x+7=0","2x^2-9x+7=0",algebra.simultaneous.solve,substitution,A,false
2024_P1_Q1.2,4,Both x-values,1,"x=7/2 or x=1","x=7/2 or x=1|x=1 or x=7/2|x=3.5 or x=1",algebra.simultaneous.solve,substitution,A,true
2024_P1_Q1.2,5,Both y-values,1,"y=-4 or y=1","y=-4 or y=1|y=1 or y=-4",algebra.simultaneous.solve,substitution,A,true
2024_P1_Q1.2,1,Write eq 1,1,"x=3-y","x=3-y",algebra.simultaneous.solve,substitution,B,false
2024_P1_Q1.2,2,Substitute,1,"y^2+(3-y)^2=2","y^2+(3-y)^2=2",algebra.simultaneous.solve,substitution,B,false
2024_P1_Q1.2,3,Standard form,1,"y^2+3y-4=0","y^2+3y-4=0",algebra.simultaneous.solve,substitution,B,false
2024_P1_Q1.2,4,Both y-values,1,"y=-4 or y=1","y=-4 or y=1|y=1 or y=-4",algebra.simultaneous.solve,substitution,B,true
2024_P1_Q1.2,5,Both x-values,1,"x=7/2 or x=1","x=7/2 or x=1|x=1 or x=7/2|x=3.5 or x=1",algebra.simultaneous.solve,substitution,B,true
2024_P1_Q1.3,1,Express as products,1,"1x2x3x...x(n+1)=1x2x3x...xnx(n+1)","1x2x3x...x(n+1)=1x2x3x...xnx(n+1)",algebra.proof,proof,A,false
2024_P1_Q1.3,2,n is odd for n>2,1,n is odd,n is odd,algebra.proof,proof,A,true
2024_P1_Q2.1.1,1,Substitute into S_n,1,"S20=20/2[2(7)+(20-1)(5)]","S20=20/2[2(7)+(20-1)(5)]|S_20=20/2[2(7)+(20-1)(5)]",sequences.arithmetic,arith,A,false
2024_P1_Q2.1.1,2,Answer,1,S20=1090,"S20=1090|S_20=1090|1090",sequences.arithmetic,arith,A,true
2024_P1_Q2.1.2,1,General term,1,"Tn=5n+2","Tn=5n+2|T_n=5n+2",sequences.arithmetic,arith,A,false
2024_P1_Q2.1.2,2,Compute S75-S20,1,"S75-S20=14400-1090","S75-S20=14400-1090|S75-S20",sequences.arithmetic,arith,A,false
2024_P1_Q2.1.2,3,Write summation,1,"sum from 21 to 75","sum from 21 to 75|∑ from 21 to 75",sequences.arithmetic,arith,A,false
2024_P1_Q2.1.2,4,Answer,1,"=13310","=13310|13310",sequences.arithmetic,arith,A,true
2024_P1_Q2.2.1,1,98th first difference,1,"=2(98)-1=195","=2(98)-1=195|195",sequences.quadratic,arith,A,false
2024_P1_Q2.2.1,2,Subtract from T98,1,"T98=9632-195","T98=9632-195",sequences.quadratic,arith,A,false
2024_P1_Q2.2.1,3,Answer,1,"T98=9437","T98=9437|9437",sequences.quadratic,arith,A,true
2024_P1_Q2.2.2,1,Second difference,1,2a=2,"2a=2",sequences.quadratic,arith,A,false
2024_P1_Q2.2.2,2,Find a,1,a=1,a=1,sequences.quadratic,arith,A,false
2024_P1_Q2.2.2,3,Find b,1,b=-2,b=-2,sequences.quadratic,arith,A,false
2024_P1_Q2.2.2,4,Find c,1,c=29,c=29,sequences.quadratic,arith,A,false
2024_P1_Q2.2.2,5,Write Tn,1,"Tn=n^2-2n+29","Tn=n^2-2n+29",sequences.quadratic,arith,A,true
2024_P1_Q3.1,1,Radius of 3rd circle,1,1.5,1.5,sequences.geometric,geom,A,true
2024_P1_Q3.2,1,State a,1,"a=36pi","a=36pi|a=36π",sequences.geometric,geom,A,false
2024_P1_Q3.2,2,State r,1,r=1/4,r=1/4,sequences.geometric,geom,A,false
2024_P1_Q3.2,3,Substitute into S_n,1,"S10=36pi((1/4)^10-1)/(1/4-1)","S10=36pi((1/4)^10-1)/(1/4-1)",sequences.geometric,geom,A,false
2024_P1_Q3.2,4,Answer,1,S10=150.80,"S10=150.80|150.80",sequences.geometric,geom,A,true
2024_P1_Q3.3,1,Radius target,1,3/256,3/256,sequences.geometric,geom,A,false
2024_P1_Q3.3,2,Set up general term,1,"3/256=6(1/2)^(n-1)","3/256=6(1/2)^(n-1)",sequences.geometric,geom,A,false
2024_P1_Q3.3,3,Simplify,1,"1/512=(1/2)^(n-1)","1/512=(1/2)^(n-1)",sequences.geometric,geom,A,false
2024_P1_Q3.3,4,Answer n,1,n=10,n=10,sequences.geometric,geom,A,true
2024_P1_Q4.1,1,Substitute into f,1,"-5/9=a^2-1","-5/9=a^2-1",functions.exponential,exponential,A,false
2024_P1_Q4.1,2,Answer a,1,a=2/3,"a=2/3",functions.exponential,exponential,A,true
2024_P1_Q4.2,1,Range,1,y>-1,"y>-1",functions.exponential,exponential,A,true
2024_P1_Q4.4,1,Substitute 19,1,"19=(2/3)^x-1","19=(2/3)^x-1",functions.exponential,exponential,A,false
2024_P1_Q4.4,2,Answer x,1,x=-3,"x=-3",functions.exponential,exponential,A,false
2024_P1_Q4.4,3,C coordinates,1,"C(-3;19/8)","C(-3;19/8)|C(-3,19/8)",functions.exponential,exponential,A,true
2024_P1_Q5.1,1,State p,1,p=-1,"p=-1",functions.hyperbola,hyperbola,A,true
2024_P1_Q5.2,1,Substitute x=1,1,"y=1-3","y=1-3",functions.hyperbola,hyperbola,A,false
2024_P1_Q5.2,2,Answer,1,y=-2,"y=-2",functions.hyperbola,hyperbola,A,true
2024_P1_Q5.3,1,Substitute point,1,"1=a/(0-1)","1=a/(0-1)",functions.hyperbola,hyperbola,A,false
2024_P1_Q5.3,2,Solve for a,1,a=-3,"a=-3",functions.hyperbola,hyperbola,A,true
2024_P1_Q5.4,1,Find x-intercept,1,x=-1/2,"x=-1/2",functions.hyperbola,hyperbola,A,false
2024_P1_Q5.4,2,State interval,1,"x in [-1/2;1]","x in [-1/2;1]|-1/2<=x<1",functions.hyperbola,hyperbola,A,true
2024_P1_Q5.5,1,Reflection in y-axis,1,reflection y-axis,"reflection in y-axis|reflection in y",functions.hyperbola,hyperbola,A,false
2024_P1_Q5.5,2,Translation right,1,translate 2 right,"translation 2 units right|translate 2 right",functions.hyperbola,hyperbola,A,true
2024_P1_Q6.1,1,Method for x,1,"x=-4/(2(-1))","x=-4/(2(-1))",functions.parabola,parabola,A,false
2024_P1_Q6.1,2,Value of x,1,x=2,"x=2",functions.parabola,parabola,A,false
2024_P1_Q6.1,3,Value of y,1,y=9,"y=9|B(2;9)|B(2,9)",functions.parabola,parabola,A,true
2024_P1_Q6.2,1,Find A,1,"A(-1;0)","A(-1;0)|A(-1,0)",functions.parabola,parabola,A,false
2024_P1_Q6.2,2,Gradient of AC,1,m=2,"m=2",functions.parabola,parabola,A,false
2024_P1_Q6.2,3,Equation of AC,1,"y=2x+2","y=2x+2",functions.parabola,parabola,A,true
2024_P1_Q6.3,1,Difference EH,1,"EH=-x^2+4x+5-(2x+2)","EH=-x^2+4x+5-(2x+2)",functions.parabola,parabola,A,false
2024_P1_Q6.3,2,Simplify,1,"EH=-x^2+2x+3","EH=-x^2+2x+3",functions.parabola,parabola,A,false
2024_P1_Q6.3,3,x-value,1,x=1,x=1,functions.parabola,parabola,A,false
2024_P1_Q6.3,4,Length,1,"EH_max=4","EH_max=4|4",functions.parabola,parabola,A,true
2024_P1_Q6.4,1,k'(x),1,"k'(x)=-2x-2m+4","k'(x)=-2x-2m+4",functions.parabola,parabola,A,false
2024_P1_Q6.4,2,k'(x)=2,1,k'(x)=2,"k'(x)=2",functions.parabola,parabola,A,false
2024_P1_Q6.4,3,x=1-m,1,x=1-m,"x=1-m",functions.parabola,parabola,A,false
2024_P1_Q6.4,4,Substitute,1,"=2(1-m)+2","=2(1-m)+2",functions.parabola,parabola,A,false
2024_P1_Q6.4,5,Answer m,1,m=-2,"m=-2",functions.parabola,parabola,A,true
2024_P1_Q7.1,1,State i,n,1,"i=6.8/400; n=40","i=6.8/400; n=40",finance.compound,finance,A,false
2024_P1_Q7.1,2,Substitute,1,"A=5000(1+6.8/400)^40","A=5000(1+6.8/400)^40",finance.compound,finance,A,false
2024_P1_Q7.1,3,Answer,1,"A=14706.56","A=14706.56|14706.56",finance.compound,finance,A,true
2024_P1_Q7.2,1,Substitute into simple decay,1,"1/2=1-4i","1/2=1-4i",finance.simple,finance,A,false
2024_P1_Q7.2,2,Answer r,1,r=12.5%,"r=12.5%|r=12.5|12.5%",finance.simple,finance,A,true
2024_P1_Q7.3.1,1,Amount paid,1,"2300.98x60","2300.98x60|138058.80",finance.compound,finance,A,false
2024_P1_Q7.3.1,2,Interest,1,"=38058.80","38058.80",finance.compound,finance,A,true
2024_P1_Q7.3.2,1,State i,n,1,"i=13.5/1200; n=24","i=13.5/1200; n=24",finance.compound,finance,A,false
2024_P1_Q7.3.2,2,Substitute P formula,1,"P=2300.98[1-(1+13.5/1200)^-n]/(13.5/1200)","P=2300.98[1-(1+13.5/1200)^-n]/(13.5/1200)",finance.compound,finance,A,false
2024_P1_Q7.3.2,3,Logs,1,"-n=log(0.766)","-n=log(0.766)|-n=log_(1.01125)(0.766)",finance.compound,finance,A,false
2024_P1_Q7.3.2,4,Answer n,1,n=23.83,"n=23.83|n=23.80",finance.compound,finance,A,false
2024_P1_Q7.3.2,5,Months earlier,1,12 months,"12 months|12",finance.compound,finance,A,true
2024_P1_Q8.1.1,1,Derivative first term,1,3,3,calculus.polynomial,derivative,A,false
2024_P1_Q8.1.1,2,Derivative second term,1,-10x,-10x,calculus.polynomial,derivative,A,true
2024_P1_Q8.1.2,1,Rewrite as powers,1,"2x^-2-x^(7/3)","2x^-2-x^(7/3)",calculus.polynomial,derivative,A,false
2024_P1_Q8.1.2,2,Derivative of first,1,-4x^-3,"-4x^-3",calculus.polynomial,derivative,A,false
2024_P1_Q8.1.2,3,Derivative of second,1,"-(7/3)x^(4/3)","-(7/3)x^(4/3)",calculus.polynomial,derivative,A,true
2024_P1_Q8.2,1,Find m,1,m=-2,"m=-2",calculus.polynomial,derivative,A,false
2024_P1_Q8.2,2,Find y,1,y=-1,"y=-1",calculus.polynomial,derivative,A,false
2024_P1_Q8.2,3,Equation of tangent,1,"y=-2x+3","y=-2x+3",calculus.polynomial,derivative,A,true
2024_P1_Q8.3.1,1,Substitute into first principles,1,"f'(x)=lim(f(x+h)-f(x))/h","f'(x)=lim(f(x+h)-f(x))/h",calculus.first_principles,first_principles,A,false
2024_P1_Q8.3.1,2,Expand f(x+h),1,"f(x+h)=-6x^2-12xh-6h^2","f(x+h)=-6x^2-12xh-6h^2",calculus.first_principles,first_principles,A,false
2024_P1_Q8.3.1,3,Simplify,1,"-12xh-6h^2","-12xh-6h^2",calculus.first_principles,first_principles,A,false
2024_P1_Q8.3.1,4,Factor out h,1,"h(-12x-6h)","h(-12x-6h)",calculus.first_principles,first_principles,A,false
2024_P1_Q8.3.1,5,Answer,1,-12x,"-12x",calculus.first_principles,first_principles,A,true
2024_P1_Q8.3.2,1,Domain of inverse,1,x<=0,"x<=0",calculus.polynomial,derivative,A,true
2024_P1_Q8.3.3,1,Swap x and y,1,x=-6y^2,"x=-6y^2",calculus.polynomial,derivative,A,false
2024_P1_Q8.3.3,2,Solve for y,1,"y=±sqrt(-x/6)","y=±sqrt(-x/6)",calculus.polynomial,derivative,A,false
2024_P1_Q8.3.3,3,Choose negative branch,1,"y=-sqrt(-x/6)","y=-sqrt(-x/6)|y=-sqrt(-1/6x)",calculus.polynomial,derivative,A,true
2024_P1_Q9.1,1,k(x),1,"1/(x-2)","1/(x-2)|k(x)=1/(x-2)",functions.hyperbola,hyperbola,A,true
2024_P1_Q9.2,1,First x-intercept,1,x=1,x=1,functions.hyperbola,hyperbola,A,false
2024_P1_Q9.2,2,Second x-intercept,1,x=5/2,"x=5/2|x=2.5",functions.hyperbola,hyperbola,A,true
2024_P1_Q9.3,1,Method,1,"5/2+1/2=3","5/2+1/2=3|3",functions.hyperbola,hyperbola,A,true
2024_P1_Q9.4,1,Point of inflection,1,7/4,"7/4|1.75",calculus.cubic,derivative,A,true
2024_P1_Q9.5,1,Range for k,1,"-9<k<-8","-9<k<-8",calculus.cubic,derivative,A,true
2024_P1_Q10.1,1,Set s'(t)=0,1,"-6t+18=0","-6t+18=0",calculus.kinematics,kinematics,A,false
2024_P1_Q10.1,2,Solve for t,1,t=3,"t=3",calculus.kinematics,kinematics,A,false
2024_P1_Q10.1,3,Answer s'(3),1,s'(3)=27,"s'(3)=27|27",calculus.kinematics,kinematics,A,true
2024_P1_Q10.2,1,Factorise,1,"-3t(t-6)=0","-3t(t-6)=0",calculus.kinematics,kinematics,A,false
2024_P1_Q10.2,2,Values t,1,"t=0 or t=6","t=0 or t=6",calculus.kinematics,kinematics,A,false
2024_P1_Q10.2,3,a and b,1,"a=-1; b=9","a=-1; b=9|a=-1 b=9",calculus.kinematics,kinematics,A,false
2024_P1_Q10.2,4,Substitute t=6,1,"s(6)=-216+324","s(6)=-216+324",calculus.kinematics,kinematics,A,false
2024_P1_Q10.2,5,Answer,1,s(6)=108,"s(6)=108|108",calculus.kinematics,kinematics,A,true
2024_P1_Q11.1,1,Venn diagram values,1,"2; 3; 4; 5; 6; 9; 11","2; 3; 4; 5; 6; 9; 11",probability.venn,venn,A,true
2024_P1_Q11.2,1,Numerator,1,"4+2+5+3","4+2+5+3|14",probability.venn,venn,A,false
2024_P1_Q11.2,2,Answer,1,7/20,"7/20|0.35",probability.venn,venn,A,true
2024_P1_Q11.3,1,P(M)xP(T),1,"22/40 x 16/40","22/40x16/40",probability.independent,probability,A,false
2024_P1_Q11.3,2,Simplify,1,11/50,"11/50",probability.independent,probability,A,false
2024_P1_Q11.3,3,P(M and T),1,7/40,"7/40",probability.independent,probability,A,false
2024_P1_Q11.3,4,Not independent,1,not independent,"not independent|events M and T are not independent",probability.independent,probability,A,true
2024_P1_Q12.1,1,Letters,1,"26x26","26x26",probability.counting,counting,A,false
2024_P1_Q12.1,2,Digits and answer,1,"10x10=67600","10x10=67600|67600",probability.counting,counting,A,true
2024_P1_Q12.2,1,18,1,18,18,probability.counting,counting,A,false
2024_P1_Q12.2,2,19,1,19,19,probability.counting,counting,A,false
2024_P1_Q12.2,3,9,1,9,9,probability.counting,counting,A,false
2024_P1_Q12.2,4,5 and answer,1,"5; 15390","5; 15390|15390",probability.counting,counting,A,true
2024_P1_Q12.3,1,24x9x25x5,1,"24x9x25x5","24x9x25x5",probability.counting,counting,A,false
2024_P1_Q12.3,2,Percentage,1,75.44%,"75.44%|75.44",probability.counting,counting,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2024_P1.csv"
csv_path.write_text(csv_content, encoding="utf-8")
lines = csv_path.read_text(encoding="utf-8").splitlines()
print(f"2024 P1 CSV: wrote {len(lines)} lines")
print("OK")

2024 P1 CSV: wrote 142 lines
OK


In [60]:
test_source = r'''"""
Test harness for step_grader.py
"""
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.tutor.step_grader import grade_steps

CASES = [
    # ============ 2023 regression (compact) ============
    ("23P1 Q1.1.1", "2023_P1_Q1.1.1",
     ["(x+4)(x-3)=0", "x+4=0 or x-3=0", "x=-4 or x=3"], 3, [], "", False),
    ("23P1 Q1.1.2", "2023_P1_Q1.1.2",
     ["3x^2-2x-6=0", "x=(2\u00b1\u221a76)/6", "x=1.79", "x=-1.12"], 4, [], "", False),
    ("23P1 Q1.2 mA", "2023_P1_Q1.2",
     ["x=2y-2", "1/(2y-2)+1/y=1", "2y^2-5y+2=0", "y=1/2 or y=2", "x=-1 or x=2"],
     5, [], "", False),
    ("23P1 Q3.2", "2023_P1_Q3.2",
     ["S22=22/2[2a+21(3)]", "S22=22a+693", "S_inf=3a/2", "22a+693=3a/2+734", "a=2"],
     5, [], "", False),
    ("23P1 Q5.6 mA", "2023_P1_Q5.6",
     ["-2x+k=8/x", "-2x^2+kx-8=0", "k^2-64<0", "-8<k<8"], 4, [], "", False),
    ("23P1 Q7.1", "2023_P1_Q7.1",
     ["f'(x)=lim(f(x+h)-f(x))/h", "f(x+h)=-4x^2-8xh-4h^2",
      "-8xh-4h^2", "h(-8x-4h)", "-8x"], 5, [], "", False),
    ("23P1 Q8.4", "2023_P1_Q8.4",
     ["f''(x)=-6x+12", "f''(x)=0", "x=2", "y=2", "f'(2)=3", "g(x)=3x-4"],
     6, [], "", False),
    ("23P2 Q1.1", "2023_P2_Q1.1",
     ["a=-23.85", "b=0.23", "y_hat=-23.85+0.23x"], 3, [], "", False),
    ("23P2 Q3.2", "2023_P2_Q3.2",
     ["m_SN=(5-(-3))/(4-(-2))", "m_SN=4/3"], 2, [], "", False),
    ("23P2 Q5.1.1", "2023_P2_Q5.1.1",
     ["x^2+y^2=r^2", "x=-2sqrt(2)", "cos beta=-2sqrt(2)/3"], 3, [], "", False),

    # ============ 2024 P1 Q1 ============
    ("24P1 Q1.1.1", "2024_P1_Q1.1.1", ["x=0", "x=3"], 2, [], "", False),
    ("24P1 Q1.1.2", "2024_P1_Q1.1.2",
     ["2x^2-4x+1=0", "x=(4\u00b1sqrt(8))/(4)", "x=1.71", "x=0.29"], 4, [], "", False),
    ("24P1 Q1.1.3", "2024_P1_Q1.1.3",
     ["(x-3)(x+1)>0", "x=3 or x=-1", "x<-1 or x>3"], 4, [], "", False),
    ("24P1 Q1.1.4", "2024_P1_Q1.1.4",
     ["2^2x-2^2*2^x-32=0", "(2^x-8)(2^x+4)=0", "2^x=8", "2^x != -4", "x=3"],
     5, [], "", False),
    ("24P1 Q1.1.5", "2024_P1_Q1.1.5",
     ["sqrt(-2x+4)=2+x", "-2x+4=(2+x)^2", "x^2+6x=0", "x=0"], 4, [], "", False),
    ("24P1 Q1.2 mA", "2024_P1_Q1.2",
     ["y=3-2x", "(3-2x)^2+(3-2x)x=2", "2x^2-9x+7=0", "x=7/2 or x=1", "y=-4 or y=1"],
     5, [], "", False),
    ("24P1 Q1.2 mB", "2024_P1_Q1.2",
     ["x=3-y", "y^2+(3-y)^2=2", "y^2+3y-4=0", "y=-4 or y=1", "x=7/2 or x=1"],
     5, [], "", False),
    ("24P1 Q1.3", "2024_P1_Q1.3",
     ["1x2x3x...x(n+1)=1x2x3x...xnx(n+1)", "n is odd"], 2, [], "", False),

    # ============ 2024 P1 Q2 ============
    ("24P1 Q2.1.1", "2024_P1_Q2.1.1",
     ["S20=20/2[2(7)+(20-1)(5)]", "S20=1090"], 2, [], "", False),
    ("24P1 Q2.1.2", "2024_P1_Q2.1.2",
     ["Tn=5n+2", "S75-S20=14400-1090", "sum from 21 to 75", "=13310"], 4, [], "", False),
    ("24P1 Q2.2.1", "2024_P1_Q2.2.1",
     ["=2(98)-1=195", "T98=9632-195", "T98=9437"], 3, [], "", False),
    ("24P1 Q2.2.2", "2024_P1_Q2.2.2",
     ["2a=2", "a=1", "b=-2", "c=29", "Tn=n^2-2n+29"], 5, [], "", False),

    # ============ 2024 P1 Q3 ============
    ("24P1 Q3.1", "2024_P1_Q3.1", ["1.5"], 1, [], "", False),
    ("24P1 Q3.2", "2024_P1_Q3.2",
     ["a=36pi", "r=1/4", "S10=36pi((1/4)^10-1)/(1/4-1)", "S10=150.80"], 4, [], "", False),
    ("24P1 Q3.3", "2024_P1_Q3.3",
     ["3/256", "3/256=6(1/2)^(n-1)", "1/512=(1/2)^(n-1)", "n=10"], 4, [], "", False),

    # ============ 2024 P1 Q4 ============
    ("24P1 Q4.1", "2024_P1_Q4.1", ["-5/9=a^2-1", "a=2/3"], 2, [], "", False),
    ("24P1 Q4.2", "2024_P1_Q4.2", ["y>-1"], 1, [], "", False),
    ("24P1 Q4.4", "2024_P1_Q4.4",
     ["19=(2/3)^x-1", "x=-3", "C(-3;19/8)"], 3, [], "", False),

    # ============ 2024 P1 Q5 ============
    ("24P1 Q5.1", "2024_P1_Q5.1", ["p=-1"], 1, [], "", False),
    ("24P1 Q5.2", "2024_P1_Q5.2", ["y=1-3", "y=-2"], 2, [], "", False),
    ("24P1 Q5.3", "2024_P1_Q5.3", ["1=a/(0-1)", "a=-3"], 2, [], "", False),
    ("24P1 Q5.4", "2024_P1_Q5.4", ["x=-1/2", "x in [-1/2;1]"], 3, [], "", False),
    ("24P1 Q5.5", "2024_P1_Q5.5",
     ["reflection y-axis", "translate 2 right"], 2, [], "", False),

    # ============ 2024 P1 Q6 ============
    ("24P1 Q6.1", "2024_P1_Q6.1",
     ["x=-4/(2(-1))", "x=2", "y=9"], 3, [], "", False),
    ("24P1 Q6.2", "2024_P1_Q6.2",
     ["A(-1;0)", "m=2", "y=2x+2"], 3, [], "", False),
    ("24P1 Q6.3", "2024_P1_Q6.3",
     ["EH=-x^2+4x+5-(2x+2)", "EH=-x^2+2x+3", "x=1", "EH_max=4"],
     4, [], "", False),
    ("24P1 Q6.4", "2024_P1_Q6.4",
     ["k'(x)=-2x-2m+4", "k'(x)=2", "x=1-m", "=2(1-m)+2", "m=-2"], 5, [], "", False),

    # ============ 2024 P1 Q7 ============
    ("24P1 Q7.1", "2024_P1_Q7.1",
     ["i=6.8/400; n=40", "A=5000(1+6.8/400)^40", "A=14706.56"], 3, [], "", False),
    ("24P1 Q7.2", "2024_P1_Q7.2", ["1/2=1-4i", "r=12.5%"], 2, [], "", False),
    ("24P1 Q7.3.1", "2024_P1_Q7.3.1",
     ["2300.98x60", "=38058.80"], 2, [], "", False),
    ("24P1 Q7.3.2", "2024_P1_Q7.3.2",
     ["i=13.5/1200; n=24", "P=2300.98[1-(1+13.5/1200)^-n]/(13.5/1200)",
      "-n=log(0.766)", "n=23.83", "12 months"], 5, [], "", False),

    # ============ 2024 P1 Q8 ============
    ("24P1 Q8.1.1", "2024_P1_Q8.1.1", ["3", "-10x"], 2, [], "", False),
    ("24P1 Q8.1.2", "2024_P1_Q8.1.2",
     ["2x^-2-x^(7/3)", "-4x^-3", "-(7/3)x^(4/3)"], 3, [], "", False),
    ("24P1 Q8.2", "2024_P1_Q8.2", ["m=-2", "y=-1", "y=-2x+3"], 3, [], "", False),
    ("24P1 Q8.3.1", "2024_P1_Q8.3.1",
     ["f'(x)=lim(f(x+h)-f(x))/h", "f(x+h)=-6x^2-12xh-6h^2",
      "-12xh-6h^2", "h(-12x-6h)", "-12x"], 5, [], "", False),
    ("24P1 Q8.3.2", "2024_P1_Q8.3.2", ["x<=0"], 1, [], "", False),
    ("24P1 Q8.3.3", "2024_P1_Q8.3.3",
     ["x=-6y^2", "y=±sqrt(-x/6)", "y=-sqrt(-x/6)"], 3, [], "", False),

    # ============ 2024 P1 Q9 ============
    ("24P1 Q9.1", "2024_P1_Q9.1", ["1/(x-2)"], 1, [], "", False),
    ("24P1 Q9.2", "2024_P1_Q9.2", ["x=1", "x=5/2"], 2, [], "", False),
    ("24P1 Q9.3", "2024_P1_Q9.3", ["5/2+1/2=3"], 1, [], "", False),
    ("24P1 Q9.4", "2024_P1_Q9.4", ["7/4"], 1, [], "", False),
    ("24P1 Q9.5", "2024_P1_Q9.5", ["-9<k<-8"], 1, [], "", False),

    # ============ 2024 P1 Q10 ============
    ("24P1 Q10.1", "2024_P1_Q10.1",
     ["-6t+18=0", "t=3", "s'(3)=27"], 3, [], "", False),
    ("24P1 Q10.2", "2024_P1_Q10.2",
     ["-3t(t-6)=0", "t=0 or t=6", "a=-1; b=9", "s(6)=-216+324", "s(6)=108"],
     5, [], "", False),

    # ============ 2024 P1 Q11 ============
    ("24P1 Q11.1", "2024_P1_Q11.1",
     ["2; 3; 4; 5; 6; 9; 11"], 1, [], "", False),
    ("24P1 Q11.2", "2024_P1_Q11.2",
     ["4+2+5+3", "7/20"], 2, [], "", False),
    ("24P1 Q11.3", "2024_P1_Q11.3",
     ["22/40x16/40", "11/50", "7/40", "not independent"], 4, [], "", False),

    # ============ 2024 P1 Q12 ============
    ("24P1 Q12.1", "2024_P1_Q12.1", ["26x26", "10x10=67600"], 2, [], "", False),
    ("24P1 Q12.2", "2024_P1_Q12.2",
     ["18", "19", "9", "5; 15390"], 4, [], "", False),
    ("24P1 Q12.3", "2024_P1_Q12.3", ["24x9x25x5", "75.44%"], 2, [], "", False),
]


def run() -> int:
    passed = failed = 0
    out = []
    for name, qid, working, exp_awarded, exp_missing, note, _xfail in CASES:
        try:
            r = grade_steps(qid, working)
            awarded = r["awarded_marks"]
            missing = sorted(s["index"] for s in r["steps"] if s["match_status"] == "NOT_MATCHED")
            ok = (awarded == exp_awarded) and (missing == sorted(exp_missing))
        except Exception as e:
            awarded = missing = None
            ok = False
            note = "EXCEPTION: " + str(e) + " | " + note

        if ok:
            passed += 1
            status = "PASS "
        else:
            failed += 1
            status = "FAIL "

        out.append("  [" + status + "] " + name)
        if note:
            out.append("           note: " + note)
        if not ok:
            out.append("           expected " + str(exp_awarded) + " missing=" + str(sorted(exp_missing)))
            out.append("           got      " + str(awarded) + " missing=" + str(missing))

    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"])
    statuses = [s["match_status"] for s in r20["steps"]]
    if statuses == ["UNCERTAIN", "UNCERTAIN", "MATCHED"]:
        passed += 1
        out.append("  [PASS ] assertion / UNCERTAIN")
    else:
        failed += 1
        out.append("  [FAIL ] assertion / UNCERTAIN")

    print("\n".join(out))
    print()
    print("  " + str(passed) + " passed  |  " + str(failed) + " failed  |  "
          + str(len(CASES) + 1) + " total")
    return failed


if __name__ == "__main__":
    sys.exit(1 if run() else 0)
'''

tt_path = ROOT / "tests" / "test_step_grader.py"
tt_path.write_text(test_source, encoding="utf-8")
print(f"test file: {len(tt_path.read_text(encoding='utf-8').splitlines())} lines")
print("OK")

test file: 200 lines
OK


In [61]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)


STDERR: Traceback (most recent call last):
  File "C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\tests\test_step_grader.py", line 200, in <module>
    sys.exit(1 if run() else 0)
                  ~~~^^
  File "C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\tests\test_step_grader.py", line 182, in run
    r20 = grade_steps("2023_P1_Q1.1.1",
                      ["x = (-1 +/- sqrt(49))/2", "x = (-1 +/- 7)/2", "x = 3 or x = -4"])
  File "C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\step_grader.py", line 110, in grade_steps
    steps_map = load_steps(csv_path)
  File "C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\step_grader.py", line 38, in load_steps
    _load_one(p, by_q)
    ~~~~~~~~~^^^^^^^^^
  File "C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\step_grader.py", line 22, in _load_one
    "marks": int(row["marks"]),
             ~~~^^^^^^^^^^^^^^
ValueError: invalid literal 

In [62]:
import csv
from pathlib import Path

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2024_P1.csv"

print(f"Checking: {csv_path.name}")
print()

bad_rows = []
with open(csv_path, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    expected_cols = len(header)
    print(f"Header: {header}")
    print(f"Expected columns: {expected_cols}")
    print()
    for lineno, row in enumerate(reader, start=2):
        if len(row) != expected_cols:
            bad_rows.append((lineno, len(row), row))
            continue
        # Try int() on the marks column (index 3)
        try:
            int(row[3])
        except ValueError:
            bad_rows.append((lineno, len(row), row))

if not bad_rows:
    print("No bad rows found — issue is elsewhere")
else:
    print(f"Found {len(bad_rows)} bad row(s):")
    print()
    for lineno, ncols, row in bad_rows:
        print(f"Line {lineno}: {ncols} cols (expected {expected_cols})")
        for i, val in enumerate(row):
            print(f"   [{i}] {val!r}")
        print()

Checking: memo_step_marks_2024_P1.csv

Header: ['question_id', 'step_index', 'step_description', 'marks', 'memo_line', 'acceptable_forms', 'skill_id', 'method_tag', 'method_group', 'is_final']
Expected columns: 10

Found 2 bad row(s):

Line 85: 11 cols (expected 10)
   [0] '2024_P1_Q7.1'
   [1] '1'
   [2] 'State i'
   [3] 'n'
   [4] '1'
   [5] 'i=6.8/400; n=40'
   [6] 'i=6.8/400; n=40'
   [7] 'finance.compound'
   [8] 'finance'
   [9] 'A'
   [10] 'false'

Line 92: 11 cols (expected 10)
   [0] '2024_P1_Q7.3.2'
   [1] '1'
   [2] 'State i'
   [3] 'n'
   [4] '1'
   [5] 'i=13.5/1200; n=24'
   [6] 'i=13.5/1200; n=24'
   [7] 'finance.compound'
   [8] 'finance'
   [9] 'A'
   [10] 'false'



In [63]:
from pathlib import Path

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2024_P1.csv"
src = csv_path.read_text(encoding="utf-8")

before = src.count("State i,n")
src = src.replace("State i,n", "State i and n")
csv_path.write_text(src, encoding="utf-8")

print(f"Replaced {before} occurrence(s) of 'State i,n'")
print()

# Verify with csv.reader
import csv
with open(csv_path, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    ncols = len(header)
    bad = 0
    for lineno, row in enumerate(reader, start=2):
        if len(row) != ncols:
            bad += 1
            print(f"Still bad at line {lineno}: {row}")
        else:
            try:
                int(row[3])
            except ValueError:
                bad += 1
                print(f"Still bad marks at line {lineno}: {row[3]!r}")

print(f"Bad rows remaining: {bad}")
print("OK" if bad == 0 else "STILL BROKEN")

Replaced 2 occurrence(s) of 'State i,n'

Bad rows remaining: 0
OK


In [64]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] 23P1 Q1.1.1
  [PASS ] 23P1 Q1.1.2
  [PASS ] 23P1 Q1.2 mA
  [PASS ] 23P1 Q3.2
  [PASS ] 23P1 Q5.6 mA
  [PASS ] 23P1 Q7.1
  [PASS ] 23P1 Q8.4
  [PASS ] 23P2 Q1.1
  [PASS ] 23P2 Q3.2
  [PASS ] 23P2 Q5.1.1
  [PASS ] 24P1 Q1.1.1
  [PASS ] 24P1 Q1.1.2
  [PASS ] 24P1 Q1.1.3
  [PASS ] 24P1 Q1.1.4
  [PASS ] 24P1 Q1.1.5
  [PASS ] 24P1 Q1.2 mA
  [PASS ] 24P1 Q1.2 mB
  [PASS ] 24P1 Q1.3
  [PASS ] 24P1 Q2.1.1
  [PASS ] 24P1 Q2.1.2
  [PASS ] 24P1 Q2.2.1
  [PASS ] 24P1 Q2.2.2
  [PASS ] 24P1 Q3.1
  [PASS ] 24P1 Q3.2
  [PASS ] 24P1 Q3.3
  [PASS ] 24P1 Q4.1
  [PASS ] 24P1 Q4.2
  [PASS ] 24P1 Q4.4
  [PASS ] 24P1 Q5.1
  [PASS ] 24P1 Q5.2
  [PASS ] 24P1 Q5.3
  [FAIL ] 24P1 Q5.4
           expected 3 missing=[]
           got      2 missing=[]
  [FAIL ] 24P1 Q5.5
           expected 2 missing=[]
           got      1 missing=[1]
  [PASS ] 24P1 Q6.1
  [PASS ] 24P1 Q6.2
  [PASS ] 24P1 Q6.3
  [PASS ] 24P1 Q6.4
  [PASS ] 24P1 Q7.1
  [PASS ] 24P1 Q7.2
  [FAIL ] 24P1 Q7.3.1
           expected 2 missing

In [65]:
from pathlib import Path

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2024_P1.csv"
src = csv_path.read_text(encoding="utf-8")

# Fix 1: Q5.5 step 1 — add "reflection y-axis" as an acceptable form
old = '2024_P1_Q5.5,1,Reflection in y-axis,1,reflection y-axis,"reflection in y-axis|reflection in y",functions.hyperbola,hyperbola,A,false'
new = '2024_P1_Q5.5,1,Reflection in y-axis,1,reflection y-axis,"reflection in y-axis|reflection in y|reflection y-axis",functions.hyperbola,hyperbola,A,false'
assert old in src, "Q5.5 row not found"
src = src.replace(old, new)

# Fix 2: Q7.3.1 step 2 — add "=38058.80" as an acceptable form
old = '2024_P1_Q7.3.1,2,Interest,1,"=38058.80","38058.80",finance.compound,finance,A,true'
new = '2024_P1_Q7.3.1,2,Interest,1,"=38058.80","38058.80|=38058.80|38058.8",finance.compound,finance,A,true'
assert old in src, "Q7.3.1 row not found"
src = src.replace(old, new)

csv_path.write_text(src, encoding="utf-8")

import csv
with open(csv_path, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    ncols = len(next(reader))
    bad = 0
    for lineno, row in enumerate(reader, start=2):
        if len(row) != ncols or (len(row) > 3 and not row[3].lstrip('-').isdigit()):
            bad += 1
            print(f"bad line {lineno}: {row}")

print(f"Patched. Bad rows remaining: {bad}")
print("OK" if bad == 0 else "STILL BROKEN")

Patched. Bad rows remaining: 0
OK


In [66]:
test_path = ROOT / "tests" / "test_step_grader.py"
src = test_path.read_text(encoding="utf-8")

old = '''    ("24P1 Q5.4", "2024_P1_Q5.4", ["x=-1/2", "x in [-1/2;1]"], 3, [], "", False),'''
new = '''    ("24P1 Q5.4", "2024_P1_Q5.4", ["x=-1/2", "x in [-1/2;1]"], 2, [], "", False),'''

assert old in src, "Q5.4 test row not found"
test_path.write_text(src.replace(old, new), encoding="utf-8")
print("Test expectation for Q5.4 patched to 2 marks")
print("OK")

Test expectation for Q5.4 patched to 2 marks
OK


In [67]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] 23P1 Q1.1.1
  [PASS ] 23P1 Q1.1.2
  [PASS ] 23P1 Q1.2 mA
  [PASS ] 23P1 Q3.2
  [PASS ] 23P1 Q5.6 mA
  [PASS ] 23P1 Q7.1
  [PASS ] 23P1 Q8.4
  [PASS ] 23P2 Q1.1
  [PASS ] 23P2 Q3.2
  [PASS ] 23P2 Q5.1.1
  [PASS ] 24P1 Q1.1.1
  [PASS ] 24P1 Q1.1.2
  [PASS ] 24P1 Q1.1.3
  [PASS ] 24P1 Q1.1.4
  [PASS ] 24P1 Q1.1.5
  [PASS ] 24P1 Q1.2 mA
  [PASS ] 24P1 Q1.2 mB
  [PASS ] 24P1 Q1.3
  [PASS ] 24P1 Q2.1.1
  [PASS ] 24P1 Q2.1.2
  [PASS ] 24P1 Q2.2.1
  [PASS ] 24P1 Q2.2.2
  [PASS ] 24P1 Q3.1
  [PASS ] 24P1 Q3.2
  [PASS ] 24P1 Q3.3
  [PASS ] 24P1 Q4.1
  [PASS ] 24P1 Q4.2
  [PASS ] 24P1 Q4.4
  [PASS ] 24P1 Q5.1
  [PASS ] 24P1 Q5.2
  [PASS ] 24P1 Q5.3
  [PASS ] 24P1 Q5.4
  [PASS ] 24P1 Q5.5
  [PASS ] 24P1 Q6.1
  [PASS ] 24P1 Q6.2
  [PASS ] 24P1 Q6.3
  [PASS ] 24P1 Q6.4
  [PASS ] 24P1 Q7.1
  [PASS ] 24P1 Q7.2
  [PASS ] 24P1 Q7.3.1
  [PASS ] 24P1 Q7.3.2
  [PASS ] 24P1 Q8.1.1
  [PASS ] 24P1 Q8.1.2
  [PASS ] 24P1 Q8.2
  [PASS ] 24P1 Q8.3.1
  [PASS ] 24P1 Q8.3.2
  [PASS ] 24P1 Q8.3.3
  [PASS ] 

In [68]:
csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final
2024_P2_Q1.1,1,Compute a,1,a=39.46,"a=39.46|a=39.456",statistics.regression,regression,A,false
2024_P2_Q1.1,2,Compute b,1,b=-0.59,"b=-0.59|b=-0.590",statistics.regression,regression,A,false
2024_P2_Q1.1,3,Equation,1,"y=39.46-0.59x","y=39.46-0.59x|y_hat=39.46-0.59x",statistics.regression,regression,A,true
2024_P2_Q1.2,1,Correlation coefficient,1,r=-0.8,"r=-0.8",statistics.regression,regression,A,true
2024_P2_Q1.3,1,Substitute 29,1,"y=39.46-0.59(29)","y=39.46-0.59(29)",statistics.regression,regression,A,false
2024_P2_Q1.3,2,Answer,1,y=22.35,"y=22.35|22.35",statistics.regression,regression,A,true
2024_P2_Q1.4,1,Predicted value,1,y=18.33,"y=18.33|18.33",statistics.regression,regression,A,true
2024_P2_Q1.5,1,No influence,1,no influence,"no influence|stays the same|remains the same",statistics.stdev,stdev,A,true
2024_P2_Q1.6,1,Identify member,1,6,6,statistics.regression,regression,A,false
2024_P2_Q1.6,2,Difference,1,10,10,statistics.regression,regression,A,true
2024_P2_Q2.1,1,State median,1,median=65,"median=65|65",statistics.quartiles,quartiles,A,true
2024_P2_Q2.2,1,State Q1,1,Q1=44,"Q1=44|44",statistics.quartiles,quartiles,A,true
2024_P2_Q2.3,1,State Q3,1,Q3=84,"Q3=84|84",statistics.quartiles,quartiles,A,false
2024_P2_Q2.3,2,IQR,1,IQR=40,"IQR=40|40",statistics.quartiles,quartiles,A,true
2024_P2_Q2.5,1,Count employees,1,34 employees,"34 employees|34",statistics.quartiles,quartiles,A,false
2024_P2_Q2.5,2,Percentage,1,56.67%,"56.67%|56.67",statistics.quartiles,quartiles,A,true
2024_P2_Q2.6,1,Number of intervals,1,3,"3",statistics.quartiles,quartiles,A,false
2024_P2_Q2.6,2,Total time,1,90 minutes,"90 minutes|90 min|1.5 hours",statistics.quartiles,quartiles,A,true
2024_P2_Q3.1,1,Gradient formula,1,"m_DC=(-9-0)/(3-(-9))","m_DC=(-9-0)/(3-(-9))|m_DC=(0-(-9))/(-9-3)",analytical.gradient,gradient,A,false
2024_P2_Q3.1,2,Gradient answer,1,m_DC=-3/4,"m_DC=-3/4|m=−3/4",analytical.gradient,gradient,A,true
2024_P2_Q3.2,1,Substitute point,1,"0=-3/4(-9)+c","0=-3/4(-9)+c",analytical.line,line,A,false
2024_P2_Q3.2,2,Equation of DC,1,"y=-3/4x-27/4","y=-3/4x-27/4|y=-3/4x-6.75",analytical.line,line,A,true
2024_P2_Q3.3,1,Substitute B into line,1,"k=-3/4(-1)-27/4","k=-3/4(-1)-27/4",analytical.line,line,A,false
2024_P2_Q3.3,2,Answer k,1,k=-6,"k=-6",analytical.line,line,A,true
2024_P2_Q3.4,1,Distance formula,1,"DC=sqrt((3+9)^2+(-9-0)^2)","DC=sqrt((3+9)^2+(-9-0)^2)",analytical.distance,distance,A,false
2024_P2_Q3.4,2,Answer DC,1,DC=15,"DC=15|15 units",analytical.distance,distance,A,true
2024_P2_Q3.5,1,DB length,1,DB=5,"DB=5",analytical.distance,distance,A,false
2024_P2_Q3.5,2,Ratio DB/DC,1,DB/DC=1/3,"DB/DC=1/3|1/3",analytical.distance,distance,A,true
2024_P2_Q3.6,1,Ratio DM/DA,1,"DM/DA=1/3","DM/DA=1/3",analytical.area,area,A,false
2024_P2_Q3.6,2,Area rule,1,"Area MBD / Area ACD = 1/2(DM)(DB) sin D / 1/2(DA)(DC) sin D","1/2(DM)(DB)sinD/1/2(DA)(DC)sinD",analytical.area,area,A,false
2024_P2_Q3.6,3,Substitute,1,"=1/3 x 1/3","1/3x1/3",analytical.area,area,A,false
2024_P2_Q3.6,4,Answer ratio,1,=1/9,"1/9",analytical.area,area,A,true
2024_P2_Q3.7,1,Gradient of AD,1,m_AD=-4,"m_AD=-4",analytical.line,line,A,false
2024_P2_Q3.7,2,Equation AD,1,"y=-4x+3","y=-4x+3",analytical.line,line,A,false
2024_P2_Q3.7,3,Substitute into circle,1,"(x-3)^2+(y+9)^2=612","(x-3)^2+(y+9)^2=612",analytical.circle,circle,A,false
2024_P2_Q3.7,4,Standard form,1,"17x^2-102x-459=0","17x^2-102x-459=0|x^2-6x-27=0",analytical.circle,circle,A,false
2024_P2_Q3.7,5,x-values,1,"x=9 or x=-3","x=9 or x=-3|x=-3 or x=9",analytical.circle,circle,A,false
2024_P2_Q3.7,6,A coordinates,1,"A(-3;15)","A(-3;15)|A(-3,15)",analytical.circle,circle,A,true
2024_P2_Q4.1,1,L coordinates,1,"L(-1;-3)","L(-1;-3)|L(-1,-3)",analytical.circle,circle,A,true
2024_P2_Q4.2,1,Gradient MP,1,m_MP=3,"m_MP=3",analytical.line,line,A,false
2024_P2_Q4.2,2,Gradient ST,1,m_ST=-1/3,"m_ST=-1/3",analytical.line,line,A,false
2024_P2_Q4.2,3,Substitute point,1,"9=-1/3(3)+c","9=-1/3(3)+c",analytical.line,line,A,false
2024_P2_Q4.2,4,Equation of tangent,1,"y=-1/3x+10","y=-1/3x+10",analytical.line,line,A,true
2024_P2_Q4.3,1,Substitute point for r2,1,"(3-1)^2+(9-3)^2=r^2","(3-1)^2+(9-3)^2=r^2",analytical.circle,circle,A,false
2024_P2_Q4.3,2,Value of r squared,1,r^2=40,"r^2=40",analytical.circle,circle,A,false
2024_P2_Q4.3,3,LHS of equation,1,"(x-1)^2+(y-3)^2=40","(x-1)^2+(y-3)^2=40",analytical.circle,circle,A,false
2024_P2_Q4.3,4,Expand to general form,1,"x^2+y^2-2x-6y-30=0","x^2+y^2-2x-6y-30=0",analytical.circle,circle,A,true
2024_P2_Q4.4,1,Substitute y=1,1,"d^2+(1)^2-2d-6(1)-30=0","d^2+(1)^2-2d-6(1)-30=0",analytical.circle,circle,A,false
2024_P2_Q4.4,2,Standard form,1,"d^2-2d-35=0","d^2-2d-35=0",analytical.circle,circle,A,false
2024_P2_Q4.4,3,Answer d,1,d=7,"d=7",analytical.circle,circle,A,true
2024_P2_Q4.5,1,tan angle POV,1,"tan POV=3","tan POV=3|tanPOV=m_PO",analytical.gradient,gradient,A,false
2024_P2_Q4.5,2,Angle POV,1,POV=71.57,"POV=71.57|POV=71.565",analytical.gradient,gradient,A,false
2024_P2_Q4.5,3,Gradient RL,1,m_RL=1/2,"m_RL=1/2",analytical.gradient,gradient,A,false
2024_P2_Q4.5,4,Angle RUN,1,RUN=26.57,"RUN=26.57|RUN=26.565",analytical.gradient,gradient,A,false
2024_P2_Q4.5,5,Angle L,1,L=45,"L=45|L_hat=45|angle L=45",analytical.gradient,gradient,A,true
2024_P2_Q4.6,1,Gradient RM,1,m_RM=-1/3,"m_RM=-1/3",analytical.line,line,A,false
2024_P2_Q4.6,2,Gradient RT,1,m_RT=3,"m_RT=3",analytical.line,line,A,false
2024_P2_Q4.6,3,Product equals -1,1,"m_RT x m_PT=-1","m_RT x m_PT=-1|m_RT*m_PT=-1",analytical.line,line,A,false
2024_P2_Q4.6,4,Conclude perpendicular,1,"PT perp RT","PT ⟂ RT|PT perp RT|PT perpendicular RT",analytical.line,line,A,true
2024_P2_Q5.1.1,1,Value of r,1,r=5,r=5,trig.unitcircle,trig,A,false
2024_P2_Q5.1.1,2,Cos A,1,"cos A=-3/5","cosA=-3/5|cos A=-3/5",trig.unitcircle,trig,A,true
2024_P2_Q5.1.2,1,Double angle formula,1,"cos 2A=2cos^2 A-1","cos2A=2cos^2A-1|cos2A=2cos2A-1",trig.identity,trig,A,false
2024_P2_Q5.1.2,2,Substitute,1,"=2(-3/5)^2-1","2(-3/5)^2-1",trig.identity,trig,A,false
2024_P2_Q5.1.2,3,Answer,1,"cos 2A=-7/25","cos2A=-7/25|-7/25",trig.identity,trig,A,true
2024_P2_Q5.1.3,1,State x,1,x=-3,x=-3,trig.identity,trig,A,false
2024_P2_Q5.1.3,2,Compound formula,1,"sin(A-B)=sinA cosB-cosA sinB","sin(A-B)=sinAcosB-cosAsinB",trig.identity,trig,A,false
2024_P2_Q5.1.3,3,Substitute values,1,"=(-4/5)(-3/5)-(-3/5)(4/5)","(-4/5)(-3/5)-(-3/5)(4/5)",trig.identity,trig,A,false
2024_P2_Q5.1.3,4,Answer,1,"sin(A-B)=24/25","sin(A-B)=24/25|24/25",trig.identity,trig,A,true
2024_P2_Q5.2,1,Multiply by 2 over 2,1,"x2/2","x2/2|multiply by 2/2",trig.identity,trig,A,false
2024_P2_Q5.2,2,Double angle,1,"=sin(a-90)/4","sin(a-90)/4",trig.identity,trig,A,false
2024_P2_Q5.2,3,Co-function,1,"=-cos a/4","-cosa/4|-cos alpha/4",trig.identity,trig,A,false
2024_P2_Q5.2,4,Answer,1,"=-p/4","-p/4",trig.identity,trig,A,true
2024_P2_Q7.1,1,State x,1,x=90,"x=90|x=90°",trig.graph,graph,A,true
2024_P2_Q7.2,1,State interval,2,"x=-180 or x in (-90;0]","x=-180 or x in (-90;0]|x=-180° or -90°<x≤0°",trig.graph,graph,A,true
2024_P2_Q7.3.1,1,State period,1,period=180,"period=180|180",trig.graph,graph,A,true
2024_P2_Q7.4,1,Rewrite 2cos^2x as tanx,1,"2cos^2x=tanx","2cos^2x=tanx",trig.equation,trig,A,false
2024_P2_Q7.4,2,Use cos2x,1,"cos2x+1=tanx","cos2x+1=tanx",trig.equation,trig,A,false
2024_P2_Q7.4,3,Answer,1,"x=45+k.180","x=45+k.180|x=45°+k.180°",trig.equation,trig,A,true
2024_P2_Q8.1,1,Trig ratio,1,"tan DAC=DC/AC","tan DAC=DC/AC|tanDAC=16/AC",trig.3d,trig3d,A,false
2024_P2_Q8.1,2,Answer AC,1,AC=15,"AC=15|AC=15m",trig.3d,trig3d,A,true
2024_P2_Q8.2,1,Cosine rule,1,"(19)^2=x^2+(15)^2-2x(15)cos105.61","19^2=x^2+15^2-2(15)x cos105.61",trig.3d,trig3d,A,false
2024_P2_Q8.2,2,Quadratic form,1,"x^2+8.07x-136=0","x^2+8.07x-136=0",trig.3d,trig3d,A,false
2024_P2_Q8.2,3,Quadratic formula,1,"x=(-8.07±sqrt(8.07^2-4(1)(-136)))/2","x=(-8.07±sqrt(8.07^2-4(1)(-136)))/2",trig.3d,trig3d,A,false
2024_P2_Q8.2,4,Length BC,1,BC=8.30,"BC=8.30|BC=8.3",trig.3d,trig3d,A,false
2024_P2_Q8.2,5,Angle BEC,1,BEC=58,"BEC=58|BEC=58°",trig.3d,trig3d,A,false
2024_P2_Q8.2,6,Length EC,1,EC=5.19,"EC=5.19|EC=5.2",trig.3d,trig3d,A,false
2024_P2_Q8.2,7,Length DE,1,DE=10.81,"DE=10.81|DE=10.8",trig.3d,trig3d,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2024_P2.csv"
csv_path.write_text(csv_content, encoding="utf-8")
lines = csv_path.read_text(encoding="utf-8").splitlines()
print(f"2024 P2 CSV: wrote {len(lines)} lines")
print("OK")

2024 P2 CSV: wrote 88 lines
OK


In [69]:
test_path = ROOT / "tests" / "test_step_grader.py"
src = test_path.read_text(encoding="utf-8")

new_cases = '''    # ============ 2024 P2 ============
    ("24P2 Q1.1", "2024_P2_Q1.1",
     ["a=39.46", "b=-0.59", "y=39.46-0.59x"], 3, [], "", False),
    ("24P2 Q1.2", "2024_P2_Q1.2", ["r=-0.8"], 1, [], "", False),
    ("24P2 Q1.3", "2024_P2_Q1.3",
     ["y=39.46-0.59(29)", "y=22.35"], 2, [], "", False),
    ("24P2 Q1.4", "2024_P2_Q1.4", ["y=18.33"], 1, [], "", False),
    ("24P2 Q1.5", "2024_P2_Q1.5", ["no influence"], 1, [], "", False),
    ("24P2 Q1.6", "2024_P2_Q1.6", ["6", "10"], 2, [], "", False),
    ("24P2 Q2.1", "2024_P2_Q2.1", ["median=65"], 1, [], "", False),
    ("24P2 Q2.2", "2024_P2_Q2.2", ["Q1=44"], 1, [], "", False),
    ("24P2 Q2.3", "2024_P2_Q2.3", ["Q3=84", "IQR=40"], 2, [], "", False),
    ("24P2 Q2.5", "2024_P2_Q2.5", ["34 employees", "56.67%"], 2, [], "", False),
    ("24P2 Q2.6", "2024_P2_Q2.6", ["3", "90 minutes"], 2, [], "", False),
    ("24P2 Q3.1", "2024_P2_Q3.1",
     ["m_DC=(-9-0)/(3-(-9))", "m_DC=-3/4"], 2, [], "", False),
    ("24P2 Q3.2", "2024_P2_Q3.2",
     ["0=-3/4(-9)+c", "y=-3/4x-27/4"], 2, [], "", False),
    ("24P2 Q3.3", "2024_P2_Q3.3",
     ["k=-3/4(-1)-27/4", "k=-6"], 2, [], "", False),
    ("24P2 Q3.4", "2024_P2_Q3.4",
     ["DC=sqrt((3+9)^2+(-9-0)^2)", "DC=15"], 2, [], "", False),
    ("24P2 Q3.5", "2024_P2_Q3.5", ["DB=5", "DB/DC=1/3"], 2, [], "", False),
    ("24P2 Q3.6", "2024_P2_Q3.6",
     ["DM/DA=1/3", "1/2(DM)(DB)sinD/1/2(DA)(DC)sinD", "1/3x1/3", "1/9"],
     4, [], "", False),
    ("24P2 Q3.7", "2024_P2_Q3.7",
     ["m_AD=-4", "y=-4x+3", "(x-3)^2+(y+9)^2=612",
      "17x^2-102x-459=0", "x=9 or x=-3", "A(-3;15)"],
     6, [], "", False),
    ("24P2 Q4.1", "2024_P2_Q4.1", ["L(-1;-3)"], 1, [], "", False),
    ("24P2 Q4.2", "2024_P2_Q4.2",
     ["m_MP=3", "m_ST=-1/3", "9=-1/3(3)+c", "y=-1/3x+10"], 4, [], "", False),
    ("24P2 Q4.3", "2024_P2_Q4.3",
     ["(3-1)^2+(9-3)^2=r^2", "r^2=40", "(x-1)^2+(y-3)^2=40",
      "x^2+y^2-2x-6y-30=0"], 4, [], "", False),
    ("24P2 Q4.4", "2024_P2_Q4.4",
     ["d^2+(1)^2-2d-6(1)-30=0", "d^2-2d-35=0", "d=7"], 3, [], "", False),
    ("24P2 Q4.5", "2024_P2_Q4.5",
     ["tan POV=3", "POV=71.57", "m_RL=1/2", "RUN=26.57", "L=45"], 5, [], "", False),
    ("24P2 Q4.6", "2024_P2_Q4.6",
     ["m_RM=-1/3", "m_RT=3", "m_RT x m_PT=-1", "PT perp RT"], 4, [], "", False),
    ("24P2 Q5.1.1", "2024_P2_Q5.1.1", ["r=5", "cos A=-3/5"], 2, [], "", False),
    ("24P2 Q5.1.2", "2024_P2_Q5.1.2",
     ["cos 2A=2cos^2 A-1", "=2(-3/5)^2-1", "cos 2A=-7/25"], 3, [], "", False),
    ("24P2 Q5.1.3", "2024_P2_Q5.1.3",
     ["x=-3", "sin(A-B)=sinA cosB-cosA sinB",
      "=(-4/5)(-3/5)-(-3/5)(4/5)", "sin(A-B)=24/25"],
     4, [], "", False),
    ("24P2 Q5.2", "2024_P2_Q5.2",
     ["x2/2", "=sin(a-90)/4", "=-cos a/4", "=-p/4"], 4, [], "", False),
    ("24P2 Q7.1", "2024_P2_Q7.1", ["x=90"], 1, [], "", False),
    ("24P2 Q7.2", "2024_P2_Q7.2", ["x=-180 or x in (-90;0]"], 2, [], "", False),
    ("24P2 Q7.3.1", "2024_P2_Q7.3.1", ["period=180"], 1, [], "", False),
    ("24P2 Q7.4", "2024_P2_Q7.4",
     ["2cos^2x=tanx", "cos2x+1=tanx", "x=45+k.180"], 3, [], "", False),
    ("24P2 Q8.1", "2024_P2_Q8.1",
     ["tan DAC=DC/AC", "AC=15"], 2, [], "", False),
    ("24P2 Q8.2", "2024_P2_Q8.2",
     ["(19)^2=x^2+(15)^2-2x(15)cos105.61", "x^2+8.07x-136=0",
      "x=(-8.07±sqrt(8.07^2-4(1)(-136)))/2", "BC=8.30",
      "BEC=58", "EC=5.19", "DE=10.81"],
     7, [], "", False),
]

'''

# Insert before the closing bracket of CASES
marker = "]\n\n\ndef run() -> int:"
assert marker in src, "CASES end marker not found"
src = src.replace(marker, new_cases + "\n\ndef run() -> int:")

test_path.write_text(src, encoding="utf-8")
print(f"test file: {len(src.splitlines())} lines")
print("OK")

test file: 264 lines
OK


In [70]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] 23P1 Q1.1.1
  [PASS ] 23P1 Q1.1.2
  [PASS ] 23P1 Q1.2 mA
  [PASS ] 23P1 Q3.2
  [PASS ] 23P1 Q5.6 mA
  [PASS ] 23P1 Q7.1
  [PASS ] 23P1 Q8.4
  [PASS ] 23P2 Q1.1
  [PASS ] 23P2 Q3.2
  [PASS ] 23P2 Q5.1.1
  [PASS ] 24P1 Q1.1.1
  [PASS ] 24P1 Q1.1.2
  [PASS ] 24P1 Q1.1.3
  [PASS ] 24P1 Q1.1.4
  [PASS ] 24P1 Q1.1.5
  [PASS ] 24P1 Q1.2 mA
  [PASS ] 24P1 Q1.2 mB
  [PASS ] 24P1 Q1.3
  [PASS ] 24P1 Q2.1.1
  [PASS ] 24P1 Q2.1.2
  [PASS ] 24P1 Q2.2.1
  [PASS ] 24P1 Q2.2.2
  [PASS ] 24P1 Q3.1
  [PASS ] 24P1 Q3.2
  [PASS ] 24P1 Q3.3
  [PASS ] 24P1 Q4.1
  [PASS ] 24P1 Q4.2
  [PASS ] 24P1 Q4.4
  [PASS ] 24P1 Q5.1
  [PASS ] 24P1 Q5.2
  [PASS ] 24P1 Q5.3
  [PASS ] 24P1 Q5.4
  [PASS ] 24P1 Q5.5
  [PASS ] 24P1 Q6.1
  [PASS ] 24P1 Q6.2
  [PASS ] 24P1 Q6.3
  [PASS ] 24P1 Q6.4
  [PASS ] 24P1 Q7.1
  [PASS ] 24P1 Q7.2
  [PASS ] 24P1 Q7.3.1
  [PASS ] 24P1 Q7.3.2
  [PASS ] 24P1 Q8.1.1
  [PASS ] 24P1 Q8.1.2
  [PASS ] 24P1 Q8.2
  [PASS ] 24P1 Q8.3.1
  [PASS ] 24P1 Q8.3.2
  [PASS ] 24P1 Q8.3.3
  [PASS ] 

In [71]:
# === Fix 1: step_grader.py — strip leading '=' in _norm ===
sg_path = ROOT / "src" / "tutor" / "step_grader.py"
src = sg_path.read_text(encoding="utf-8")

old = """    s = re.sub(r"^\\s*(therefore|so|hence|thus)\\b\\s*", "", s)
    s = re.sub(r"(\\d),(\\d)", r"\\1.\\2", s)
    return s"""
new = """    s = re.sub(r"^\\s*(therefore|so|hence|thus)\\b\\s*", "", s)
    s = re.sub(r"^=+\\s*", "", s)  # strip leading '=' (continuation lines)
    s = re.sub(r"(\\d),(\\d)", r"\\1.\\2", s)
    return s"""

assert old in src, "_norm anchor not found"
src = src.replace(old, new)
sg_path.write_text(src, encoding="utf-8")
print("step_grader.py: leading '=' now stripped in _norm")

# === Fix 2: CSV — add learner's cosine-rule form for Q8.2 step 1 ===
csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2024_P2.csv"
src = csv_path.read_text(encoding="utf-8")

old = '2024_P2_Q8.2,1,Cosine rule,1,"(19)^2=x^2+(15)^2-2x(15)cos105.61","19^2=x^2+15^2-2(15)x cos105.61",trig.3d,trig3d,A,false'
new = '2024_P2_Q8.2,1,Cosine rule,1,"(19)^2=x^2+(15)^2-2x(15)cos105.61","19^2=x^2+15^2-2(15)x cos105.61|(19)^2=x^2+(15)^2-2x(15)cos105.61|(19)^2=x^2+(15)^2-2(x)(15)cos105.61",trig.3d,trig3d,A,false'

assert old in src, "Q8.2 row not found"
src = src.replace(old, new)
csv_path.write_text(src, encoding="utf-8")
print("CSV: Q8.2 step 1 now has 3 acceptable forms")

print("OK")

step_grader.py: leading '=' now stripped in _norm
CSV: Q8.2 step 1 now has 3 acceptable forms
OK


In [72]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] 23P1 Q1.1.1
  [PASS ] 23P1 Q1.1.2
  [PASS ] 23P1 Q1.2 mA
  [PASS ] 23P1 Q3.2
  [PASS ] 23P1 Q5.6 mA
  [PASS ] 23P1 Q7.1
  [PASS ] 23P1 Q8.4
  [PASS ] 23P2 Q1.1
  [PASS ] 23P2 Q3.2
  [PASS ] 23P2 Q5.1.1
  [PASS ] 24P1 Q1.1.1
  [PASS ] 24P1 Q1.1.2
  [PASS ] 24P1 Q1.1.3
  [PASS ] 24P1 Q1.1.4
  [PASS ] 24P1 Q1.1.5
  [PASS ] 24P1 Q1.2 mA
  [PASS ] 24P1 Q1.2 mB
  [PASS ] 24P1 Q1.3
  [PASS ] 24P1 Q2.1.1
  [PASS ] 24P1 Q2.1.2
  [PASS ] 24P1 Q2.2.1
  [PASS ] 24P1 Q2.2.2
  [PASS ] 24P1 Q3.1
  [PASS ] 24P1 Q3.2
  [PASS ] 24P1 Q3.3
  [PASS ] 24P1 Q4.1
  [PASS ] 24P1 Q4.2
  [PASS ] 24P1 Q4.4
  [PASS ] 24P1 Q5.1
  [PASS ] 24P1 Q5.2
  [PASS ] 24P1 Q5.3
  [PASS ] 24P1 Q5.4
  [PASS ] 24P1 Q5.5
  [PASS ] 24P1 Q6.1
  [PASS ] 24P1 Q6.2
  [PASS ] 24P1 Q6.3
  [PASS ] 24P1 Q6.4
  [PASS ] 24P1 Q7.1
  [PASS ] 24P1 Q7.2
  [PASS ] 24P1 Q7.3.1
  [PASS ] 24P1 Q7.3.2
  [PASS ] 24P1 Q8.1.1
  [PASS ] 24P1 Q8.1.2
  [PASS ] 24P1 Q8.2
  [PASS ] 24P1 Q8.3.1
  [PASS ] 24P1 Q8.3.2
  [PASS ] 24P1 Q8.3.3
  [PASS ] 

In [73]:
csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final
2025_P1_Q1.1.1,1,First root,1,x=-5,x=-5,algebra.quadratic.solve,factorise,A,true
2025_P1_Q1.1.1,2,Second root,1,x=2,x=2,algebra.quadratic.solve,factorise,A,true
2025_P1_Q1.1.2,1,Rearrange standard form,1,"5x^2+9x+2=0","5x^2+9x+2=0",algebra.quadratic.solve,formula,A,false
2025_P1_Q1.1.2,2,Substitute into formula,1,"x=(-9±sqrt(41))/10","x=(-9±sqrt(41))/10|x=(-9±√41)/10",algebra.quadratic.solve,formula,A,false
2025_P1_Q1.1.2,3,First root,1,x=-0.26,"x=-0.26|x=-0,26",algebra.quadratic.solve,formula,A,true
2025_P1_Q1.1.2,4,Second root,1,x=-1.54,"x=-1.54|x=-1,54",algebra.quadratic.solve,formula,A,true
2025_P1_Q1.1.3,1,Standard form,1,"8x^2-2x>0","8x^2-2x>0",algebra.quadratic.inequality,inequality,A,false
2025_P1_Q1.1.3,2,Factorise,1,"2x(4x-1)>0","2x(4x-1)>0",algebra.quadratic.inequality,inequality,A,false
2025_P1_Q1.1.3,3,Critical values,1,"x=0; x=1/4","x=0; x=1/4|x=0 or x=1/4",algebra.quadratic.inequality,inequality,A,false
2025_P1_Q1.1.3,4,Solution,1,"x<0 or x>1/4","x<0 or x>1/4",algebra.quadratic.inequality,inequality,A,true
2025_P1_Q1.1.4,1,Factorise,1,"(2.2^x-1)(2^x-4)=0","(2.2^x-1)(2^x-4)=0",algebra.exponential.solve,exponential,A,false
2025_P1_Q1.1.4,2,Both equations,1,"2^x=1/2 or 2^x=4","2^x=1/2 or 2^x=4",algebra.exponential.solve,exponential,A,false
2025_P1_Q1.1.4,3,First x,1,x=-1,x=-1,algebra.exponential.solve,exponential,A,true
2025_P1_Q1.1.4,4,Second x,1,x=2,x=2,algebra.exponential.solve,exponential,A,true
2025_P1_Q1.1.5,1,Square both sides,1,"1/sqrt(x)+2=(1/sqrt(x))^2","1/sqrt(x)+2=(1/sqrt(x))^2",algebra.surd.solve,surd,A,false
2025_P1_Q1.1.5,2,Isolate surd,1,"1/sqrt(x)=1/sqrt(x)-2","1/sqrt(x)=1/sqrt(x)-2",algebra.surd.solve,surd,A,false
2025_P1_Q1.1.5,3,Square again,1,"(1/sqrt(x))^2=(1/sqrt(x)-2)^2","(1/sqrt(x))^2=(1/sqrt(x)-2)^2",algebra.surd.solve,surd,A,false
2025_P1_Q1.1.5,4,Standard form,1,"4x^2-5x+1=0","4x^2-5x+1=0",algebra.surd.solve,surd,A,false
2025_P1_Q1.1.5,5,Answer with selection,1,"x=1/4","x=1/4|x≠1, x=1/4",algebra.surd.solve,surd,A,true
2025_P1_Q1.2,1,Write eq 1,1,"x=y+2","x=y+2",algebra.simultaneous.solve,substitution,A,false
2025_P1_Q1.2,2,Substitute into eq 2,1,"5(y+2)y=(y+2)^2+6","5(y+2)y=(y+2)^2+6",algebra.simultaneous.solve,substitution,A,false
2025_P1_Q1.2,3,Standard form,1,"2y^2+3y-5=0","2y^2+3y-5=0",algebra.simultaneous.solve,substitution,A,false
2025_P1_Q1.2,4,y-values,1,"y=-5/2 or y=1","y=-5/2 or y=1|y=1 or y=-5/2",algebra.simultaneous.solve,substitution,A,true
2025_P1_Q1.2,5,x-values,1,"x=-1/2 or x=3","x=-1/2 or x=3|x=3 or x=-1/2",algebra.simultaneous.solve,substitution,A,true
2025_P1_Q1.2,1,Write eq 1,1,"y=x-2","y=x-2",algebra.simultaneous.solve,substitution,B,false
2025_P1_Q1.2,2,Substitute into eq 2,1,"5x(x-2)-6=x^2","5x(x-2)-6=x^2",algebra.simultaneous.solve,substitution,B,false
2025_P1_Q1.2,3,Standard form,1,"2x^2-5x-3=0","2x^2-5x-3=0",algebra.simultaneous.solve,substitution,B,false
2025_P1_Q1.2,4,x-values,1,"x=-1/2 or x=3","x=-1/2 or x=3|x=3 or x=-1/2",algebra.simultaneous.solve,substitution,B,true
2025_P1_Q1.2,5,y-values,1,"y=-5/2 or y=1","y=-5/2 or y=1|y=1 or y=-5/2",algebra.simultaneous.solve,substitution,B,true
2025_P1_Q2.1.1,1,Equate ratios,1,"(t-2)/(t+10)=(t+4)/(t-2)","(t-2)/(t+10)=(t+4)/(t-2)",sequences.geometric,geom,A,false
2025_P1_Q2.1.1,2,Cross multiply,1,"(t-2)^2=(t+4)(t+10)","(t-2)^2=(t+4)(t+10)",sequences.geometric,geom,A,false
2025_P1_Q2.1.1,3,Answer t,1,t=-2,t=-2,sequences.geometric,geom,A,true
2025_P1_Q2.1.2,1,State a and r,1,"a=8; r=-1/2","8 and -1/2|a=8, r=-1/2",sequences.geometric,geom,A,false
2025_P1_Q2.1.2,2,Substitute into Tn,1,"T25=8(-1/2)^24","T25=8(-1/2)^24",sequences.geometric,geom,A,false
2025_P1_Q2.1.2,3,Answer in exponential form,1,"T25=(1/2)^21","T25=(1/2)^21|(2)^-21|(8)^-7|(128)^-3|(2097152)^-1",sequences.geometric,geom,A,true
2025_P1_Q2.1.3,1,Substitute into S_inf,1,"S_inf=8/(1-(-1/2))","S_inf=8/(1-(-1/2))",sequences.geometric,geom,A,false
2025_P1_Q2.1.3,2,Answer,1,"S_inf=16/3","S_inf=16/3|5.33",sequences.geometric,geom,A,true
2025_P1_Q2.2.1,1,Compute T14-T6,1,"T14-T6=4x8","T14-T6=4x8|T14-T6=32",sequences.arithmetic,arith,A,false
2025_P1_Q2.2.1,2,Answer,1,"=32","32",sequences.arithmetic,arith,A,true
2025_P1_Q2.2.2,1,Number of terms,1,"n=118-k","n=118-k",sequences.arithmetic,arith,A,false
2025_P1_Q2.2.2,2,Last term,1,"T117=467","T117=467",sequences.arithmetic,arith,A,false
2025_P1_Q2.2.2,3,Substitute S formula,1,"S_(118-k)=(118-k)/2[4k-1+467]","S_(118-k)=(118-k)/2[4k-1+467]",sequences.arithmetic,arith,A,false
2025_P1_Q2.2.2,4,Standard form,1,"2k^2-3k-819=0","2k^2-3k-819=0",sequences.arithmetic,arith,A,false
2025_P1_Q2.2.2,5,Answer with selection,1,"k=21","k=21",sequences.arithmetic,arith,A,true
2025_P1_Q3.1,1,First differences,1,"35; 33; 31; 29","35; 33; 31; 29|35 33 31 29",sequences.quadratic,arith,A,false
2025_P1_Q3.1,2,Answer T5,1,T5=164,"T5=164|164",sequences.quadratic,arith,A,true
2025_P1_Q3.2,1,2a=-2,1,"2a=-2","2a=-2",sequences.quadratic,arith,A,false
2025_P1_Q3.2,2,3(-1)+b=35,1,"3(-1)+b=35","3(-1)+b=35",sequences.quadratic,arith,A,false
2025_P1_Q3.2,3,-1+38+c=36,1,"-1+38+c=36","-1+38+c=36",sequences.quadratic,arith,A,false
2025_P1_Q3.2,4,Write Tn,1,"Tn=-n^2+38n-1","Tn=-n^2+38n-1",sequences.quadratic,arith,A,true
2025_P1_Q3.3,1,Method for n,1,"n=-38/(2(-1))","n=-38/(2(-1))|T_n'=-2n+38=0",sequences.quadratic,arith,A,false
2025_P1_Q3.3,2,Value n,1,n=19,"n=19",sequences.quadratic,arith,A,false
2025_P1_Q3.3,3,Answer T19,1,T19=360,"T19=360|360",sequences.quadratic,arith,A,true
2025_P1_Q3.4,1,Solve for n,1,n=35,"n=35",sequences.quadratic,arith,A,true
2025_P1_Q4.1,1,Compute t,1,t=-1,"t=-1",functions.log,log,A,true
2025_P1_Q4.2,1,x-intercept A,1,"A(1;0)","A(1;0)|A(1,0)",functions.log,log,A,true
2025_P1_Q4.3,1,Swap x and y,1,"x=log_(1/3) y","x=log_(1/3) y",functions.log,log,A,false
2025_P1_Q4.3,2,Answer inverse,1,"y=3^-x","y=3^-x|y=(1/3)^x",functions.log,log,A,true
2025_P1_Q4.4,1,Asymptote,1,y=0,"y=0",functions.log,log,A,true
2025_P1_Q4.6,1,Range,1,"0<y<3","0<y<3|y in (0;3)",functions.log,log,A,true
2025_P1_Q5.1,1,Domain,1,"x in R, x != 3","x in R, x≠3|x∈R, x≠3",functions.hyperbola,hyperbola,A,true
2025_P1_Q5.2,1,Range,1,y<=8,"y<=8",functions.hyperbola,hyperbola,A,true
2025_P1_Q5.3.1,1,Interval answer,2,"3<=x<=5","3<=x<=5|x in [3;5]",functions.hyperbola,hyperbola,A,true
2025_P1_Q5.3.2,1,First interval,1,x<1,"x<1",functions.hyperbola,hyperbola,A,false
2025_P1_Q5.3.2,2,Second interval,1,x>5,"x>5",functions.hyperbola,hyperbola,A,true
2025_P1_Q5.4,1,p and q values,1,"p=3; q=8","p=3; q=8",functions.parabola,parabola,A,false
2025_P1_Q5.4,2,Substitute point,1,"6=a(5-3)^2+8","6=a(5-3)^2+8",functions.parabola,parabola,A,false
2025_P1_Q5.4,3,Simplify,1,"y=-1/2(x-3)^2+8","y=-1/2(x-3)^2+8",functions.parabola,parabola,A,true
2025_P1_Q5.5,1,Find x-intercepts,1,"x^2-6x-7=0","x^2-6x-7=0",functions.parabola,parabola,A,false
2025_P1_Q5.5,2,x=7 or x=-1,1,"x=7 or x=-1","x=7 or x=-1",functions.parabola,parabola,A,false
2025_P1_Q5.5,3,M coordinates,1,"M(-1;0)","M(-1;0)|M(-1,0)",functions.parabola,parabola,A,false
2025_P1_Q5.5,4,T coordinates,1,"T(7/2;0)","T(7/2;0)|T(3.5;0)",functions.parabola,parabola,A,false
2025_P1_Q5.5,5,Length MT,1,"MT=9/2","MT=9/2|MT=4.5",functions.parabola,parabola,A,true
2025_P1_Q5.6,1,Derivative,1,"f'(x)=-x+3","f'(x)=-x+3",calculus.polynomial,derivative,A,false
2025_P1_Q5.6,2,Gradient at 5,1,"m=f'(5)=-2","m=f'(5)=-2",calculus.polynomial,derivative,A,false
2025_P1_Q5.6,3,Substitute point,1,"6=-2(5)+c","6=-2(5)+c",calculus.polynomial,derivative,A,false
2025_P1_Q5.6,4,Equation of tangent,1,"y=-2x+16","y=-2x+16",calculus.polynomial,derivative,A,true
2025_P1_Q6.1,1,x-intercept,1,"(-p;0)","(-p;0)|(-p,0)",functions.hyperbola,hyperbola,A,true
2025_P1_Q6.2,1,q=1+c,1,"q=1+c","q=1+c",functions.hyperbola,hyperbola,A,false
2025_P1_Q6.2,2,f(x) form,1,"f(x)=a/(x+c)+1+c","f(x)=a/(x+c)+1+c",functions.hyperbola,hyperbola,A,false
2025_P1_Q6.2,3,Solve a and c,1,"a=2; c=-2","a=2; c=-2|a=2, c=-2",functions.hyperbola,hyperbola,A,false
2025_P1_Q6.2,4,q value,1,q=-1,"q=-1",functions.hyperbola,hyperbola,A,false
2025_P1_Q6.2,5,Answer f,1,"f(x)=2/(x-2)-1","f(x)=2/(x-2)-1",functions.hyperbola,hyperbola,A,true
2025_P1_Q6.3,1,Translation down,1,translate 1 down,"1 unit down|translation 1 unit down",functions.hyperbola,hyperbola,A,false
2025_P1_Q6.3,2,Or right,1,translate 1 right,"1 unit right|translation 1 unit right",functions.hyperbola,hyperbola,A,true
2025_P1_Q7.1,1,Substitute into compound,1,"A=40000(1+7.8%)^5","A=40000(1+7.8%)^5|A=40000(1+0.078)^5",finance.compound,finance,A,false
2025_P1_Q7.1,2,Answer,1,A=58230.94,"A=58230.94|58230.94",finance.compound,finance,A,true
2025_P1_Q7.2,1,i value,1,i=0.058/4,"i=0.058/4",finance.future_value,finance,A,false
2025_P1_Q7.2,2,Substitute into F,1,"F=2300((1+0.058/4)^n-1)/(0.058/4)","F=2300((1+0.058/4)^n-1)/(0.058/4)",finance.future_value,finance,A,false
2025_P1_Q7.2,3,Compute future value,1,F=66411.60,"F=66411.60|66411.60",finance.future_value,finance,A,false
2025_P1_Q7.2,4,Answer,1,F=66411.60,"66411.60|F=66411.60",finance.future_value,finance,A,true
2025_P1_Q7.3.1,1,Amount after 12 months,1,"A=900000(1+0.068/12)^12","A=900000(1+0.068/12)^12",finance.compound,finance,A,false
2025_P1_Q7.3.1,2,Substitute P formula,1,"915386.86=10000[1-(1+0.068/12)^-n]/(0.068/12)","915386.86=10000[1-(1+0.068/12)^-n]/(0.068/12)",finance.present_value,finance,A,false
2025_P1_Q7.3.1,3,Logs,1,"-n=log_(1+0.068/12)(0.4812)","-n=log_(1+0.068/12)(0.4812)",finance.present_value,finance,A,false
2025_P1_Q7.3.1,4,n value,1,n=129.42,"n=129.42|n=129.419",finance.present_value,finance,A,false
2025_P1_Q7.3.1,5,Final answer,1,n=133,"n=133|133",finance.present_value,finance,A,true
2025_P1_Q7.3.2,1,Balance after 129,1,"P=4173.55","P=4173.55|4173.55",finance.present_value,finance,A,false
2025_P1_Q7.3.2,2,Final payment,1,"=4197.21","=4197.21|4197.21",finance.present_value,finance,A,true
2025_P1_Q8.1,1,f(x+h),1,"f(x+h)=-2(x+h)+3","f(x+h)=-2(x+h)+3",calculus.first_principles,first_principles,A,false
2025_P1_Q8.1,2,Simplify numerator,1,"-2h","-2h",calculus.first_principles,first_principles,A,false
2025_P1_Q8.1,3,Substitute into formula,1,"f'(x)=lim(-2h)/h","f'(x)=lim(-2h)/h",calculus.first_principles,first_principles,A,false
2025_P1_Q8.1,4,Answer,1,-2,"-2",calculus.first_principles,first_principles,A,true
2025_P1_Q8.2.1,1,First term,1,-12x^3,"-12x^3|-12x3",calculus.polynomial,derivative,A,false
2025_P1_Q8.2.1,2,Second term,1,2,"2",calculus.polynomial,derivative,A,true
2025_P1_Q8.2.2,1,Rewrite as 2x^2 + x^-2,1,"y=2x^2+x^-2","y=2x^2+x^-2",calculus.polynomial,derivative,A,false
2025_P1_Q8.2.2,2,Derivative first term,1,"4x","4x",calculus.polynomial,derivative,A,false
2025_P1_Q8.2.2,3,Derivative second term,1,"-2x^-3","-2x^-3",calculus.polynomial,derivative,A,true
2025_P1_Q9.1,1,Find f prime,1,"f'(x)=3x^2-16x+5","f'(x)=3x^2-16x+5",calculus.cubic,derivative,A,false
2025_P1_Q9.1,2,Set f prime = 0,1,"f'(x)=0","f'(x)=0|3x^2-16x+5=0",calculus.cubic,derivative,A,false
2025_P1_Q9.1,3,x-value,1,"x=1/3 or x=5","x=1/3 or x=5|x=5 or x=1/3",calculus.cubic,derivative,A,false
2025_P1_Q9.1,4,E coordinates,1,"E(5;-36)","E(5;-36)|E(5,-36)",calculus.cubic,derivative,A,true
2025_P1_Q9.2,1,Second derivative,1,"f''(x)=6x-16","f''(x)=6x-16",calculus.cubic,derivative,A,false
2025_P1_Q9.2,2,f''<0,1,"f''(x)<0","6x-16<0",calculus.cubic,derivative,A,false
2025_P1_Q9.2,3,Answer,1,x<8/3,"x<8/3",calculus.cubic,derivative,A,true
2025_P1_Q9.3,1,x-intercepts,1,"(-1;0) and (7;0)","(-1;0) and (7;0)|(-1,0) and (7,0)",calculus.cubic,derivative,A,false
2025_P1_Q9.3,2,First interval,1,"-1<x<2","-1<x<2",calculus.cubic,derivative,A,false
2025_P1_Q9.3,3,Second interval,1,"8/3<x<7","8/3<x<7",calculus.cubic,derivative,A,true
2025_P1_Q9.4,1,Set up equation,1,"3x^2-16x+16=0","3x^2-16x+16=0|3x^2-16x+5=-11",calculus.cubic,derivative,A,false
2025_P1_Q9.4,2,x-values,1,"x=4 or x=4/3","x=4 or x=4/3|x=4/3 or x=4",calculus.cubic,derivative,A,false
2025_P1_Q9.4,3,First t,1,t=14,"t=14",calculus.cubic,derivative,A,false
2025_P1_Q9.4,4,Second t,1,"t=634/27","t=634/27|23.48|23.48",calculus.cubic,derivative,A,true
2025_P1_Q10.1,1,Equation for h,1,"2x+2h=50","2x+2h=50",calculus.optimisation,optimisation,A,false
2025_P1_Q10.1,2,Radius relation,1,"r=x/(2pi)","r=x/(2pi)",calculus.optimisation,optimisation,A,false
2025_P1_Q10.1,3,Substitute into V,1,"V=25x^2/(4pi)-x^3/(4pi)","V=25x^2/(4pi)-x^3/(4pi)",calculus.optimisation,optimisation,A,true
2025_P1_Q10.2,1,Derivative=0,1,"50x-3x^2=0","50x-3x^2=0",calculus.optimisation,optimisation,A,false
2025_P1_Q10.2,2,Factor,1,"x(50-3x)=0","x(50-3x)=0",calculus.optimisation,optimisation,A,false
2025_P1_Q10.2,3,Answer with selection,1,"x=50/3","x=50/3|x=16.67",calculus.optimisation,optimisation,A,true
2025_P1_Q11.1.1,1,Independence equation,1,"36/210=90/210 x e/210","36/210=90/210 x e/210|P(M and J)=P(M)P(J)",probability.independent,probability,A,false
2025_P1_Q11.1.1,2,Solve for e,1,e=84,e=84,probability.independent,probability,A,true
2025_P1_Q11.1.2,1,Find d,1,d=126,d=126,probability.tables,tables,A,false
2025_P1_Q11.1.2,2,Find b,1,b=72,b=72,probability.tables,tables,A,false
2025_P1_Q11.1.2,3,Answer,1,"72/210=12/35","72/210=12/35|12/35",probability.tables,tables,A,true
2025_P1_Q11.2,1,Set up equation,1,"3/4(3x)+1/4 x=7/12","3/4(3x)+1/4 x=7/12",probability.tables,tables,A,false
2025_P1_Q11.2,2,Solve,1,"30x=7","30x=7",probability.tables,tables,A,false
2025_P1_Q11.2,3,x value,1,x=7/30,x=7/30,probability.tables,tables,A,false
2025_P1_Q11.2,4,Answer cups,1,28 cups,"28 cups|28",probability.tables,tables,A,true
2025_P1_Q11.3.1,1,n(S),1,"n(S)=7!","n(S)=7!|7!=504",probability.counting,counting,A,true
2025_P1_Q11.3.2,1,Count outcomes,1,"5x6!+4x6!+3x6!+2x6!+1x6!","5x6!+4x6!+3x6!+2x6!+1x6!",probability.counting,counting,A,false
2025_P1_Q11.3.2,2,Simplify,1,"6!(15)","6!(15)|15x720",probability.counting,counting,A,false
2025_P1_Q11.3.2,3,Answer,1,15/56,"15/56|0.27",probability.counting,counting,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2025_P1.csv"
csv_path.write_text(csv_content, encoding="utf-8")
lines = csv_path.read_text(encoding="utf-8").splitlines()
print(f"2025 P1 CSV: wrote {len(lines)} lines")
print("OK")

2025 P1 CSV: wrote 141 lines
OK


In [74]:
import csv
csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2025_P1.csv"
with open(csv_path, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    ncols = len(header)
    bad = 0
    for lineno, row in enumerate(reader, start=2):
        if len(row) != ncols:
            bad += 1
            print(f"bad cols line {lineno}: {len(row)} vs {ncols}")
            print(f"  {row}")
        else:
            try:
                int(row[3])
            except ValueError:
                bad += 1
                print(f"bad marks line {lineno}: {row[3]!r}")
print(f"Bad rows: {bad}")
print("OK" if bad == 0 else "FIX NEEDED")

Bad rows: 0
OK


In [76]:
import re
test_path = ROOT / "tests" / "test_step_grader.py"
src = test_path.read_text(encoding="utf-8")

new_cases = '''    # ============ 2025 P1 ============
    ("25P1 Q1.1.1", "2025_P1_Q1.1.1", ["x=-5", "x=2"], 2, [], "", False),
    ("25P1 Q1.1.2", "2025_P1_Q1.1.2",
     ["5x^2+9x+2=0", "x=(-9\u00b1sqrt(41))/10", "x=-0.26", "x=-1.54"], 4, [], "", False),
    ("25P1 Q1.1.3", "2025_P1_Q1.1.3",
     ["8x^2-2x>0", "2x(4x-1)>0", "x=0; x=1/4", "x<0 or x>1/4"], 4, [], "", False),
    ("25P1 Q1.1.4", "2025_P1_Q1.1.4",
     ["(2.2^x-1)(2^x-4)=0", "2^x=1/2 or 2^x=4", "x=-1", "x=2"], 4, [], "", False),
    ("25P1 Q1.1.5", "2025_P1_Q1.1.5",
     ["1/sqrt(x)+2=(1/sqrt(x))^2", "1/sqrt(x)=1/sqrt(x)-2",
      "(1/sqrt(x))^2=(1/sqrt(x)-2)^2", "4x^2-5x+1=0", "x=1/4"],
     5, [], "", False),
    ("25P1 Q1.2 mA", "2025_P1_Q1.2",
     ["x=y+2", "5(y+2)y=(y+2)^2+6", "2y^2+3y-5=0",
      "y=-5/2 or y=1", "x=-1/2 or x=3"], 5, [], "", False),
    ("25P1 Q1.2 mB", "2025_P1_Q1.2",
     ["y=x-2", "5x(x-2)-6=x^2", "2x^2-5x-3=0",
      "x=-1/2 or x=3", "y=-5/2 or y=1"], 5, [], "", False),
    ("25P1 Q2.1.1", "2025_P1_Q2.1.1",
     ["(t-2)/(t+10)=(t+4)/(t-2)", "(t-2)^2=(t+4)(t+10)", "t=-2"], 3, [], "", False),
    ("25P1 Q2.1.2", "2025_P1_Q2.1.2",
     ["a=8; r=-1/2", "T25=8(-1/2)^24", "T25=(1/2)^21"], 3, [], "", False),
    ("25P1 Q2.1.3", "2025_P1_Q2.1.3",
     ["S_inf=8/(1-(-1/2))", "S_inf=16/3"], 2, [], "", False),
    ("25P1 Q2.2.1", "2025_P1_Q2.2.1",
     ["T14-T6=4x8", "=32"], 2, [], "", False),
    ("25P1 Q2.2.2", "2025_P1_Q2.2.2",
     ["n=118-k", "T117=467", "S_(118-k)=(118-k)/2[4k-1+467]",
      "2k^2-3k-819=0", "k=21"], 5, [], "", False),
    ("25P1 Q3.1", "2025_P1_Q3.1",
     ["35; 33; 31; 29", "T5=164"], 2, [], "", False),
    ("25P1 Q3.2", "2025_P1_Q3.2",
     ["2a=-2", "3(-1)+b=35", "-1+38+c=36", "Tn=-n^2+38n-1"], 4, [], "", False),
    ("25P1 Q3.3", "2025_P1_Q3.3",
     ["n=-38/(2(-1))", "n=19", "T19=360"], 3, [], "", False),
    ("25P1 Q3.4", "2025_P1_Q3.4", ["n=35"], 1, [], "", False),
    ("25P1 Q4.1", "2025_P1_Q4.1", ["t=-1"], 1, [], "", False),
    ("25P1 Q4.2", "2025_P1_Q4.2", ["A(1;0)"], 1, [], "", False),
    ("25P1 Q4.3", "2025_P1_Q4.3", ["x=log_(1/3) y", "y=3^-x"], 2, [], "", False),
    ("25P1 Q5.1", "2025_P1_Q5.1", ["x in R, x != 3"], 1, [], "", False),
    ("25P1 Q5.2", "2025_P1_Q5.2", ["y<=8"], 1, [], "", False),
    ("25P1 Q5.3.1", "2025_P1_Q5.3.1", ["3<=x<=5"], 2, [], "", False),
    ("25P1 Q5.4", "2025_P1_Q5.4",
     ["p=3; q=8", "6=a(5-3)^2+8", "y=-1/2(x-3)^2+8"], 3, [], "", False),
    ("25P1 Q5.5", "2025_P1_Q5.5",
     ["x^2-6x-7=0", "x=7 or x=-1", "M(-1;0)", "T(7/2;0)", "MT=9/2"],
     5, [], "", False),
    ("25P1 Q5.6", "2025_P1_Q5.6",
     ["f'(x)=-x+3", "m=f'(5)=-2", "6=-2(5)+c", "y=-2x+16"], 4, [], "", False),
    ("25P1 Q6.1", "2025_P1_Q6.1", ["(-p;0)"], 1, [], "", False),
    ("25P1 Q6.2", "2025_P1_Q6.2",
     ["q=1+c", "f(x)=a/(x+c)+1+c", "a=2; c=-2", "q=-1", "f(x)=2/(x-2)-1"],
     5, [], "", False),
    ("25P1 Q7.1", "2025_P1_Q7.1",
     ["A=40000(1+7.8%)^5", "A=58230.94"], 2, [], "", False),
    ("25P1 Q7.2", "2025_P1_Q7.2",
     ["i=0.058/4", "F=2300((1+0.058/4)^n-1)/(0.058/4)",
      "F=66411.60", "F=66411.60"], 4, [], "", False),
    ("25P1 Q7.3.1", "2025_P1_Q7.3.1",
     ["A=900000(1+0.068/12)^12",
      "915386.86=10000[1-(1+0.068/12)^-n]/(0.068/12)",
      "-n=log_(1+0.068/12)(0.4812)", "n=129.42", "n=133"], 5, [], "", False),
    ("25P1 Q7.3.2", "2025_P1_Q7.3.2",
     ["P=4173.55", "=4197.21"], 2, [], "", False),
    ("25P1 Q8.1", "2025_P1_Q8.1",
     ["f(x+h)=-2(x+h)+3", "-2h", "f'(x)=lim(-2h)/h", "-2"], 4, [], "", False),
    ("25P1 Q8.2.1", "2025_P1_Q8.2.1", ["-12x^3", "2"], 2, [], "", False),
    ("25P1 Q8.2.2", "2025_P1_Q8.2.2",
     ["y=2x^2+x^-2", "4x", "-2x^-3"], 3, [], "", False),
    ("25P1 Q9.1", "2025_P1_Q9.1",
     ["f'(x)=3x^2-16x+5", "f'(x)=0", "x=1/3 or x=5", "E(5;-36)"], 4, [], "", False),
    ("25P1 Q9.2", "2025_P1_Q9.2",
     ["f''(x)=6x-16", "f''(x)<0", "x<8/3"], 3, [], "", False),
    ("25P1 Q9.3", "2025_P1_Q9.3",
     ["(-1;0) and (7;0)", "-1<x<2", "8/3<x<7"], 3, [], "", False),
    ("25P1 Q9.4", "2025_P1_Q9.4",
     ["3x^2-16x+16=0", "x=4 or x=4/3", "t=14", "t=634/27"], 4, [], "", False),
    ("25P1 Q10.1", "2025_P1_Q10.1",
     ["2x+2h=50", "r=x/(2pi)", "V=25x^2/(4pi)-x^3/(4pi)"], 3, [], "", False),
    ("25P1 Q10.2", "2025_P1_Q10.2",
     ["50x-3x^2=0", "x(50-3x)=0", "x=50/3"], 3, [], "", False),
    ("25P1 Q11.1.1", "2025_P1_Q11.1.1",
     ["36/210=90/210 x e/210", "e=84"], 2, [], "", False),
    ("25P1 Q11.1.2", "2025_P1_Q11.1.2",
     ["d=126", "b=72", "72/210=12/35"], 3, [], "", False),
    ("25P1 Q11.2", "2025_P1_Q11.2",
     ["3/4(3x)+1/4 x=7/12", "30x=7", "x=7/30", "28 cups"], 4, [], "", False),
    ("25P1 Q11.3.1", "2025_P1_Q11.3.1", ["n(S)=7!"], 1, [], "", False),
    ("25P1 Q11.3.2", "2025_P1_Q11.3.2",
     ["5x6!+4x6!+3x6!+2x6!+1x6!", "6!(15)", "15/56"], 3, [], "", False),
'''

# Find the closing bracket of CASES followed by blank lines and def run()
pattern = re.compile(r"\n\]\s*\n\s*\n\s*\ndef run\(\)")
match = pattern.search(src)
assert match, "Could not find end of CASES block"

# Insert new_cases before the ']'
# Replace "]\n\n\ndef run" with "new_cases]\n\n\ndef run"
# The match starts at the newline before ]
insert_at = match.start() + 1  # position of ']'
# Find the closing bracket
close_bracket_pos = src.index("]", insert_at)
new_src = src[:close_bracket_pos] + new_cases + src[close_bracket_pos:]

test_path.write_text(new_src, encoding="utf-8")
print(f"Inserted at position {close_bracket_pos}")
print(f"test file: {len(new_src.splitlines())} lines")
print("OK")

Inserted at position 10358
test file: 354 lines
OK


In [77]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] 23P1 Q1.1.1
  [PASS ] 23P1 Q1.1.2
  [PASS ] 23P1 Q1.2 mA
  [PASS ] 23P1 Q3.2
  [PASS ] 23P1 Q5.6 mA
  [PASS ] 23P1 Q7.1
  [PASS ] 23P1 Q8.4
  [PASS ] 23P2 Q1.1
  [PASS ] 23P2 Q3.2
  [PASS ] 23P2 Q5.1.1
  [PASS ] 24P1 Q1.1.1
  [PASS ] 24P1 Q1.1.2
  [PASS ] 24P1 Q1.1.3
  [PASS ] 24P1 Q1.1.4
  [PASS ] 24P1 Q1.1.5
  [PASS ] 24P1 Q1.2 mA
  [PASS ] 24P1 Q1.2 mB
  [PASS ] 24P1 Q1.3
  [PASS ] 24P1 Q2.1.1
  [PASS ] 24P1 Q2.1.2
  [PASS ] 24P1 Q2.2.1
  [PASS ] 24P1 Q2.2.2
  [PASS ] 24P1 Q3.1
  [PASS ] 24P1 Q3.2
  [PASS ] 24P1 Q3.3
  [PASS ] 24P1 Q4.1
  [PASS ] 24P1 Q4.2
  [PASS ] 24P1 Q4.4
  [PASS ] 24P1 Q5.1
  [PASS ] 24P1 Q5.2
  [PASS ] 24P1 Q5.3
  [PASS ] 24P1 Q5.4
  [PASS ] 24P1 Q5.5
  [PASS ] 24P1 Q6.1
  [PASS ] 24P1 Q6.2
  [PASS ] 24P1 Q6.3
  [PASS ] 24P1 Q6.4
  [PASS ] 24P1 Q7.1
  [PASS ] 24P1 Q7.2
  [PASS ] 24P1 Q7.3.1
  [PASS ] 24P1 Q7.3.2
  [PASS ] 24P1 Q8.1.1
  [PASS ] 24P1 Q8.1.2
  [PASS ] 24P1 Q8.2
  [PASS ] 24P1 Q8.3.1
  [PASS ] 24P1 Q8.3.2
  [PASS ] 24P1 Q8.3.3
  [PASS ] 

In [78]:
# === Fix 1: step_grader.py — normalise != to ≠ ===
sg_path = ROOT / "src" / "tutor" / "step_grader.py"
src = sg_path.read_text(encoding="utf-8")

old = """    s = re.sub(r"^=+\\s*", "", s)  # strip leading '=' (continuation lines)
    s = re.sub(r"(\\d),(\\d)", r"\\1.\\2", s)
    return s"""
new = """    s = re.sub(r"^=+\\s*", "", s)  # strip leading '=' (continuation lines)
    s = s.replace("!=", "\\u2260")  # != to ≠
    s = re.sub(r"(\\d),(\\d)", r"\\1.\\2", s)
    return s"""

assert old in src, "_norm anchor not found"
src = src.replace(old, new)
sg_path.write_text(src, encoding="utf-8")
print("step_grader.py: != now normalised to ≠")

# === Fix 2: CSV — add memo form to Q9.2 step 2 ===
csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2025_P1.csv"
src = csv_path.read_text(encoding="utf-8")

old = "2025_P1_Q9.2,2,f''<0,1,\"f''(x)<0\",\"6x-16<0\",calculus.cubic,derivative,A,false"
new = "2025_P1_Q9.2,2,f''<0,1,\"f''(x)<0\",\"f''(x)<0|6x-16<0\",calculus.cubic,derivative,A,false"

assert old in src, "Q9.2 row not found"
src = src.replace(old, new)
csv_path.write_text(src, encoding="utf-8")
print("CSV: Q9.2 step 2 now has both forms")

print("OK")

step_grader.py: != now normalised to ≠
CSV: Q9.2 step 2 now has both forms
OK


In [80]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] 23P1 Q1.1.1
  [PASS ] 23P1 Q1.1.2
  [PASS ] 23P1 Q1.2 mA
  [PASS ] 23P1 Q3.2
  [PASS ] 23P1 Q5.6 mA
  [PASS ] 23P1 Q7.1
  [PASS ] 23P1 Q8.4
  [PASS ] 23P2 Q1.1
  [PASS ] 23P2 Q3.2
  [PASS ] 23P2 Q5.1.1
  [PASS ] 24P1 Q1.1.1
  [PASS ] 24P1 Q1.1.2
  [PASS ] 24P1 Q1.1.3
  [PASS ] 24P1 Q1.1.4
  [PASS ] 24P1 Q1.1.5
  [PASS ] 24P1 Q1.2 mA
  [PASS ] 24P1 Q1.2 mB
  [PASS ] 24P1 Q1.3
  [PASS ] 24P1 Q2.1.1
  [PASS ] 24P1 Q2.1.2
  [PASS ] 24P1 Q2.2.1
  [PASS ] 24P1 Q2.2.2
  [PASS ] 24P1 Q3.1
  [PASS ] 24P1 Q3.2
  [PASS ] 24P1 Q3.3
  [PASS ] 24P1 Q4.1
  [PASS ] 24P1 Q4.2
  [PASS ] 24P1 Q4.4
  [PASS ] 24P1 Q5.1
  [PASS ] 24P1 Q5.2
  [PASS ] 24P1 Q5.3
  [PASS ] 24P1 Q5.4
  [PASS ] 24P1 Q5.5
  [PASS ] 24P1 Q6.1
  [PASS ] 24P1 Q6.2
  [PASS ] 24P1 Q6.3
  [PASS ] 24P1 Q6.4
  [PASS ] 24P1 Q7.1
  [PASS ] 24P1 Q7.2
  [PASS ] 24P1 Q7.3.1
  [PASS ] 24P1 Q7.3.2
  [PASS ] 24P1 Q8.1.1
  [PASS ] 24P1 Q8.1.2
  [PASS ] 24P1 Q8.2
  [PASS ] 24P1 Q8.3.1
  [PASS ] 24P1 Q8.3.2
  [PASS ] 24P1 Q8.3.3
  [PASS ] 

In [81]:
csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final
2025_P2_Q1.1,1,Compute a,1,"a=331397.20","a=331397.20|a=331397.2",statistics.regression,regression,A,false
2025_P2_Q1.1,2,Compute b,1,"b=-22988.32","b=-22988.32",statistics.regression,regression,A,false
2025_P2_Q1.1,3,Regression equation,1,"y_hat=331397.20-22988.32x","y_hat=331397.20-22988.32x|y=331397.20-22988.32x",statistics.regression,regression,A,true
2025_P2_Q1.2,1,Substitute 5,1,"y_hat=331397.20-22988.32(5)","y_hat=331397.20-22988.32(5)",statistics.regression,regression,A,false
2025_P2_Q1.2,2,Answer,1,"y_hat=216455.60","y_hat=216455.60|216455.60",statistics.regression,regression,A,true
2025_P2_Q1.3,1,Strong correlation,1,"strong correlation","strong correlation|r=-0.95",statistics.regression,regression,A,false
2025_P2_Q1.3,2,Valid prediction,1,valid,"valid|prediction is valid",statistics.regression,regression,A,true
2025_P2_Q1.4,1,Average decrease,1,"22988.32","22988.32|R22988.32",statistics.regression,regression,A,true
2025_P2_Q2.1.1,1,Total number of people,1,70,70,statistics.frequency,frequency,A,true
2025_P2_Q2.1.2,1,Subtract,1,"67-40","67-40",statistics.frequency,frequency,A,false
2025_P2_Q2.1.2,2,Answer,1,27,27,statistics.frequency,frequency,A,true
2025_P2_Q2.1.4,1,Skew description,1,skewed right,"skewed to the right|positively skewed|skewed right",statistics.frequency,frequency,A,true
2025_P2_Q2.2,1,Equation for mean,1,"(11+14+19+20+8+10+2+14+x)/9","(11+14+19+20+8+10+2+14+x)/9",statistics.mean,mean,A,false
2025_P2_Q2.2,2,Solve for x,1,x=10,x=10,statistics.mean,mean,A,false
2025_P2_Q2.2,3,Standard deviation,1,"sigma=5.23","sigma=5.23|5.23",statistics.stdev,stdev,A,false
2025_P2_Q2.2,4,Interval,1,"(6.77;17.23)","(6.77;17.23)|6.77<x<17.23",statistics.stdev,stdev,A,false
2025_P2_Q2.2,5,Answer count,1,3 players,"3 players|3",statistics.stdev,stdev,A,true
2025_P2_Q3.1,1,Distance formula,1,"QR=sqrt((-4-12)^2+(-6-2)^2)","QR=sqrt((-4-12)^2+(-6-2)^2)",analytical.distance,distance,A,false
2025_P2_Q3.1,2,Answer,1,"QR=8sqrt(5)","QR=8sqrt(5)|8√5",analytical.distance,distance,A,true
2025_P2_Q3.2,1,Gradient formula,1,"m_QR=(-6-2)/(-4-12)","m_QR=(-6-2)/(-4-12)|m_QR=(2-(-6))/(12-(-4))",analytical.gradient,gradient,A,false
2025_P2_Q3.2,2,Answer,1,"m_QR=1/2","m_QR=1/2",analytical.gradient,gradient,A,true
2025_P2_Q3.3,1,tan theta = gradient,1,"tan theta=1/2","tan theta=1/2|tanθ=m_QR",analytical.gradient,gradient,A,false
2025_P2_Q3.3,2,Answer theta,1,"theta=26.57","theta=26.57|θ=26.57",analytical.gradient,gradient,A,true
2025_P2_Q3.4,1,Substitute point,1,"-6=1/2(-4)+c","-6=1/2(-4)+c",analytical.line,line,A,false
2025_P2_Q3.4,2,Equation,1,"y=1/2x-4","y=1/2x-4",analytical.line,line,A,true
2025_P2_Q3.5,1,x_s,1,"x_s=15","x_s=15",analytical.line,line,A,false
2025_P2_Q3.5,2,y_s,1,"y_s=16","y_s=16|S(15;16)",analytical.line,line,A,true
2025_P2_Q3.6,1,Gradient PT,1,"m_PT=-2","m_PT=-2",analytical.line,line,A,false
2025_P2_Q3.6,2,Equation PT,1,"y=-2x+6","y=-2x+6",analytical.line,line,A,false
2025_P2_Q3.6,3,Solve for x,1,x=4,x=4,analytical.line,line,A,false
2025_P2_Q3.6,4,T coordinates,1,"T(4;-2)","T(4;-2)|T(4,-2)",analytical.line,line,A,true
2025_P2_Q3.7,1,Length PT,1,"PT=5sqrt(5)","PT=5sqrt(5)|5√5",analytical.area,area,A,false
2025_P2_Q3.7,2,Area of PQRS,1,"Area=200","Area=200|200 units^2",analytical.area,area,A,true
2025_P2_Q4.1,1,Value of p,1,p=-6,"p=-6",analytical.circle,circle,A,true
2025_P2_Q4.2,1,AM and MD,1,"AM=q+1; MD=q","AM=q+1; MD=q",analytical.circle,circle,A,false
2025_P2_Q4.2,2,Pythagoras,1,"(q+1)^2=(q-1)^2+q^2","(q+1)^2=(q-1)^2+q^2",analytical.circle,circle,A,false
2025_P2_Q4.2,3,Standard form,1,"q^2-4q=0","q^2-4q=0|q(q-4)=0",analytical.circle,circle,A,false
2025_P2_Q4.2,4,Answer q,1,q=4,q=4,analytical.circle,circle,A,true
2025_P2_Q4.3,1,LHS/RHS,1,"(x+6)^2+(y-4)^2=25","(x+6)^2+(y-4)^2=25",analytical.circle,circle,A,true
2025_P2_Q4.4,1,Answer radius,1,3,3,analytical.circle,circle,A,true
2025_P2_Q4.5,1,Substitute y=0,1,"(x+6)^2+(0-4)^2=25","(x+6)^2+(0-4)^2=25",analytical.circle,circle,A,false
2025_P2_Q4.5,2,A coordinates,1,"A(-9;0)","A(-9;0)|A(-9,0)",analytical.circle,circle,A,false
2025_P2_Q4.5,3,B coordinates,1,"B(-3;0)","B(-3;0)|B(-3,0)",analytical.circle,circle,A,true
2025_P2_Q4.6,1,m_MB,1,"m_MB=-4/3","m_MB=-4/3",analytical.line,line,A,false
2025_P2_Q4.6,2,m_BC,1,"m_BC=3/4","m_BC=3/4",analytical.line,line,A,false
2025_P2_Q4.6,3,Substitute point,1,"0=3/4(-3)+c","0=3/4(-3)+c",analytical.line,line,A,false
2025_P2_Q4.6,4,Answer,1,"y=3/4x+9/4","y=3/4x+9/4",analytical.line,line,A,true
2025_P2_Q4.7,1,x_C,1,x_C=-6,"x_C=-6",analytical.circle,circle,A,false
2025_P2_Q4.7,2,y_C,1,"y_C=-9/4","y_C=-9/4|y_C=-2.25",analytical.circle,circle,A,true
2025_P2_Q4.8,1,tan alpha=3/4,1,"tan alpha=3/4","tan alpha=3/4",analytical.gradient,gradient,A,false
2025_P2_Q4.8,2,alpha value,1,"alpha=36.87","alpha=36.87|α=36.87",analytical.gradient,gradient,A,false
2025_P2_Q4.8,3,tan beta,1,"tan beta=-3/4","tan beta=-3/4",analytical.gradient,gradient,A,false
2025_P2_Q4.8,4,beta value,1,"beta=143.13","beta=143.13|β=143.13",analytical.gradient,gradient,A,false
2025_P2_Q4.8,5,Answer ACB,1,"ACB=106.26","ACB=106.26|ACB=106.26°",analytical.gradient,gradient,A,true
2025_P2_Q5.1.1,1,Third side,1,"sqrt(k^2+1)","sqrt(k^2+1)|√(k²+1)",trig.identity,trig,A,false
2025_P2_Q5.1.1,2,Answer cos40,1,"cos40=k/sqrt(k^2+1)","cos40=k/sqrt(k^2+1)",trig.identity,trig,A,true
2025_P2_Q5.1.2,1,sin50 expression,1,"sin50=k/sqrt(k^2+1)","sin50=k/sqrt(k^2+1)",trig.identity,trig,A,false
2025_P2_Q5.1.2,2,Double angle,1,"-2(1-2sin^2 25)","-2(1-2sin^2 25)|−2(1−2sin²25)",trig.identity,trig,A,false
2025_P2_Q5.1.2,3,Substitute,1,"=sin50/-2cos50","sin50/-2cos50",trig.identity,trig,A,false
2025_P2_Q5.1.2,4,Quotient identity,1,"=-1/2 tan50","-1/2 tan50",trig.identity,trig,A,false
2025_P2_Q5.1.2,5,Answer,1,"=-k/2","-k/2|-1/2 k",trig.identity,trig,A,true
2025_P2_Q5.1.3,1,sin10 = sin(50-40),1,"sin10=sin(50-40)","sin10=sin(50-40)",trig.identity,trig,A,false
2025_P2_Q5.1.3,2,Expansion,1,"=sin50cos40-cos50sin40","sin50cos40-cos50sin40",trig.identity,trig,A,false
2025_P2_Q5.1.3,3,Substitute,1,"=(k)(k)-(1)(1)","(k)(k)-(1)(1)",trig.identity,trig,A,false
2025_P2_Q5.1.3,4,Answer,1,"=(k^2-1)/(k^2+1)","(k^2-1)/(k^2+1)",trig.identity,trig,A,true
2025_P2_Q5.2.1,1,sin(540+x),1,"sin(540+x)=-sinx","sin(540+x)=-sinx",trig.reduction,trig,A,false
2025_P2_Q5.2.1,2,cos(90+x),1,"cos(90+x)=-sinx","cos(90+x)=-sinx",trig.reduction,trig,A,false
2025_P2_Q5.2.1,3,sin(-x),1,"sin(-x)=-sinx","sin(-x)=-sinx",trig.reduction,trig,A,false
2025_P2_Q5.2.1,4,Answer,1,"=-sinx","-sinx",trig.reduction,trig,A,true
2025_P2_Q5.2.2,1,State interval,2,"x in (180;360)","x in (180;360)|180<x<360",trig.reduction,trig,A,true
2025_P2_Q7.1,1,State period,1,180,180,trig.graph,graph,A,true
2025_P2_Q7.3,1,Substitute x+45,1,"h(x)=cos2(x+45)","h(x)=cos2(x+45)",trig.graph,graph,A,false
2025_P2_Q7.3,2,Simplify,1,"=-sin2x","-sin2x",trig.graph,graph,A,true
2025_P2_Q7.4,1,Range,1,"y in [-1;1]","y in [-1;1]|-1<=y<=1",trig.graph,graph,A,true
2025_P2_Q7.5,1,tan2x=1,1,"tan2x=1","tan2x=1",trig.equation,trig,A,false
2025_P2_Q7.5,2,x=22.5,1,"x=22.5","x=22.5|x=22.5°",trig.equation,trig,A,false
2025_P2_Q7.5,3,First interval,1,"0<=x<=22.5","0<=x<=22.5|0°≤x≤22.5°",trig.equation,trig,A,false
2025_P2_Q7.5,4,Second interval,1,"112.5<=x<135","112.5<=x<135|112.5°≤x<135°",trig.equation,trig,A,true
2025_P2_Q8.1,1,BC=2AB,1,"BC=2AB","BC=2AB",trig.3d,trig3d,A,false
2025_P2_Q8.1,2,Substitute into area,1,"648=2AB*AB","648=2AB*AB|648=2AB^2",trig.3d,trig3d,A,false
2025_P2_Q8.1,3,AB value,1,AB=18,"AB=18",trig.3d,trig3d,A,true
2025_P2_Q8.2,1,Substitute into Pythagoras,1,"AC^2=18^2+36^2","AC^2=18^2+36^2",trig.3d,trig3d,A,false
2025_P2_Q8.2,2,Answer,1,"AC=18sqrt(5)","AC=18sqrt(5)|40.25",trig.3d,trig3d,A,true
2025_P2_Q8.3,1,Sine rule,1,"KC/sin104=36/sin52.6","KC/sin104=36/sin52.6",trig.3d,trig3d,A,false
2025_P2_Q8.3,2,Answer KC,1,KC=43.97,"KC=43.97|43.97",trig.3d,trig3d,A,true
2025_P2_Q8.4,1,AK length,1,"AK=18sqrt(2)","AK=18sqrt(2)|25.46",trig.3d,trig3d,A,false
2025_P2_Q8.4,2,Cosine rule,1,"43.97^2=(18sqrt(2))^2+(18sqrt(5))^2-2(18sqrt(2))(18sqrt(5))cosKAC","43.97^2=(18sqrt(2))^2+(18sqrt(5))^2-2(18sqrt(2))(18sqrt(5))cosKAC",trig.3d,trig3d,A,false
2025_P2_Q8.4,3,Simplify,1,"cos KAC=0.16","cos KAC=0.16",trig.3d,trig3d,A,false
2025_P2_Q8.4,4,Answer,1,"KAC=80.60","KAC=80.60|80.60",trig.3d,trig3d,A,true
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2025_P2.csv"
csv_path.write_text(csv_content, encoding="utf-8")
lines = csv_path.read_text(encoding="utf-8").splitlines()
print(f"2025 P2 CSV: wrote {len(lines)} lines")
print("OK")

2025 P2 CSV: wrote 90 lines
OK


In [82]:
import csv
csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2025_P2.csv"
with open(csv_path, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    ncols = len(next(reader))
    bad = 0
    for lineno, row in enumerate(reader, start=2):
        if len(row) != ncols:
            bad += 1
            print(f"bad cols line {lineno}: {len(row)} vs {ncols}")
            print(f"  {row}")
        else:
            try:
                int(row[3])
            except ValueError:
                bad += 1
                print(f"bad marks line {lineno}: {row[3]!r}")
print(f"Bad rows: {bad}")
print("OK" if bad == 0 else "FIX NEEDED")

Bad rows: 0
OK


In [83]:
import re
test_path = ROOT / "tests" / "test_step_grader.py"
src = test_path.read_text(encoding="utf-8")

new_cases = '''    # ============ 2025 P2 ============
    ("25P2 Q1.1", "2025_P2_Q1.1",
     ["a=331397.20", "b=-22988.32", "y_hat=331397.20-22988.32x"], 3, [], "", False),
    ("25P2 Q1.2", "2025_P2_Q1.2",
     ["y_hat=331397.20-22988.32(5)", "y_hat=216455.60"], 2, [], "", False),
    ("25P2 Q1.3", "2025_P2_Q1.3",
     ["strong correlation", "valid"], 2, [], "", False),
    ("25P2 Q1.4", "2025_P2_Q1.4", ["22988.32"], 1, [], "", False),
    ("25P2 Q2.1.1", "2025_P2_Q2.1.1", ["70"], 1, [], "", False),
    ("25P2 Q2.1.2", "2025_P2_Q2.1.2", ["67-40", "27"], 2, [], "", False),
    ("25P2 Q2.1.4", "2025_P2_Q2.1.4", ["skewed right"], 1, [], "", False),
    ("25P2 Q2.2", "2025_P2_Q2.2",
     ["(11+14+19+20+8+10+2+14+x)/9", "x=10",
      "sigma=5.23", "(6.77;17.23)", "3 players"], 5, [], "", False),
    ("25P2 Q3.1", "2025_P2_Q3.1",
     ["QR=sqrt((-4-12)^2+(-6-2)^2)", "QR=8sqrt(5)"], 2, [], "", False),
    ("25P2 Q3.2", "2025_P2_Q3.2",
     ["m_QR=(-6-2)/(-4-12)", "m_QR=1/2"], 2, [], "", False),
    ("25P2 Q3.3", "2025_P2_Q3.3",
     ["tan theta=1/2", "theta=26.57"], 2, [], "", False),
    ("25P2 Q3.4", "2025_P2_Q3.4",
     ["-6=1/2(-4)+c", "y=1/2x-4"], 2, [], "", False),
    ("25P2 Q3.5", "2025_P2_Q3.5", ["x_s=15", "y_s=16"], 2, [], "", False),
    ("25P2 Q3.6", "2025_P2_Q3.6",
     ["m_PT=-2", "y=-2x+6", "x=4", "T(4;-2)"], 4, [], "", False),
    ("25P2 Q3.7", "2025_P2_Q3.7",
     ["PT=5sqrt(5)", "Area=200"], 2, [], "", False),
    ("25P2 Q4.1", "2025_P2_Q4.1", ["p=-6"], 1, [], "", False),
    ("25P2 Q4.2", "2025_P2_Q4.2",
     ["AM=q+1; MD=q", "(q+1)^2=(q-1)^2+q^2", "q^2-4q=0", "q=4"],
     4, [], "", False),
    ("25P2 Q4.3", "2025_P2_Q4.3", ["(x+6)^2+(y-4)^2=25"], 1, [], "", False),
    ("25P2 Q4.4", "2025_P2_Q4.4", ["3"], 1, [], "", False),
    ("25P2 Q4.5", "2025_P2_Q4.5",
     ["(x+6)^2+(0-4)^2=25", "A(-9;0)", "B(-3;0)"], 3, [], "", False),
    ("25P2 Q4.6", "2025_P2_Q4.6",
     ["m_MB=-4/3", "m_BC=3/4", "0=3/4(-3)+c", "y=3/4x+9/4"], 4, [], "", False),
    ("25P2 Q4.7", "2025_P2_Q4.7", ["x_C=-6", "y_C=-9/4"], 2, [], "", False),
    ("25P2 Q4.8", "2025_P2_Q4.8",
     ["tan alpha=3/4", "alpha=36.87", "tan beta=-3/4", "beta=143.13", "ACB=106.26"],
     5, [], "", False),
    ("25P2 Q5.1.1", "2025_P2_Q5.1.1",
     ["sqrt(k^2+1)", "cos40=k/sqrt(k^2+1)"], 2, [], "", False),
    ("25P2 Q5.1.2", "2025_P2_Q5.1.2",
     ["sin50=k/sqrt(k^2+1)", "-2(1-2sin^2 25)", "sin50/-2cos50",
      "-1/2 tan50", "-k/2"], 5, [], "", False),
    ("25P2 Q5.1.3", "2025_P2_Q5.1.3",
     ["sin10=sin(50-40)", "sin50cos40-cos50sin40", "(k)(k)-(1)(1)",
      "(k^2-1)/(k^2+1)"], 4, [], "", False),
    ("25P2 Q5.2.1", "2025_P2_Q5.2.1",
     ["sin(540+x)=-sinx", "cos(90+x)=-sinx", "sin(-x)=-sinx", "-sinx"],
     4, [], "", False),
    ("25P2 Q5.2.2", "2025_P2_Q5.2.2", ["x in (180;360)"], 2, [], "", False),
    ("25P2 Q7.1", "2025_P2_Q7.1", ["180"], 1, [], "", False),
    ("25P2 Q7.3", "2025_P2_Q7.3",
     ["h(x)=cos2(x+45)", "-sin2x"], 2, [], "", False),
    ("25P2 Q7.4", "2025_P2_Q7.4", ["y in [-1;1]"], 1, [], "", False),
    ("25P2 Q7.5", "2025_P2_Q7.5",
     ["tan2x=1", "x=22.5", "0<=x<=22.5", "112.5<=x<135"], 4, [], "", False),
    ("25P2 Q8.1", "2025_P2_Q8.1",
     ["BC=2AB", "648=2AB*AB", "AB=18"], 3, [], "", False),
    ("25P2 Q8.2", "2025_P2_Q8.2",
     ["AC^2=18^2+36^2", "AC=18sqrt(5)"], 2, [], "", False),
    ("25P2 Q8.3", "2025_P2_Q8.3",
     ["KC/sin104=36/sin52.6", "KC=43.97"], 2, [], "", False),
    ("25P2 Q8.4", "2025_P2_Q8.4",
     ["AK=18sqrt(2)",
      "43.97^2=(18sqrt(2))^2+(18sqrt(5))^2-2(18sqrt(2))(18sqrt(5))cosKAC",
      "cos KAC=0.16", "KAC=80.60"], 4, [], "", False),
'''

pattern = re.compile(r"\n\]\s*\n\s*\n\s*\ndef run\(\)")
match = pattern.search(src)
assert match, "Could not find end of CASES block"

close_bracket_pos = src.index("]", match.start() + 1)
new_src = src[:close_bracket_pos] + new_cases + src[close_bracket_pos:]
test_path.write_text(new_src, encoding="utf-8")
print(f"test file: {len(new_src.splitlines())} lines")
print("OK")

test file: 423 lines
OK


In [84]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] 23P1 Q1.1.1
  [PASS ] 23P1 Q1.1.2
  [PASS ] 23P1 Q1.2 mA
  [PASS ] 23P1 Q3.2
  [PASS ] 23P1 Q5.6 mA
  [PASS ] 23P1 Q7.1
  [PASS ] 23P1 Q8.4
  [PASS ] 23P2 Q1.1
  [PASS ] 23P2 Q3.2
  [PASS ] 23P2 Q5.1.1
  [PASS ] 24P1 Q1.1.1
  [PASS ] 24P1 Q1.1.2
  [PASS ] 24P1 Q1.1.3
  [PASS ] 24P1 Q1.1.4
  [PASS ] 24P1 Q1.1.5
  [PASS ] 24P1 Q1.2 mA
  [PASS ] 24P1 Q1.2 mB
  [PASS ] 24P1 Q1.3
  [PASS ] 24P1 Q2.1.1
  [PASS ] 24P1 Q2.1.2
  [PASS ] 24P1 Q2.2.1
  [PASS ] 24P1 Q2.2.2
  [PASS ] 24P1 Q3.1
  [PASS ] 24P1 Q3.2
  [PASS ] 24P1 Q3.3
  [PASS ] 24P1 Q4.1
  [PASS ] 24P1 Q4.2
  [PASS ] 24P1 Q4.4
  [PASS ] 24P1 Q5.1
  [PASS ] 24P1 Q5.2
  [PASS ] 24P1 Q5.3
  [PASS ] 24P1 Q5.4
  [PASS ] 24P1 Q5.5
  [PASS ] 24P1 Q6.1
  [PASS ] 24P1 Q6.2
  [PASS ] 24P1 Q6.3
  [PASS ] 24P1 Q6.4
  [PASS ] 24P1 Q7.1
  [PASS ] 24P1 Q7.2
  [PASS ] 24P1 Q7.3.1
  [PASS ] 24P1 Q7.3.2
  [PASS ] 24P1 Q8.1.1
  [PASS ] 24P1 Q8.1.2
  [PASS ] 24P1 Q8.2
  [PASS ] 24P1 Q8.3.1
  [PASS ] 24P1 Q8.3.2
  [PASS ] 24P1 Q8.3.3
  [PASS ] 

In [85]:
sg_path = ROOT / "src" / "tutor" / "step_grader.py"
src = sg_path.read_text(encoding="utf-8")

# 1. Loader: read match_mode (default "exact")
old = '''                "method_group": mg,
                "is_final": str(row.get("is_final", "false")).strip().lower() == "true",
            })'''
new = '''                "method_group": mg,
                "is_final": str(row.get("is_final", "false")).strip().lower() == "true",
                "match_mode": (row.get("match_mode") or "exact").strip().lower() or "exact",
            })'''
assert old in src, "loader patch anchor not found"
src = src.replace(old, new)

# 2. _line_matches: accept mode parameter
old = '''def _line_matches(line: str, forms: list[str], is_final: bool = False) -> bool:
    lt = _tokens(line)
    if not lt:
        return False
    lt_set = set(lt)
    for form in forms:
        ft = _tokens(form)
        if not ft:
            continue
        if is_final:
            if set(ft).issubset(lt_set):
                return True
        else:
            if ft == lt:
                return True
    return False'''
new = '''def _line_matches(line: str, forms: list[str], is_final: bool = False, mode: str = "exact") -> bool:
    if mode == "keyword":
        nline = _norm(line)
        if not nline:
            return False
        for form in forms:
            nf = _norm(form)
            if nf and nf in nline:
                return True
        return False
    lt = _tokens(line)
    if not lt:
        return False
    lt_set = set(lt)
    for form in forms:
        ft = _tokens(form)
        if not ft:
            continue
        if is_final:
            if set(ft).issubset(lt_set):
                return True
        else:
            if ft == lt:
                return True
    return False'''
assert old in src, "_line_matches anchor not found"
src = src.replace(old, new)

# 3. _grade_group: read mode from step, pass to matcher
old = '''        is_final = step.get("is_final", False)
        for i, line in enumerate(learner_lines):
            if i in used and not is_final:
                continue
            if _line_matches(line, step["forms"], is_final=is_final):'''
new = '''        is_final = step.get("is_final", False)
        mode = step.get("match_mode", "exact")
        for i, line in enumerate(learner_lines):
            if i in used and not is_final:
                continue
            if _line_matches(line, step["forms"], is_final=is_final, mode=mode):'''
assert old in src, "_grade_group anchor not found"
src = src.replace(old, new)

sg_path.write_text(src, encoding="utf-8")
print(f"step_grader.py: {len(src.splitlines())} lines")
print("OK — keyword mode added")

step_grader.py: 187 lines
OK — keyword mode added


In [86]:
csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final,match_mode
2023_P2_Q5.2.1,1,Factor cos squared from numerator,1,"cos^2x(cos^2x+sin^2x)","cos^2x(cos^2x+sin^2x)|cos²x(cos²x+sin²x)",trig.identity,proof,A,false,keyword
2023_P2_Q5.2.1,2,Apply sin²+cos²=1,1,"sin^2x+cos^2x=1","sin^2x+cos^2x=1|cos^2x+sin^2x=1",trig.identity,proof,A,false,keyword
2023_P2_Q5.2.1,3,Apply cos²=1-sin²,1,"1-sin^2x","1-sin^2x|cos^2x=1-sin^2x",trig.identity,proof,A,false,keyword
2023_P2_Q5.2.1,4,Factorise difference of squares,1,"(1-sinx)(1+sinx)","(1-sinx)(1+sinx)|(1+sinx)(1-sinx)",trig.identity,proof,A,true,keyword
2023_P2_Q5.3.1,1,Rewrite sin as cos,1,"sin(A-B)=cos[90-(A-B)]","sin(A-B)=cos[90|cos[90-(A-B)]",trig.identity,proof,A,false,keyword
2023_P2_Q5.3.1,2,Apply compound formula,1,"cos[(90-A)-B]","cos[(90-A)-B]|(90-A)-B|cos(90-A)cos(B)",trig.identity,proof,A,false,keyword
2023_P2_Q5.3.1,3,Co-function and answer,1,"=sinA cosB - cosA sinB","sinA cosB - cosA sinB|sinAcosB-cosAsinB",trig.identity,proof,A,true,keyword
2023_P2_Q5.4,1,Rewrite 3x,1,"3x=2x+x","3x=2x+x|3x = (2x + x)",trig.identity,proof,A,false,keyword
2023_P2_Q5.4,2,Expand sin(2x+x),1,"sin2x cosx + cos2x sinx","sin2x cosx + cos2x sinx|sin2xcosx+cos2xsinx",trig.identity,proof,A,false,keyword
2023_P2_Q5.4,3,Expand sin(2x-x),1,"sin2x cosx - cos2x sinx","sin2x cosx - cos2x sinx|sin2xcosx-cos2xsinx",trig.identity,proof,A,false,keyword
2023_P2_Q5.4,4,Combine numerator,1,"=2 sin2x cosx","2 sin2x cosx|2sin2xcosx",trig.identity,proof,A,false,keyword
2023_P2_Q5.4,5,Double angle of cos2x,1,"cos2x+1=2cos^2x","cos2x+1=2cos^2x|cos2x+1=2cos²x",trig.identity,proof,A,false,keyword
2023_P2_Q5.4,6,Answer,1,"=2 sinx","=2 sinx|2 sinx|2sinx",trig.identity,proof,A,true,keyword
2023_P2_Q8.1,1,Construction,1,"Draw KO produced","draw ko|ko produced|construction",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.1,2,Ext angle of triangle,1,"O1=K1+T [ext angle]","ext angle|exterior angle|ext ∠",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.1,3,Opp equal sides,1,"K1=T [opp equal sides]","opp equal|equal sides|opp ∠s|opposite sides",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.1,4,O1=2K1 and O2=2K2,1,"O1=2K1 and O2=2K2","2k1|2k2|2(k1+k2)",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.1,5,Conclusion,1,"TOP=2TKP","top=2tkp|top = 2tkp|2tkp",euclidean.circle,euclid,A,true,keyword
2023_P2_Q8.2,1,Angles in same segment,1,"D1=AFG=x [tan-chord]","tan-chord|tan chord|D1=AFG",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.2,2,DC parallel FG,1,"DC parallel FG","dc parallel fg|dc || fg|parallel",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.2,3,Proportionality,1,"AG/AC = AF/AD","ag/ac = af/ad|proportional|prop theorem",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.2,4,Answer AG.AD=AC.AF,1,"AG.AD=AC.AF","ag.ad = ac.af|ag×ad|ag ad = ac af",euclidean.circle,euclid,A,true,keyword
2023_P2_Q8.3.1,1,Angle in semi-circle,1,"OMB=90 [angle in semi-circle]","angle in semi|90° in semi|semi-circle",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.3.1,2,Statement and reason,1,"OMB=90° [S/R]","omb=90|omb = 90",euclidean.circle,euclid,A,true,keyword
2023_P2_Q8.3.2,1,AB length,1,"AB=10sqrt(3)","10sqrt(3)|10√3|ab=10√3",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.3.2,2,MB length,1,"MB=5sqrt(3)","5sqrt(3)|5√3|mb=5√3",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.3.2,3,Pythagoras,1,"OB^2 = OM^2 + MB^2","pythagoras|om^2+mb^2|ob²=om²+mb²",euclidean.circle,euclid,A,false,keyword
2023_P2_Q8.3.2,4,Answer OB,1,"OB=10","ob=10|ob = 10 units",euclidean.circle,euclid,A,true,keyword
2023_P2_Q9.1,1,Proportionality,1,"FB/EB = DA/EA","fb/eb=da/ea|prop theorem|fb/eb = da/ea",euclidean.similarity,euclid,A,false,keyword
2023_P2_Q9.1,2,Substitute,1,"FB/4p = 21/7p","fb/4p=21/7p",euclidean.similarity,euclid,A,false,keyword
2023_P2_Q9.1,3,Answer,1,"FB=12","fb=12|fb=12 units",euclidean.similarity,euclid,A,true,keyword
2023_P2_Q9.2,1,Common angle E,1,"E is common","common angle|e is common|shared angle",euclidean.similarity,euclid,A,false,keyword
2023_P2_Q9.2,2,Corresponding angles,1,"EDF=A and EFD=B [corresp]","corresp|corresponding|edf=a",euclidean.similarity,euclid,A,false,keyword
2023_P2_Q9.2,3,Similar triangles,1,"EDF || EAB","edf || eab|edf ||| eab|similar",euclidean.similarity,euclid,A,true,keyword
2023_P2_Q9.3,1,Proportionality,1,"DF/AB = ED/EA","df/ab=ed/ea|prop theorem",euclidean.similarity,euclid,A,false,keyword
2023_P2_Q9.3,2,DF value,1,"DF=6","df=6",euclidean.similarity,euclid,A,false,keyword
2023_P2_Q9.3,3,FC value,1,"FC=8","fc=8",euclidean.similarity,euclid,A,true,keyword
2023_P2_Q10.1,1,Ext angle of cyclic quad,1,"S3=PQR [ext angle]","ext angle|exterior angle|cyclic quad",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.1,2,Opp equal sides,1,"R3=PQR [opp equal sides]","opp equal|opposite sides|equal sides",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.1,3,Angles in same segment,1,"S4=R3 [same seg]","same seg|same segment|s3=s4",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.1,4,S3=S4,1,"S3=S4","s3=s4",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.1,5,Conclusion,1,"Proven","proven|therefore",euclidean.circle,euclid,A,true,keyword
2023_P2_Q10.2,1,tan-chord,1,"R1+R2=PQR [tan-chord]","tan-chord|tan chord",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.2,2,S4=PQR,1,"S4=PQR","s4=pqr",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.2,3,S4=R1+R2,1,"S4=R1+R2","s4=r1+r2",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.2,4,Converse cyclic quad,1,"SMRC is cyclic quad","cyclic quad|smrc",euclidean.circle,euclid,A,true,keyword
2023_P2_Q10.3,1,Ext angle of triangle,1,"S3=R2+P2 [ext angle]","ext angle|exterior angle|s3=r2+p2",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.3,2,Ext angle again,1,"S4=P1+A2 [ext angle]","ext angle|exterior angle|s4=p1+a2",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.3,3,Combine,1,"R2+P2=A2+P1","r2+p2=a2+p1",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.3,4,Equal angles,1,"P1=R2 and P2=A2","p1=r2 and p2=a2|p1=r2|p2=a2",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.3,5,Converse tan-chord,1,"converse tan-chord","converse tan-chord|converse tangent",euclidean.circle,euclid,A,false,keyword
2023_P2_Q10.3,6,Conclusion RP tangent,1,"RP is tangent","rp is tangent|rp tangent",euclidean.circle,euclid,A,true,keyword
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2023_P2_proofs.csv"
csv_path.write_text(csv_content, encoding="utf-8")
print(f"2023 P2 proofs CSV: {len(csv_content.splitlines())} lines")
print("OK")

2023 P2 proofs CSV: 53 lines
OK


In [87]:
csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final,match_mode
2024_P2_Q6.1.1,1,Rewrite as subtraction,1,"cos(x+y)=cos(x-(-y))","cos(x+y)=cos(x-(-y))|cos(x+y)=cos(x-(-y))",trig.identity,proof,A,false,keyword
2024_P2_Q6.1.1,2,Expand,1,"=cosx cos(-y)+sinx sin(-y)","cosx cos(-y)+sinx sin(-y)|cosxcos(-y)+sinxsin(-y)",trig.identity,proof,A,true,keyword
2024_P2_Q6.1.2,1,cos(90-x)=sinx,1,"cos(90-x)=sinx","cos(90-x)=sinx|sinx",trig.identity,proof,A,false,keyword
2024_P2_Q6.1.2,2,sin(-y)=-siny,1,"sin(-y)=-siny","sin(-y)=-siny",trig.identity,proof,A,false,keyword
2024_P2_Q6.1.2,3,cos(180+x)=-cosx,1,"cos(180+x)=-cosx","cos(180+x)=-cosx",trig.identity,proof,A,false,keyword
2024_P2_Q6.1.2,4,cos(360+y)=cosy,1,"cos(360+y)=cosy","cos(360+y)=cosy|cos(360+y)",trig.identity,proof,A,false,keyword
2024_P2_Q6.1.2,5,sin(360-x)=-sinx,1,"sin(360-x)=-sinx","sin(360-x)=-sinx",trig.identity,proof,A,false,keyword
2024_P2_Q6.1.2,6,Compound formula,1,"=tan(x+y)","tan(x+y)|tan(x+y)=RHS",trig.identity,proof,A,true,keyword
2024_P2_Q6.2,1,Square both sides,1,"6sin^2x-11cos(90+x)+7=4","6sin^2x-11cos(90+x)+7=4|6sin^2x",trig.identity,proof,A,false,keyword
2024_P2_Q6.2,2,cos(90+x)=-sinx,1,"cos(90+x)=-sinx","cos(90+x)=-sinx|-sinx",trig.identity,proof,A,false,keyword
2024_P2_Q6.2,3,Factor,1,"(3sinx+1)(2sinx+3)=0","(3sinx+1)(2sinx+3)=0|factor",trig.identity,proof,A,false,keyword
2024_P2_Q6.2,4,Both equations,1,"sinx=-1/3 or sinx=-3/2","sinx=-1/3|sinx=-3/2",trig.identity,proof,A,false,keyword
2024_P2_Q6.2,5,Answers,1,"x=199.47 or x=340.53","199.47|340.53",trig.identity,proof,A,true,keyword
2024_P2_Q6.3.1,1,Factor,1,"=4(1-2sin^2x)/3","4(1-2sin^2x)/3|factor",trig.identity,proof,A,false,keyword
2024_P2_Q6.3.1,2,Rewrite as 4cos2x/3,1,"=4cos2x/3","4cos2x/3|cos2x",trig.identity,proof,A,false,keyword
2024_P2_Q6.3.1,3,Maximum value,1,"max=4/3","4/3|max = 4/3",trig.identity,proof,A,true,keyword
2024_P2_Q6.3.2,1,Answer x,1,"x=180","x=180|x=180°",trig.identity,proof,A,true,keyword
2024_P2_Q9.1,1,OMB=90,1,"OMB=90 [angle in semi-circle]","angle in semi-circle|omb=90",euclidean.circle,euclid,A,false,keyword
2024_P2_Q9.1,2,Statement and reason,1,"S/R","S/R|omb=90|angle in semi",euclidean.circle,euclid,A,true,keyword
2024_P2_Q9.2,1,B=80 [half of A1],1,"B=80","b=80|b_hat=80",euclidean.circle,euclid,A,false,keyword
2024_P2_Q9.2,2,Opp angles of cyclic quad,1,"D=100 [opp angles]","opp angles|cyclic quad|d=100",euclidean.circle,euclid,A,false,keyword
2024_P2_Q9.2,3,Sum of angles,1,"C2=40 [sum of angles]","sum of angles|sum of ∠s|c2=40",euclidean.circle,euclid,A,false,keyword
2024_P2_Q9.2,4,Equal angles imply equal sides,1,"AD=DC","ad=dc|equal sides",euclidean.circle,euclid,A,true,keyword
2024_P2_Q10.1,1,Construction,1,"Join SB and TA","join sb|join ta|construction",euclidean.similarity,euclid,A,false,keyword
2024_P2_Q10.1,2,Area of RAB,1,"area RAB = 1/2 RA x h1","area rab|1/2 ra",euclidean.similarity,euclid,A,false,keyword
2024_P2_Q10.1,3,Area of ASB,1,"area ASB = 1/2 AS x h1","area asb|1/2 as",euclidean.similarity,euclid,A,false,keyword
2024_P2_Q10.1,4,Areas equal,1,"area ASB = area ABT","area asb = area abt|equal areas",euclidean.similarity,euclid,A,false,keyword
2024_P2_Q10.1,5,Conclusion,1,"RA/AS = RB/BT","ra/as=rb/bt",euclidean.similarity,euclid,A,true,keyword
2024_P2_Q10.2,1,Perpendicular from centre,1,"PR=RS","pr=rs|perpendicular from centre|midpoint chord",euclidean.circle,euclid,A,false,keyword
2024_P2_Q10.2,2,Radii equal,1,"PO=OW","po=ow|radii",euclidean.circle,euclid,A,false,keyword
2024_P2_Q10.2,3,Midpoint theorem,1,"OR=1/2 WS","or=1/2ws|or:ws=1:2|midpoint theorem",euclidean.circle,euclid,A,false,keyword
2024_P2_Q10.2,4,Ratio OR:WS,1,"OR:WS=1:2","or:ws=1:2",euclidean.circle,euclid,A,true,keyword
2024_P2_Q11.1,1,Tan-chord D1,1,"D1=EAg=x [tan-chord]","tan-chord|tan chord|d1=eag",euclidean.circle,euclid,A,false,keyword
2024_P2_Q11.1,2,Tan-chord C1,1,"C1=D1=x [tan-chord]","tan-chord|tan chord|c1=d1",euclidean.circle,euclid,A,false,keyword
2024_P2_Q11.1,3,Vert opp angles,1,"C4=C1=x","vert opp|vertical|c4=c1",euclidean.circle,euclid,A,false,keyword
2024_P2_Q11.1,4,Tan-chord AFG,1,"AFG=EAg=x","tan-chord|tan chord|afg=eag",euclidean.circle,euclid,A,true,keyword
2024_P2_Q11.2,1,DC parallel FG,1,"D1=AFG=x [corresp]","corresp|dc parallel fg|dc || fg",euclidean.circle,euclid,A,false,keyword
2024_P2_Q11.2,2,Proportionality,1,"AG/AC = AF/AD","ag/ac = af/ad|prop theorem",euclidean.circle,euclid,A,false,keyword
2024_P2_Q11.2,3,Answer AG.AD=AC.AF,1,"AG.AD=AC.AF","ag.ad=ac.af",euclidean.circle,euclid,A,true,keyword
2024_P2_Q11.3,1,Angles in same seg,1,"G=B2 [same seg]","same seg|same segment|g=b2",euclidean.circle,euclid,A,false,keyword
2024_P2_Q11.3,2,Substitute AFG=C1,1,"AFG=C1=x","afg=c1",euclidean.circle,euclid,A,false,keyword
2024_P2_Q11.3,3,Sum of angles,1,"A3=A2 [sum]","sum of angles|a3=a2",euclidean.circle,euclid,A,false,keyword
2024_P2_Q11.3,4,Similar triangles,1,"AGF || ABC","agf || abc|agf ||| abc",euclidean.circle,euclid,A,true,keyword
2024_P2_Q11.4,1,Ratio from similarity,1,"GF/BC = AF/AC","gf/bc=af/ac",euclidean.similarity,euclid,A,false,keyword
2024_P2_Q11.4,2,GF expression,1,"GF=BC.AF/AC","gf=bc.af/ac",euclidean.similarity,euclid,A,false,keyword
2024_P2_Q11.4,3,Second similarity,1,"ACD || FGC","acd || fgc|acd ||| fgc",euclidean.similarity,euclid,A,false,keyword
2024_P2_Q11.4,4,Ratio,1,"AC/GF = AD/FC","ac/gf=ad/fc",euclidean.similarity,euclid,A,false,keyword
2024_P2_Q11.4,5,Substitute,1,"GF^2 = AF.FC.BC/AD","gf^2 = af.fc.bc/ad|gf²=af.fc.bc/ad",euclidean.similarity,euclid,A,false,keyword
2024_P2_Q11.4,6,Answer,1,"Answer","gf^2|gf²",euclidean.similarity,euclid,A,true,keyword
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2024_P2_proofs.csv"
csv_path.write_text(csv_content, encoding="utf-8")
print(f"2024 P2 proofs CSV: {len(csv_content.splitlines())} lines")
print("OK")

2024 P2 proofs CSV: 50 lines
OK


In [88]:
csv_content = '''question_id,step_index,step_description,marks,memo_line,acceptable_forms,skill_id,method_tag,method_group,is_final,match_mode
2025_P2_Q6,1,Expand tan,1,"tan(180-x)=-tanx","tan(180-x)=-tanx|tan(180-x)=-tan",trig.identity,proof,A,false,keyword
2025_P2_Q6,2,cos^2=1-sin^2,1,"1-cos^2x=sin^2x","1-cos^2x=sin^2x|1-cos²x=sin²x",trig.identity,proof,A,false,keyword
2025_P2_Q6,3,Quotient identity,1,"sinx/cosx = tanx","sinx/cosx=tanx|quotient identity",trig.identity,proof,A,false,keyword
2025_P2_Q6,4,Simplify,1,"= (sin^3x - cos^3x)/(-cosx)","(sin^3x - cos^3x)/(-cosx)|sin^3x-cos^3x",trig.identity,proof,A,false,keyword
2025_P2_Q6,5,Difference of cubes,1,"(sinx-cosx)(sin^2x+sinxcosx+cos^2x)","difference of cubes|sin^2x+sinxcosx+cos^2x",trig.identity,proof,A,false,keyword
2025_P2_Q6,6,Answer,1,"=RHS","rhs|1+sinxcosx",trig.identity,proof,A,true,keyword
2025_P2_Q9.1,1,Construction,1,"Draw KO produced","draw ko|ko produced|construction",euclidean.circle,euclid,A,false,keyword
2025_P2_Q9.1,2,Ext angle of triangle,1,"O1=K1+T [ext angle]","ext angle|exterior angle",euclidean.circle,euclid,A,false,keyword
2025_P2_Q9.1,3,Opp equal sides,1,"K1=T [opp equal sides]","opp equal|equal sides|opposite sides",euclidean.circle,euclid,A,false,keyword
2025_P2_Q9.1,4,O1=2K1,1,"O1=2K1","o1=2k1|2k1",euclidean.circle,euclid,A,false,keyword
2025_P2_Q9.1,5,Conclusion,1,"TOP=2TKP","top=2tkp|2tkp",euclidean.circle,euclid,A,true,keyword
2025_P2_Q9.2.1,1,O2=64 [angle at centre],1,"O2=64 [angle at centre]","angle at centre|o2=64|64",euclidean.circle,euclid,A,false,keyword
2025_P2_Q9.2.1,2,Statement and reason,1,"S/R","s/r|o2=64",euclidean.circle,euclid,A,true,keyword
2025_P2_Q9.2.2,1,Tangent perpendicular to diameter,1,"PMF=90 [tan perp diameter]","tan perp|tangent perpendicular|pmf=90",euclidean.circle,euclid,A,false,keyword
2025_P2_Q9.2.2,2,Sum of angles,1,"M=26 [sum of angles]","sum of angles|m=26",euclidean.circle,euclid,A,false,keyword
2025_P2_Q9.2.2,3,Answer,1,"26","26|m=26",euclidean.circle,euclid,A,true,keyword
2025_P2_Q10.1,1,Sum of angles,1,"R=55 [sum of angles]","sum of angles|r=55|r=55°",euclidean.circle,euclid,A,false,keyword
2025_P2_Q10.1,2,Same segment,1,"QTS=55 [same seg]","same seg|same segment|qts=55",euclidean.circle,euclid,A,false,keyword
2025_P2_Q10.1,3,Answer,1,"QTS=55","qts=55|55",euclidean.circle,euclid,A,true,keyword
2025_P2_Q10.2,1,Opp angles of cyclic quad,1,"SPQ=125","opp angles|cyclic quad|spq=125",euclidean.circle,euclid,A,false,keyword
2025_P2_Q10.2,2,Given S1=R=55,1,"S1=R=55","s1=r=55|s1=r",euclidean.circle,euclid,A,false,keyword
2025_P2_Q10.2,3,Co-interior angles,1,"PQ parallel SR","co-int|co-interior|pq parallel sr",euclidean.circle,euclid,A,true,keyword
2025_P2_Q10.3,1,Co-interior Q1=90,1,"Q1=90","co-int|q1=90",euclidean.circle,euclid,A,false,keyword
2025_P2_Q10.3,2,Converse angle in semi,1,"PT is diameter","converse|pt is diameter|pt diameter",euclidean.circle,euclid,A,false,keyword
2025_P2_Q10.3,3,Answer,1,"PT is diameter","pt is diameter",euclidean.circle,euclid,A,true,keyword
2025_P2_Q11.1.1,1,FD/CF from proportional,1,"FD/CF = GA/CF","fd/cf = ga/cf|fd/cf=ga/cf",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.1.1,2,Answer,1,"=2/3","2/3|fd=2/3cf",euclidean.similarity,euclid,A,true,keyword
2025_P2_Q11.1.2,1,FD expression,1,"FD=2/3CF","fd=2/3cf",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.1.2,2,Substitute,1,"FD=2/3(2x)","fd=2/3(2x)|fd=4/3x",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.1.2,3,BA/ED ratio,1,"BA/ED","ba/ed|ba/ed ratio",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.1.2,4,Answer,1,"=4/3x","4/3x",euclidean.similarity,euclid,A,true,keyword
2025_P2_Q11.1.3,1,Area difference,1,"GFDA = CDA - CGF","gfda = cda - cgf|area difference",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.1.3,2,Area GCF,1,"1/2(GC)(FC)sinC","1/2(gc)(fc)sinc|area gcf",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.1.3,3,Area CDA - CGF,1,"1/2 AC.CD sinC - 1/2 GC.CF sinC","ac.cd sinc - gc.cf sinc",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.1.3,4,Answer,1,"=9/16","9/16",euclidean.similarity,euclid,A,true,keyword
2025_P2_Q11.2.1,1,Proportional,1,"QE/PR = QW/PR","qe/pr=qw/pr|prop theorem",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.2.1,2,Answer,1,"Proven","proven",euclidean.similarity,euclid,A,true,keyword
2025_P2_Q11.2.2,1,Similar triangles,1,"PQ/QR = QZ/QR","pq/qr=qz/qr",euclidean.similarity,euclid,A,true,keyword
2025_P2_Q11.2.3,1,Common angle,1,"Q2 common","common angle|q2 common",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.2.3,2,Tan-chord theorem,1,"S1=R2 [tan-chord]","tan-chord|tan chord|s1=r2",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.2.3,3,Third angle,1,"Z1=QSR","z1=qsr",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.2.3,4,Similar,1,"QSZ || QRS","qsz || qrs|qsz ||| qrs",euclidean.similarity,euclid,A,true,keyword
2025_P2_Q11.2.4,1,Ratio,1,"QS/QR = QZ/QS","qs/qr=qz/qs",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.2.4,2,Square,1,"QS^2=QZ.QR","qs^2=qz.qr|qs²=qz.qr",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.2.4,3,Answer PQ=QS,1,"PQ=QS","pq=qs",euclidean.similarity,euclid,A,true,keyword
2025_P2_Q11.2.5,1,First ratio,1,"PQ/RQ = PZ/PR","pq/rq=pz/pr",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.2.5,2,PR from 11.2.1,1,"PR = PW.QR/QE","pr=pw.qr/qe",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.2.5,3,Set equal,1,"PW.QR/QE = QR.PZ/PQ","pw.qr/qe=qr.pz/pq",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.2.5,4,PQ from 11.2.2,1,"PQ=sqrt(RQ.QZ)","pq=sqrt(rq.qz)",euclidean.similarity,euclid,A,false,keyword
2025_P2_Q11.2.5,5,Answer,1,"PW = QE.PZ/sqrt(RQ.QZ)","pw=qe.pz/sqrt(rq.qz)",euclidean.similarity,euclid,A,true,keyword
'''

csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2025_P2_proofs.csv"
csv_path.write_text(csv_content, encoding="utf-8")
print(f"2025 P2 proofs CSV: {len(csv_content.splitlines())} lines")
print("OK")

2025 P2 proofs CSV: 51 lines
OK


In [89]:
import re
test_path = ROOT / "tests" / "test_step_grader.py"
src = test_path.read_text(encoding="utf-8")

new_cases = '''    # ============ Proofs & EUCL ============
    # 2023 P2
    ("23P2 Q5.2.1", "2023_P2_Q5.2.1",
     ["cos^2x(cos^2x+sin^2x)", "sin^2x+cos^2x=1", "1-sin^2x", "(1-sinx)(1+sinx)"],
     4, [], "", False),
    ("23P2 Q5.3.1", "2023_P2_Q5.3.1",
     ["sin(A-B)=cos[90-(A-B)]", "cos[(90-A)-B]", "sinA cosB - cosA sinB"],
     3, [], "", False),
    ("23P2 Q5.4", "2023_P2_Q5.4",
     ["3x=2x+x", "sin2x cosx + cos2x sinx", "sin2x cosx - cos2x sinx",
      "2 sin2x cosx", "cos2x+1=2cos^2x", "=2 sinx"], 6, [], "", False),
    ("23P2 Q8.1", "2023_P2_Q8.1",
     ["Draw KO produced", "O1=K1+T (ext angle of triangle)",
      "K1=T (opp equal sides)", "O1=2K1 and O2=2K2", "TOP=2TKP"],
     5, [], "", False),
    ("23P2 Q8.2", "2023_P2_Q8.2",
     ["D1=AFG=x (tan-chord)", "DC parallel FG", "AG/AC = AF/AD",
      "AG.AD=AC.AF"], 4, [], "", False),
    ("23P2 Q8.3.1", "2023_P2_Q8.3.1",
     ["OMB=90 angle in semi-circle", "OMB=90 S/R"], 2, [], "", False),
    ("23P2 Q8.3.2", "2023_P2_Q8.3.2",
     ["AB=10sqrt(3)", "MB=5sqrt(3)", "OB^2 = OM^2 + MB^2", "OB=10"],
     4, [], "", False),
    ("23P2 Q9.1", "2023_P2_Q9.1",
     ["FB/EB = DA/EA prop theorem", "FB/4p = 21/7p", "FB=12"],
     3, [], "", False),
    ("23P2 Q9.2", "2023_P2_Q9.2",
     ["E is common", "EDF=A corresponding", "EDF || EAB"],
     3, [], "", False),
    ("23P2 Q9.3", "2023_P2_Q9.3",
     ["DF/AB = ED/EA", "DF=6", "FC=8"], 3, [], "", False),
    ("23P2 Q10.1", "2023_P2_Q10.1",
     ["S3=PQR ext angle", "R3=PQR opp equal sides",
      "S4=R3 same seg", "S3=S4", "Proven"], 5, [], "", False),
    ("23P2 Q10.2", "2023_P2_Q10.2",
     ["R1+R2=PQR tan-chord", "S4=PQR", "S4=R1+R2", "SMRC is cyclic quad"],
     4, [], "", False),
    ("23P2 Q10.3", "2023_P2_Q10.3",
     ["S3=R2+P2 ext angle", "S4=P1+A2 ext angle", "R2+P2=A2+P1",
      "P1=R2 and P2=A2", "converse tan-chord", "RP is tangent"],
     6, [], "", False),
    # 2024 P2
    ("24P2 Q6.1.1", "2024_P2_Q6.1.1",
     ["cos(x+y)=cos(x-(-y))", "cosx cos(-y)+sinx sin(-y)"], 2, [], "", False),
    ("24P2 Q6.1.2", "2024_P2_Q6.1.2",
     ["cos(90-x)=sinx", "sin(-y)=-siny", "cos(180+x)=-cosx",
      "cos(360+y)=cosy", "sin(360-x)=-sinx", "=tan(x+y)"],
     6, [], "", False),
    ("24P2 Q6.2", "2024_P2_Q6.2",
     ["6sin^2x-11cos(90+x)+7=4", "cos(90+x)=-sinx",
      "(3sinx+1)(2sinx+3)=0", "sinx=-1/3", "x=199.47 or x=340.53"],
     5, [], "", False),
    ("24P2 Q6.3.1", "2024_P2_Q6.3.1",
     ["4(1-2sin^2x)/3", "4cos2x/3", "max=4/3"], 3, [], "", False),
    ("24P2 Q6.3.2", "2024_P2_Q6.3.2", ["x=180"], 1, [], "", False),
    ("24P2 Q9.1", "2024_P2_Q9.1",
     ["OMB=90 angle in semi-circle", "OMB=90 S/R"], 2, [], "", False),
    ("24P2 Q9.2", "2024_P2_Q9.2",
     ["B=80", "D=100 opp angles", "C2=40 sum of angles", "AD=DC"],
     4, [], "", False),
    ("24P2 Q10.1", "2024_P2_Q10.1",
     ["Join SB and TA", "area RAB = 1/2 RA x h1", "area ASB = 1/2 AS x h1",
      "area ASB = area ABT", "RA/AS = RB/BT"], 5, [], "", False),
    ("24P2 Q10.2", "2024_P2_Q10.2",
     ["PR=RS perpendicular from centre", "PO=OW radii",
      "OR=1/2WS midpoint theorem", "OR:WS=1:2"], 4, [], "", False),
    ("24P2 Q11.1", "2024_P2_Q11.1",
     ["D1=EAg=x tan-chord", "C1=D1=x tan-chord",
      "C4=C1=x vert opp", "AFG=EAg=x tan-chord"], 4, [], "", False),
    ("24P2 Q11.2", "2024_P2_Q11.2",
     ["D1=AFG=x corresp", "AG/AC = AF/AD", "AG.AD=AC.AF"], 3, [], "", False),
    ("24P2 Q11.3", "2024_P2_Q11.3",
     ["G=B2 same seg", "AFG=C1=x", "A3=A2 sum", "AGF || ABC"],
     4, [], "", False),
    ("24P2 Q11.4", "2024_P2_Q11.4",
     ["GF/BC = AF/AC", "GF=BC.AF/AC", "ACD || FGC",
      "AC/GF = AD/FC", "GF^2 = AF.FC.BC/AD", "GF^2"],
     6, [], "", False),
    # 2025 P2
    ("25P2 Q6", "2025_P2_Q6",
     ["tan(180-x)=-tanx", "1-cos^2x=sin^2x", "sinx/cosx = tanx",
      "= (sin^3x - cos^3x)/(-cosx)",
      "(sinx-cosx)(sin^2x+sinxcosx+cos^2x)", "=RHS"],
     6, [], "", False),
    ("25P2 Q9.1", "2025_P2_Q9.1",
     ["Draw KO produced", "O1=K1+T ext angle",
      "K1=T opp equal sides", "O1=2K1", "TOP=2TKP"], 5, [], "", False),
    ("25P2 Q9.2.1", "2025_P2_Q9.2.1",
     ["O2=64 angle at centre", "S/R"], 2, [], "", False),
    ("25P2 Q9.2.2", "2025_P2_Q9.2.2",
     ["PMF=90 tan perp diameter", "M=26 sum of angles", "26"],
     3, [], "", False),
    ("25P2 Q10.1", "2025_P2_Q10.1",
     ["R=55 sum of angles", "QTS=55 same seg", "QTS=55"], 3, [], "", False),
    ("25P2 Q10.2", "2025_P2_Q10.2",
     ["SPQ=125 opp angles", "S1=R=55", "PQ parallel SR co-int"],
     3, [], "", False),
    ("25P2 Q10.3", "2025_P2_Q10.3",
     ["Q1=90 co-int", "PT is diameter converse", "PT is diameter"],
     3, [], "", False),
    ("25P2 Q11.1.1", "2025_P2_Q11.1.1",
     ["FD/CF = GA/CF", "=2/3"], 2, [], "", False),
    ("25P2 Q11.1.2", "2025_P2_Q11.1.2",
     ["FD=2/3CF", "FD=2/3(2x)", "BA/ED ratio", "=4/3x"],
     4, [], "", False),
    ("25P2 Q11.1.3", "2025_P2_Q11.1.3",
     ["GFDA = CDA - CGF", "1/2(GC)(FC)sinC",
      "1/2 AC.CD sinC - 1/2 GC.CF sinC", "=9/16"], 4, [], "", False),
    ("25P2 Q11.2.1", "2025_P2_Q11.2.1",
     ["QE/PR = QW/PR prop theorem", "Proven"], 2, [], "", False),
    ("25P2 Q11.2.2", "2025_P2_Q11.2.2",
     ["PQ/QR = QZ/QR"], 1, [], "", False),
    ("25P2 Q11.2.3", "2025_P2_Q11.2.3",
     ["Q2 common", "S1=R2 tan-chord", "Z1=QSR", "QSZ || QRS"],
     4, [], "", False),
    ("25P2 Q11.2.4", "2025_P2_Q11.2.4",
     ["QS/QR = QZ/QS", "QS^2=QZ.QR", "PQ=QS"], 3, [], "", False),
    ("25P2 Q11.2.5", "2025_P2_Q11.2.5",
     ["PQ/RQ = PZ/PR", "PR = PW.QR/QE",
      "PW.QR/QE = QR.PZ/PQ", "PQ=sqrt(RQ.QZ)",
      "PW = QE.PZ/sqrt(RQ.QZ)"], 5, [], "", False),
'''

pattern = re.compile(r"\n\]\s*\n\s*\n\s*\ndef run\(\)")
match = pattern.search(src)
assert match, "Could not find end of CASES block"

close_bracket_pos = src.index("]", match.start() + 1)
new_src = src[:close_bracket_pos] + new_cases + src[close_bracket_pos:]
test_path.write_text(new_src, encoding="utf-8")
print(f"test file: {len(new_src.splitlines())} lines")
print("OK")

test file: 544 lines
OK


In [90]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] 23P1 Q1.1.1
  [PASS ] 23P1 Q1.1.2
  [PASS ] 23P1 Q1.2 mA
  [PASS ] 23P1 Q3.2
  [PASS ] 23P1 Q5.6 mA
  [PASS ] 23P1 Q7.1
  [PASS ] 23P1 Q8.4
  [PASS ] 23P2 Q1.1
  [PASS ] 23P2 Q3.2
  [PASS ] 23P2 Q5.1.1
  [PASS ] 24P1 Q1.1.1
  [PASS ] 24P1 Q1.1.2
  [PASS ] 24P1 Q1.1.3
  [PASS ] 24P1 Q1.1.4
  [PASS ] 24P1 Q1.1.5
  [PASS ] 24P1 Q1.2 mA
  [PASS ] 24P1 Q1.2 mB
  [PASS ] 24P1 Q1.3
  [PASS ] 24P1 Q2.1.1
  [PASS ] 24P1 Q2.1.2
  [PASS ] 24P1 Q2.2.1
  [PASS ] 24P1 Q2.2.2
  [PASS ] 24P1 Q3.1
  [PASS ] 24P1 Q3.2
  [PASS ] 24P1 Q3.3
  [PASS ] 24P1 Q4.1
  [PASS ] 24P1 Q4.2
  [PASS ] 24P1 Q4.4
  [PASS ] 24P1 Q5.1
  [PASS ] 24P1 Q5.2
  [PASS ] 24P1 Q5.3
  [PASS ] 24P1 Q5.4
  [PASS ] 24P1 Q5.5
  [PASS ] 24P1 Q6.1
  [PASS ] 24P1 Q6.2
  [PASS ] 24P1 Q6.3
  [PASS ] 24P1 Q6.4
  [PASS ] 24P1 Q7.1
  [PASS ] 24P1 Q7.2
  [PASS ] 24P1 Q7.3.1
  [PASS ] 24P1 Q7.3.2
  [PASS ] 24P1 Q8.1.1
  [PASS ] 24P1 Q8.1.2
  [PASS ] 24P1 Q8.2
  [PASS ] 24P1 Q8.3.1
  [PASS ] 24P1 Q8.3.2
  [PASS ] 24P1 Q8.3.3
  [PASS ] 

In [91]:
sg_path = ROOT / "src" / "tutor" / "step_grader.py"
src = sg_path.read_text(encoding="utf-8")

old = '''    if mode == "keyword":
        nline = _norm(line)
        if not nline:
            return False
        for form in forms:
            nf = _norm(form)
            if nf and nf in nline:
                return True
        return False'''

new = '''    if mode == "keyword":
        nline = _norm(line)
        if not nline:
            return False
        # Compact all whitespace: "X = Y" must match "X=Y"
        nline_compact = re.sub(r"\\s+", "", nline)
        for form in forms:
            nf = _norm(form)
            if not nf:
                continue
            nf_compact = re.sub(r"\\s+", "", nf)
            if nf_compact and nf_compact in nline_compact:
                return True
        return False'''

assert old in src, "_line_matches keyword block not found"
src = src.replace(old, new)
sg_path.write_text(src, encoding="utf-8")
print("OK — engine now treats all whitespace as insignificant in keyword mode")

OK — engine now treats all whitespace as insignificant in keyword mode


In [92]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] 23P1 Q1.1.1
  [PASS ] 23P1 Q1.1.2
  [PASS ] 23P1 Q1.2 mA
  [PASS ] 23P1 Q3.2
  [PASS ] 23P1 Q5.6 mA
  [PASS ] 23P1 Q7.1
  [PASS ] 23P1 Q8.4
  [PASS ] 23P2 Q1.1
  [PASS ] 23P2 Q3.2
  [PASS ] 23P2 Q5.1.1
  [PASS ] 24P1 Q1.1.1
  [PASS ] 24P1 Q1.1.2
  [PASS ] 24P1 Q1.1.3
  [PASS ] 24P1 Q1.1.4
  [PASS ] 24P1 Q1.1.5
  [PASS ] 24P1 Q1.2 mA
  [PASS ] 24P1 Q1.2 mB
  [PASS ] 24P1 Q1.3
  [PASS ] 24P1 Q2.1.1
  [PASS ] 24P1 Q2.1.2
  [PASS ] 24P1 Q2.2.1
  [PASS ] 24P1 Q2.2.2
  [PASS ] 24P1 Q3.1
  [PASS ] 24P1 Q3.2
  [PASS ] 24P1 Q3.3
  [PASS ] 24P1 Q4.1
  [PASS ] 24P1 Q4.2
  [PASS ] 24P1 Q4.4
  [PASS ] 24P1 Q5.1
  [PASS ] 24P1 Q5.2
  [PASS ] 24P1 Q5.3
  [PASS ] 24P1 Q5.4
  [PASS ] 24P1 Q5.5
  [PASS ] 24P1 Q6.1
  [PASS ] 24P1 Q6.2
  [PASS ] 24P1 Q6.3
  [PASS ] 24P1 Q6.4
  [PASS ] 24P1 Q7.1
  [PASS ] 24P1 Q7.2
  [PASS ] 24P1 Q7.3.1
  [PASS ] 24P1 Q7.3.2
  [PASS ] 24P1 Q8.1.1
  [PASS ] 24P1 Q8.1.2
  [PASS ] 24P1 Q8.2
  [PASS ] 24P1 Q8.3.1
  [PASS ] 24P1 Q8.3.2
  [PASS ] 24P1 Q8.3.3
  [PASS ] 

In [93]:
csv_path = ROOT / "data" / "processed" / "memo_step_marks" / "memo_step_marks_2025_P2_proofs.csv"
src = csv_path.read_text(encoding="utf-8")

old = '2025_P2_Q11.1.3,3,Area CDA - CGF,1,"1/2 AC.CD sinC - 1/2 GC.CF sinC","ac.cd sinc - gc.cf sinc",euclidean.similarity,euclid,A,false,keyword'
new = '2025_P2_Q11.1.3,3,Area CDA - CGF,1,"1/2 AC.CD sinC - 1/2 GC.CF sinC","ac.cdsinc|gc.cfsinc|ac.cd sinc",euclidean.similarity,euclid,A,false,keyword'

assert old in src, "Q11.1.3 row not found"
src = src.replace(old, new)
csv_path.write_text(src, encoding="utf-8")
print("CSV: Q11.1.3 step 3 keyword widened to individual area terms")
print("OK")

CSV: Q11.1.3 step 3 keyword widened to individual area terms
OK


In [94]:
import subprocess, sys
r = subprocess.run([sys.executable, str(ROOT / "tests" / "test_step_grader.py")],
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr)
print("Exit code:", r.returncode)

  [PASS ] 23P1 Q1.1.1
  [PASS ] 23P1 Q1.1.2
  [PASS ] 23P1 Q1.2 mA
  [PASS ] 23P1 Q3.2
  [PASS ] 23P1 Q5.6 mA
  [PASS ] 23P1 Q7.1
  [PASS ] 23P1 Q8.4
  [PASS ] 23P2 Q1.1
  [PASS ] 23P2 Q3.2
  [PASS ] 23P2 Q5.1.1
  [PASS ] 24P1 Q1.1.1
  [PASS ] 24P1 Q1.1.2
  [PASS ] 24P1 Q1.1.3
  [PASS ] 24P1 Q1.1.4
  [PASS ] 24P1 Q1.1.5
  [PASS ] 24P1 Q1.2 mA
  [PASS ] 24P1 Q1.2 mB
  [PASS ] 24P1 Q1.3
  [PASS ] 24P1 Q2.1.1
  [PASS ] 24P1 Q2.1.2
  [PASS ] 24P1 Q2.2.1
  [PASS ] 24P1 Q2.2.2
  [PASS ] 24P1 Q3.1
  [PASS ] 24P1 Q3.2
  [PASS ] 24P1 Q3.3
  [PASS ] 24P1 Q4.1
  [PASS ] 24P1 Q4.2
  [PASS ] 24P1 Q4.4
  [PASS ] 24P1 Q5.1
  [PASS ] 24P1 Q5.2
  [PASS ] 24P1 Q5.3
  [PASS ] 24P1 Q5.4
  [PASS ] 24P1 Q5.5
  [PASS ] 24P1 Q6.1
  [PASS ] 24P1 Q6.2
  [PASS ] 24P1 Q6.3
  [PASS ] 24P1 Q6.4
  [PASS ] 24P1 Q7.1
  [PASS ] 24P1 Q7.2
  [PASS ] 24P1 Q7.3.1
  [PASS ] 24P1 Q7.3.2
  [PASS ] 24P1 Q8.1.1
  [PASS ] 24P1 Q8.1.2
  [PASS ] 24P1 Q8.2
  [PASS ] 24P1 Q8.3.1
  [PASS ] 24P1 Q8.3.2
  [PASS ] 24P1 Q8.3.3
  [PASS ] 